# 🎙️ ㄚ亮笑長的 Colab 轉錄助手

「ㄚ亮笑長的內容助手」Colab 版本（版本日期：2026.11.23）  
🌐 同功能網頁版本：[ㄚ亮笑長的內容創作助手 (https://ctb.52hal.cc)](https://ctb.52hal.cc)  
📖 新手圖文使用說明：[線上完整操作指南 (https://hsuliang.github.io/cta/)](https://hsuliang.github.io/cta/)

### ✨ 功能與流程簡介
- 🎧 **音檔／YouTube 轉錄軌**：開啟 **T4 GPU**，依序執行 **步驟 1 ➔ 步驟 2A ➔ 步驟 3 ➔ 步驟 4 ➔ 步驟 5**。
- 📝 **支援現成 SRT / TXT 匯入（免開 GPU）**：已有字幕或逐字稿時，**免執行步驟 1 與步驟 3**，直接於 **步驟 2B** 或 **步驟 4** 匯入文稿即可產出全套作品。

## <font color="#00796b"><b>1. 檢查 GPU 與安裝套件</b></font>

In [ ]:
# @title 🟦 **1. 檢查 GPU 與安裝套件**
import os, shutil, subprocess, sys
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

clear_output(wait=True)
print('階段 1/6：檢查 GPU。', flush=True)
wait_step1 = widgets.HTML(value='<b>階段 1/6：檢查 GPU 與安裝套件</b> <progress style="accent-color:green"></progress>')
display(wait_step1)

if shutil.which('nvidia-smi') is None:
    wait_step1.value = '❌ 未偵測到 GPU，請至「執行階段 → 變更執行階段類型」選擇 GPU。'
    raise RuntimeError('未偵測到 GPU，請在執行階段類型選擇 GPU（例如 T4）後重新連線。')

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False)
if gpu.returncode != 0 or not gpu.stdout.strip():
    wait_step1.value = '❌ GPU 狀態異常，請在執行階段類型確認選擇 GPU。'
    raise RuntimeError('沒有偵測到 GPU，請先在執行階段類型選擇 GPU。')
print(gpu.stdout.strip(), flush=True)

print('階段 1/6：正在安裝必要套件（Faster-Whisper、OpenCC、yt-dlp）...', flush=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'faster-whisper==1.1.1',
    'av<18',
    'opencc-python-reimplemented==0.1.7',
    'nvidia-cublas-cu12',
    'nvidia-cudnn-cu12',
    'yt-dlp'
], check=True)

work = Path('/content/alta_transcribe_work')
work.mkdir(parents=True, exist_ok=True)

wait_step1.value = '✅ 階段 1/6 完成：GPU、語音轉錄與 YouTube 下載環境皆已就緒。'
print('階段 1/6 完成：GPU 與環境已就緒。可繼續執行步驟 2。', flush=True)


def stream_command(command, env, cwd, on_line=None):
    '''即時傳遞子行程輸出到 Notebook 介面'''
    with subprocess.Popen(command, env=env, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            if on_line is None:
                print(line, end='', flush=True)
            else:
                on_line(line)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

## <font color="#1565c0"><b>2A. 音檔／YouTube 轉錄（選擇音檔來源 & 詞庫設定）</b></font>

In [ ]:
# @title 🟩 **2A. 選擇音檔／YouTube 來源 & 詞庫設定**
# @markdown ### 🎧 音檔與 YouTube 來源設定
音檔來源模式 = "Google 雲端硬碟（單一音檔 — 互動式下拉選單）" # @param ["Google 雲端硬碟（單一音檔 — 互動式下拉選單）", "🎬 YouTube 網址直轉（自動下載音軌並記住影片 ID）", "Google 雲端硬碟（資料夾整批全自動轉錄）", "💻 本機上傳（音訊或視訊檔）"]
YouTube影片或直播網址 = "" # @param {type:"string"}
雲端硬碟指定路徑_留空自動跳出選單 = "" # @param {type:"string"}
自動將成果存回雲端硬碟 = True # @param {type:"boolean"}
雲端成果儲存母資料夾 = "Colab Notebooks/轉錄成果_ㄚ亮笑長助手" # @param {type:"string"}
整批轉錄時略過已有SRT成果之音檔 = False # @param {type:"boolean"}

# @markdown ### 📖 錯字替換與專有名詞詞庫（雲端永久記憶）
從電腦上傳匯入詞庫JSON設定檔 = False # @param {type:"boolean"}
錯字替換規則 = "阿亮校長->ㄚ亮笑長, 阿亮笑長->ㄚ亮笑長, 丫亮校長->ㄚ亮笑長, 阿林校長->ㄚ亮笑長, 思辯->思辨, 居酒屋外->居久屋, 瀑瀑聊聊->噗噗聊聊, 千億科技->謙懿科技, 捷可達->傑可達, 換名->冠名" # @param {type:"string"}
正向專有名詞 = "ㄚ亮笑長, 噗噗聊聊, 居久屋微醺夜, 居久屋課程筆記, 謙懿科技, 傑可達數位, T++教育+年華, 數位科技思辨課, NotebookLM, Gemini" # @param {type:"string"}
禁用或常見誤聽參考詞 = "" # @param {type:"string"}

audio_source_type = 音檔來源模式
source_type = 音檔來源模式
youtube_url = YouTube影片或直播網址
drive_path = 雲端硬碟指定路徑_留空自動跳出選單
auto_save_to_drive = 自動將成果存回雲端硬碟
drive_output_folder = 雲端成果儲存母資料夾
skip_already_transcribed = 整批轉錄時略過已有SRT成果之音檔
upload_dict_json_from_pc = 從電腦上傳匯入詞庫JSON設定檔
typo_replace_rules = 錯字替換規則
positive_terms = 正向專有名詞
negative_terms = 禁用或常見誤聽參考詞

import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

clear_output(wait=True)

work = Path('/content/alta_transcribe_work')
work.mkdir(parents=True, exist_ok=True)

cleanup_names = (
    'input.media', 'output_stem.txt', 'job_queue.json',
    'transcript.txt', 'transcript.srt', 'transcript.vtt', 'segments.json',
    'raw_transcript.txt', 'raw_transcript.srt', 'raw_transcript.vtt', 'raw_segments.json',
    'article_and_summary.md', 'article_and_summary.txt', 'chapters.txt',
    'blog_post.html', 'blog_post.md', 'titles_and_shownotes.md',
    'social_posts.md', 'visual_prompts.md'
)
for name in cleanup_names:
    p = work / name
    if p.exists() or p.is_symlink():
        try:
            p.unlink()
        except Exception:
            pass

for sub_d in ('jobs', 'yt_downloads'):
    if (work / sub_d).is_dir():
        shutil.rmtree(work / sub_d, ignore_errors=True)

for zip_file in work.glob('*.zip'):
    try:
        zip_file.unlink()
    except Exception:
        pass

MEDIA_EXTS = {'.m4a', '.mp3', '.wav', '.mp4', '.aac', '.flac', '.ogg', '.mkv', '.mov', '.webm', '.wma', '.m4v'}
TEXT_EXTS = {'.srt', '.vtt', '.txt', '.md'}
OUTPUT_FOLDER_NAME = '轉錄成果_ㄚ亮笑長助手'
CTB_MEMORY_FOLDER_NAME = 'CTB_詞庫設定_永久記憶'

# 預先寫入共用字幕與文稿解析核心模組 (p0_subtitles.py)，讓使用者就算跳過步驟 1 與步驟 3 也能直接解析 SRT/TXT！
_p0_sub_code = "# Subtitle formatting, Chinese conversion, SRT/VTT/TXT parser, and ctb.52hal.cc deterministic cleaning utilities.\nimport math\nimport re\n\ntry:\n    from opencc import OpenCC\n    _OPENCC_INSTANCE = OpenCC('s2twp')\nexcept Exception:\n    _OPENCC_INSTANCE = None\n\n\ndef to_trad_tw(text):\n    if not text:\n        return ''\n    if _OPENCC_INSTANCE is not None:\n        try:\n            return _OPENCC_INSTANCE.convert(text)\n        except Exception:\n            return text\n    return text\n\n\ndef clean_text_deterministic(text, replace_rules=None):\n    if not text:\n        return '', 0\n    s = to_trad_tw(text.strip())\n    s = re.sub(r'^[\\[\\(【（]?\\s*(?:音樂|music|Music|MUSIC|♪+|♫+)\\s*[\\]\\)】）]?$', '【音樂】', s)\n    s = re.sub(r'\\b([A-Z])(?:\\s+([A-Z]))+\\b', lambda m: re.sub(r'\\s+', '', m.group(0)), s)\n    applied_count = 0\n    for rule in (replace_rules or []):\n        orig = rule.get('original', '')\n        repl = rule.get('replacement', '')\n        if orig and orig in s:\n            applied_count += s.count(orig)\n            s = s.replace(orig, repl)\n    return s.strip(), applied_count\n\n\ndef parse_timestamp_to_ms(ts_str):\n    s = (ts_str or '').strip().replace(',', '.')\n    parts = s.split(':')\n    try:\n        if len(parts) == 3:\n            hh = int(parts[0])\n            mm = int(parts[1])\n            sec_f = float(parts[2])\n        elif len(parts) == 2:\n            hh = 0\n            mm = int(parts[0])\n            sec_f = float(parts[1])\n        else:\n            return 0\n        return max(0, int(round((hh * 3600 + mm * 60 + sec_f) * 1000)))\n    except Exception:\n        return 0\n\n\ndef parse_existing_text_or_srt(raw_content, ext='.srt', replace_rules=None):\n    '''\n    解析現成的 .srt / .vtt / .txt / .md 或是直接貼上的純文字，並自動執行確定性錯字清洗：\n    回傳：(cues_tuples, converted_segments, total_rule_hits, has_real_timestamps, duration_sec)\n    - cues_tuples: [(start_ms, end_ms, text), ...]\n    '''\n    text_body = (raw_content or '').lstrip('\\ufeff').replace('\\r\\n', '\\n').replace('\\r', '\\n').strip()\n    if not text_body:\n        raise ValueError('匯入的文稿內容為空白！')\n\n    # 自動偵測是否包含 SRT / VTT 時間碼 (00:00:00,000 --> 00:00:03,000)\n    ts_pattern = re.compile(\n        r'((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})\\s*-->\\s*((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})'\n    )\n    has_ts_arrows = bool(ts_pattern.search(text_body))\n    cues = []\n    converted_segments = []\n    total_hits = 0\n\n    if has_ts_arrows:\n        blocks = re.split(r'\\n\\s*\\n+', text_body)\n        prev_end = 0\n        for blk in blocks:\n            lines = [ln.strip() for ln in blk.strip().splitlines() if ln.strip()]\n            if not lines:\n                continue\n            if lines[0].upper().startswith('WEBVTT'):\n                lines = lines[1:]\n                if not lines:\n                    continue\n            m_ts = None\n            ts_line_idx = -1\n            for idx_l, ln in enumerate(lines[:3]):\n                m = ts_pattern.search(ln)\n                if m:\n                    m_ts = m\n                    ts_line_idx = idx_l\n                    break\n            if not m_ts:\n                continue\n            start_ms = parse_timestamp_to_ms(m_ts.group(1))\n            end_ms = parse_timestamp_to_ms(m_ts.group(2))\n            if end_ms <= start_ms:\n                end_ms = start_ms + 2000\n            text_lines = lines[ts_line_idx + 1:]\n            # 自動移除既有連帶的台呼水印，避免重複疊加\n            text_lines = [tl for tl in text_lines if tl.strip() != '【ㄚ亮笑長的 Colab轉錄助手】']\n            joined_text = ' '.join(text_lines).strip()\n            if not joined_text:\n                continue\n            cleaned_t, hits = clean_text_deterministic(joined_text, replace_rules=replace_rules)\n            total_hits += hits\n            if cleaned_t:\n                cues.append((start_ms, end_ms, cleaned_t))\n                converted_segments.append({\n                    'start': round(start_ms / 1000.0, 3),\n                    'end': round(end_ms / 1000.0, 3),\n                    'text': joined_text,\n                    'text_zh_tw': cleaned_t\n                })\n                prev_end = max(prev_end, end_ms)\n\n        if cues:\n            duration_sec = max(1.0, round(prev_end / 1000.0, 2))\n            return cues, converted_segments, total_hits, True, duration_sec\n\n    # 純文字 (.txt / .md / 直接貼上文字)：智慧依换行與句號/問號/驚嘆號切分句子，賦予預估閱讀時間軸\n    raw_lines = [ln.strip() for ln in text_body.splitlines() if ln.strip()]\n    sentences = []\n    for ln in raw_lines:\n        # 若單行很長，依全形句號、問號、驚嘆號、分號適度拆句，方便字幕與段落對齊\n        if len(ln) > 45:\n            parts = re.split(r'(?<=[。！？；!?])', ln)\n            for p in parts:\n                p_s = p.strip()\n                if p_s:\n                    sentences.append(p_s)\n        else:\n            sentences.append(ln)\n\n    curr_ms = 0\n    for sent in sentences:\n        if sent == '【ㄚ亮笑長的 Colab轉錄助手】':\n            continue\n        cleaned_t, hits = clean_text_deterministic(sent, replace_rules=replace_rules)\n        total_hits += hits\n        if not cleaned_t:\n            continue\n        # 依字數估算自然語速（約每秒 4.5 字，最短 2 秒，最長 8 秒）\n        dur_ms = max(2000, min(8000, int(round(len(cleaned_t) / 4.5 * 1000))))\n        start_ms = curr_ms\n        end_ms = curr_ms + dur_ms\n        cues.append((start_ms, end_ms, cleaned_t))\n        converted_segments.append({\n            'start': round(start_ms / 1000.0, 3),\n            'end': round(end_ms / 1000.0, 3),\n            'text': sent,\n            'text_zh_tw': cleaned_t\n        })\n        curr_ms = end_ms\n\n    if not cues:\n        raise ValueError('未能從文稿中解析出任何有效句子！')\n    duration_sec = max(1.0, round(curr_ms / 1000.0, 2))\n    return cues, converted_segments, total_hits, False, duration_sec\n\n\ndef subtitle_cues(segments, duration):\n    if not isinstance(duration, (int, float)) or not math.isfinite(duration) or duration <= 0:\n        raise ValueError('無效的音訊長度')\n    max_ms = math.floor(duration * 1000)\n    cues = []\n    previous_ms = 0\n    sorted_segments = sorted(segments, key=lambda s: s['start'])\n    for segment in sorted_segments:\n        start, end = segment['start'], segment['end']\n        if not (isinstance(start, (int, float)) and isinstance(end, (int, float)) and math.isfinite(start) and math.isfinite(end)):\n            continue\n        start_ms = max(previous_ms, round(start * 1000))\n        end_ms = min(round(end * 1000), max_ms)\n        if end_ms <= start_ms:\n            end_ms = min(start_ms + 100, max_ms)\n        if start_ms >= end_ms:\n            continue\n        text = segment['text'].strip()\n        if text:\n            cues.append((start_ms, end_ms, text))\n            previous_ms = end_ms\n    if not cues:\n        raise ValueError('未產生任何有效字幕區間')\n    return cues\n\n\ndef stamp(milliseconds, separator):\n    hours, milliseconds = divmod(milliseconds, 3600000)\n    minutes, milliseconds = divmod(milliseconds, 60000)\n    seconds, milliseconds = divmod(milliseconds, 1000)\n    return f'{hours:02}:{minutes:02}:{seconds:02}{separator}{milliseconds:03}'\n\n\ndef render(cues, kind):\n    if kind not in ('srt', 'vtt'):\n        raise ValueError('不支援的字幕格式')\n    separator = ',' if kind == 'srt' else '.'\n    blocks = []\n    for index, (start, end, text) in enumerate(cues, 1):\n        prefix = (str(index) + chr(10)) if kind == 'srt' else ''\n        time_part = stamp(start, separator) + ' --> ' + stamp(end, separator)\n        blocks.append(prefix + time_part + chr(10) + text)\n    header = ('WEBVTT' + chr(10) + chr(10)) if kind == 'vtt' else ''\n    return header + (chr(10) + chr(10)).join(blocks) + chr(10)\n\n\ndef traditional_segments(segments, replace_rules=None):\n    out = []\n    total_replaced = 0\n    for segment in segments:\n        tw_text = to_trad_tw(segment['text'])\n        cleaned_text, count = clean_text_deterministic(tw_text, replace_rules=replace_rules)\n        total_replaced += count\n        out.append({**segment, 'text_zh_tw': cleaned_text})\n    return out, total_replaced\n\n\ndef _timed_chars(segment):\n    text = segment['text']\n    words = segment.get('words') or []\n    words_clean = re.sub(r'\\s+', '', ''.join(w['word'] for w in words))\n    text_clean = re.sub(r'\\s+', '', text)\n    if words_clean == text_clean and len(words) > 0 and all(\n        isinstance(w.get('start'), (int, float)) and isinstance(w.get('end'), (int, float))\n        and math.isfinite(w['start']) and math.isfinite(w['end'])\n        and segment['start'] <= w['start'] < w['end'] <= segment['end']\n        for w in words\n    ):\n        result = []\n        for word in words:\n            value = word['word'].strip()\n            if not value:\n                continue\n            span = word['end'] - word['start']\n            for i, ch in enumerate(value):\n                c_start = word['start'] + span * (i / len(value))\n                c_end = word['start'] + span * ((i + 1) / len(value))\n                result.append((ch, c_start, c_end))\n        if result:\n            return result, 'word_timestamps'\n\n    span = max(0.01, segment['end'] - segment['start'])\n    return [(ch, segment['start'] + span * i / len(text),\n             segment['start'] + span * (i + 1) / len(text))\n            for i, ch in enumerate(text)], 'segment_interpolated'\n\n\ndef build_outputs_from_cues(cues, converted_segments, intro_music_seconds=0, txt_lines=None):\n    srt_cues = list(cues)\n    vtt_cues = list(cues)\n\n    first_start, first_end, first_text = srt_cues[0]\n    promo_text = '【ㄚ亮笑長的 Colab轉錄助手】'\n\n    if intro_music_seconds > 0:\n        music_ms = round(intro_music_seconds * 1000)\n        if music_ms <= first_start:\n            srt_cues.insert(0, (0, music_ms, '【音樂】' + chr(10) + promo_text))\n            vtt_cues.insert(0, (0, music_ms, '【音樂】'))\n        else:\n            if promo_text not in first_text:\n                srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n    else:\n        if promo_text not in first_text:\n            srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n\n    if txt_lines is None:\n        txt_lines = [row['text_zh_tw'] for row in converted_segments]\n\n    outputs = {\n        'transcript.txt': chr(10).join(txt_lines) + chr(10),\n        'transcript.srt': render(srt_cues, 'srt'),\n        'transcript.vtt': render(vtt_cues, 'vtt')\n    }\n    return outputs\n\n\ndef friendly_exports(segments, duration, split=True, punctuation=False, max_chars=22, max_seconds=6, intro_music_seconds=0, replace_rules=None):\n    if max_chars < 4 or max_seconds <= 0:\n        raise ValueError('無效的字幕長度設定')\n    display = []\n    timing_methods = set()\n    total_rule_hits = 0\n    for index, segment in enumerate(segments):\n        raw_text = segment['text'].strip()\n        if not raw_text:\n            continue\n        chars, method = _timed_chars(segment)\n        if segment['end'] > duration and segment['start'] < duration:\n            scale = (duration - segment['start']) / (segment['end'] - segment['start'])\n            chars = [(char, segment['start'] + (start - segment['start']) * scale,\n                      segment['start'] + (end - segment['start']) * scale)\n                     for char, start, end in chars]\n            method = 'segment_end_adjusted'\n        timing_methods.add(method)\n        start = 0\n        while start < len(chars):\n            end = start + 1\n            limit_break = False\n            while end < len(chars):\n                candidate = ''.join(c[0] for c in chars[start:end + 1])\n                if split and (len(to_trad_tw(candidate)) > max_chars or\n                              chars[end][2] - chars[start][1] > max_seconds):\n                    limit_break = True\n                    break\n                if split and chars[end - 1][0] in '。！？；!?;':\n                    break\n                if split and punctuation and chars[end][1] - chars[end - 1][2] >= 0.65:\n                    break\n                end += 1\n            if (limit_break and end == len(chars) - 1 and end - start > 1\n                    and chars[end - 1][0].isalnum() and chars[end][0].isalnum()\n                    and chars[end][1] - chars[end - 1][2] < 0.65\n                    and chars[end][2] - chars[end - 1][1] <= max_seconds\n                    and len(to_trad_tw(chars[end - 1][0] + chars[end][0])) <= max_chars):\n                end -= 1\n            text = to_trad_tw(''.join(c[0] for c in chars[start:end])).strip()\n            text, hits = clean_text_deterministic(text, replace_rules=replace_rules)\n            total_rule_hits += hits\n            if text:\n                last_chunk = end == len(chars)\n                next_start = segments[index + 1]['start'] if index + 1 < len(segments) else duration\n                gap = next_start - segment['end'] if last_chunk else chars[end][1] - chars[end - 1][2]\n                if punctuation and gap >= 0.65 and text[-1] not in '，。！？；：、,.!?;:':\n                    text += '。'\n                cue_start, cue_end = chars[start][1], chars[end - 1][2]\n                display.append({'start': cue_start, 'end': cue_end, 'text': text})\n            start = end\n\n    cues = subtitle_cues(display, duration)\n    converted, seg_hits = traditional_segments(segments, replace_rules=replace_rules)\n    outputs = build_outputs_from_cues(cues, converted, intro_music_seconds=intro_music_seconds, txt_lines=[t for _, _, t in cues])\n    return converted, outputs, sorted(timing_methods), cues, max(total_rule_hits, seg_hits)\n"
(work / 'p0_subtitles.py').write_text(_p0_sub_code, encoding='utf-8')
if str(work) not in sys.path:
    sys.path.insert(0, str(work))
from p0_subtitles import parse_existing_text_or_srt, build_outputs_from_cues


def render_stepper_banner(active_step=2, mode_badge='語音轉錄軌'):
    '''渲染頂部 Web App 級流程導航狀態橫幅'''
    steps_info = [
        (1, '1. GPU 環境'),
        (2, '2. 來源與詞庫'),
        (3, '3. 語音轉錄'),
        (4, '4. AI 創作工作台'),
        (5, '5. 打包下載')
    ]
    pills = []
    for s_num, s_label in steps_info:
        if s_num == active_step:
            bg, fg, bd = '#ffd54f', '#004d40', '2px solid #ffffff'
            fw = 'bold'
        elif s_num < active_step:
            bg, fg, bd = 'rgba(255,255,255,0.22)', '#e0f2f1', '1px solid rgba(255,255,255,0.35)'
            fw = 'normal'
        else:
            bg, fg, bd = 'rgba(0,0,0,0.18)', '#b2dfdb', '1px solid rgba(255,255,255,0.15)'
            fw = 'normal'
        pills.append(
            f'<span style="display:inline-block; padding:4px 11px; margin:2px 4px; border-radius:999px; background:{bg}; color:{fg}; border:{bd}; font-weight:{fw}; font-size:12.5px;">{s_label}</span>'
        )
    pills_html = ' <span style="color:#80cbc4;">➔</span> '.join(pills)
    return (
        '<div style="background:linear-gradient(135deg, #004d40 0%, #00796b 100%); color:#ffffff; padding:14px 18px; border-radius:12px; margin-bottom:12px; box-shadow:0 3px 10px rgba(0,77,64,0.18);">'
        '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:8px;">'
        '<div style="font-size:17px; font-weight:bold; letter-spacing:0.5px;">🎙️ ㄚ亮笑長的 Colab 全媒體創作助手 </div>'
        f'<div style="font-size:12.5px; background:rgba(255,255,255,0.16); padding:3px 10px; border-radius:6px;">當前模式：<b>{mode_badge}</b></div>'
        '</div>'
        f'<div style="margin-top:4px;">{pills_html}</div>'
        '</div>'
    )


stepper_box_s2 = widgets.HTML(value=render_stepper_banner(2, '📝 現成文稿直通軌（免跑步驟 3）' if ('現成' in source_type or '直接貼上' in source_type) else '🎧 語音轉錄軌'))
wait_step2 = widgets.HTML(value='<b>階段 2/6：準備來源與同步雲端永久記憶詞庫...</b> <progress style="accent-color:green"></progress>')
display(stepper_box_s2, wait_step2)

def extract_youtube_id(url_or_id):
    '''從各種 YouTube 網址格式提取 11 碼影片 ID'''
    s = (url_or_id or '').strip()
    if not s:
        return ''
    if re.fullmatch(r'[A-Za-z0-9_-]{11}', s):
        return s
    m = re.search(r'(?:v=|youtu\.be/|youtube\.com/(?:embed|v|shorts|live)/)([A-Za-z0-9_-]{11})', s)
    return m.group(1) if m else ''


def ensure_drive_mounted_if_needed():
    '''確保 Google 雲端硬碟已掛載'''
    my_drive_root = Path('/content/drive/MyDrive')
    if not my_drive_root.exists():
        from google.colab import drive
        print('階段 2/6：正在掛載 Google 雲端硬碟；如出現授權畫面，請點選確認...', flush=True)
        try:
            drive.mount('/content/drive', timeout_ms=120000)
        except Exception as error:
            wait_step2.value = '❌ 雲端硬碟掛載失敗或逾時，請確認授權畫面後重試。'
            raise RuntimeError('雲端硬碟掛載失敗或逾時。') from error
    return my_drive_root


def get_ctb_memory_dir():
    '''取得雲端硬碟中的 CTB 詞庫永久記憶資料夾路徑'''
    my_drive_root = Path('/content/drive/MyDrive')
    if not my_drive_root.exists():
        return work / CTB_MEMORY_FOLDER_NAME
    colab_nb = my_drive_root / 'Colab Notebooks'
    base_d = colab_nb if colab_nb.exists() else my_drive_root
    mem_dir = base_d / CTB_MEMORY_FOLDER_NAME
    mem_dir.mkdir(parents=True, exist_ok=True)
    return mem_dir


def ingest_ctb_json_content(raw_json_str, rep_map, pos_list, neg_list):
    '''
    解析來自 ctb.52hal.cc 的 JSON 設定檔（自動識別 Batch_Replace_Rules 或 AI_Terminology_Rules 格式）
    回傳：(新增/更新的替換規則數, 新增的專有名詞數)
    '''
    added_rep = 0
    added_term = 0
    try:
        data = json.loads(raw_json_str)
    except Exception:
        return 0, 0

    items = []
    if isinstance(data, list):
        items = data
    elif isinstance(data, dict):
        for k in ('rules', 'replace_rules', 'terms', 'terminology', 'items', 'data'):
            if isinstance(data.get(k), list):
                items.extend(data[k])

    for item in items:
        if not isinstance(item, dict):
            continue
        # 情況 1：ctb.52hal.cc 錯字替換規則 {"original": "...", "replacement": "..."}
        if 'original' in item and 'replacement' in item:
            orig = str(item.get('original') or '').strip()
            repl = str(item.get('replacement') or '').strip()
            if orig:
                rep_map[orig] = repl
                added_rep += 1
        # 情況 2：ctb.52hal.cc 專有名詞規則 {"type": "positive" | "negative", "term": "..."}
        elif 'term' in item:
            term_val = str(item.get('term') or '').strip()
            term_type = str(item.get('type') or 'positive').strip().lower()
            if term_val:
                if term_type == 'negative':
                    if term_val not in neg_list:
                        neg_list.append(term_val)
                else:
                    if term_val not in pos_list:
                        pos_list.append(term_val)
                added_term += 1
    return added_rep, added_term


def load_and_sync_ctb_dictionaries():
    '''
    完整執行「雲端永久記憶自動載入 ➔ 電腦上傳新 JSON 匯入 ➔ 表單文字合併 ➔ 自動存回雲端永久記憶」
    '''
    if auto_save_to_drive or ('Google 雲端硬碟' in source_type) or upload_dict_json_from_pc:
        ensure_drive_mounted_if_needed()

    mem_dir = get_ctb_memory_dir()
    mem_dir.mkdir(parents=True, exist_ok=True)
    rep_json_path = mem_dir / 'Batch_Replace_Rules.json'
    term_json_path = mem_dir / 'AI_Terminology_Rules.json'

    rep_map = {}
    pos_list = []
    neg_list = []
    loaded_sources = []

    # 1. 自動掃描並載入雲端硬碟永久記憶資料夾 (以及 Colab Notebooks 根目錄下任何 CTB JSON 檔案)
    candidate_json_files = []
    if rep_json_path.is_file():
        candidate_json_files.append(rep_json_path)
    if term_json_path.is_file():
        candidate_json_files.append(term_json_path)
    for extra_j in sorted(mem_dir.glob('*.json')):
        if extra_j not in candidate_json_files:
            candidate_json_files.append(extra_j)
    if mem_dir.parent.exists():
        for root_j in sorted(mem_dir.parent.glob('*Rules*.json')):
            if root_j not in candidate_json_files:
                candidate_json_files.append(root_j)

    for jf in candidate_json_files:
        try:
            r_cnt, t_cnt = ingest_ctb_json_content(jf.read_text(encoding='utf-8'), rep_map, pos_list, neg_list)
            if r_cnt > 0 or t_cnt > 0:
                loaded_sources.append(f'{jf.name} (替換:{r_cnt}/名詞:{t_cnt})')
        except Exception:
            pass

    # 2. 若使用者勾選「從電腦上傳匯入 ctb.52hal.cc 的 JSON 設定檔」
    if upload_dict_json_from_pc:
        from google.colab import files
        print('📥 請點擊下方「選擇檔案」從電腦選取詞庫 JSON 設定檔（可同時多選 Batch_Replace_Rules*.json 與 AI_Terminology_Rules*.json）：', flush=True)
        uploaded_jsons = files.upload()
        for u_name, u_bytes in (uploaded_jsons or {}).items():
            try:
                u_text = u_bytes.decode('utf-8', errors='replace')
                r_cnt, t_cnt = ingest_ctb_json_content(u_text, rep_map, pos_list, neg_list)
                # 清理當前目錄下的上傳暫存檔
                if Path(u_name).is_file():
                    Path(u_name).unlink()
                if r_cnt > 0 or t_cnt > 0:
                    loaded_sources.append(f'電腦上傳:{u_name} (替換:{r_cnt}/名詞:{t_cnt})')
                    print(f'✅ 已成功匯入 {u_name}：新增/更新 {r_cnt} 組錯字替換、{t_cnt} 個專有名詞！', flush=True)
                else:
                    print(f'⚠️ {u_name} 未偵測到相容的詞庫欄位（original/replacement 或 type/term）。', flush=True)
            except Exception as u_err:
                print(f'⚠️ 解析 {u_name} 時發生錯誤：{u_err}', flush=True)

    # 3. 合併上方表單文字框填寫的規則（表單文字可隨時補充新詞）
    for item in re.split(r'[,，;；]+', (typo_replace_rules or '').replace(chr(10), ',')):
        item = item.strip()
        if '->' in item:
            orig, repl = item.split('->', 1)
        elif '=>' in item:
            orig, repl = item.split('=>', 1)
        else:
            continue
        if orig.strip() and orig.strip() not in rep_map:
            rep_map[orig.strip()] = repl.strip()

    for x in re.split(r'[,，;；]+', (positive_terms or '').replace(chr(10), ',')):
        val = x.strip()
        if val and val not in pos_list:
            pos_list.append(val)

    for x in re.split(r'[,，;；]+', (negative_terms or '').replace(chr(10), ',')):
        val = x.strip()
        if val and val not in neg_list:
            neg_list.append(val)

    # 4. 自動將合併後的完整詞庫以 100% ctb.52hal.cc 原生 JSON 格式寫回雲端硬碟永久記憶！
    ctb_replace_array = [{'original': k, 'replacement': v} for k, v in rep_map.items()]
    ctb_term_array = (
        [{'type': 'positive', 'term': t} for t in pos_list] +
        [{'type': 'negative', 'term': t} for t in neg_list]
    )
    try:
        rep_json_path.write_text(json.dumps(ctb_replace_array, ensure_ascii=False, indent=2), encoding='utf-8')
        term_json_path.write_text(json.dumps(ctb_term_array, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as w_err:
        print(f'⚠️ 寫入雲端詞庫永久記憶時發生警告：{w_err}', flush=True)

    return {
        'replace_rules': ctb_replace_array,
        'positive_terms': pos_list,
        'negative_terms': neg_list,
        'memory_dir': str(mem_dir),
        'loaded_sources': loaded_sources
    }


def resolve_job_drive_save_dir(stem_name, audio_parent_dir=None):
    '''
    計算該音檔在雲端硬碟的專屬儲存路徑：
    會在指定的「轉錄成果_ㄚ亮笑長助手」母資料夾下，自動建立「與音檔同名（stem_name）」的子資料夾！
    '''
    if not auto_save_to_drive:
        return ''
    my_drive_root = ensure_drive_mounted_if_needed()
    custom_out = (drive_output_folder or '').strip().strip("'").strip('"')
    if custom_out:
        p = Path(custom_out)
        if not p.is_absolute():
            base_dir = my_drive_root / p
        elif str(p).startswith('/MyDrive/'):
            base_dir = Path('/content/drive') / str(p).lstrip('/')
        else:
            base_dir = p
    else:
        if audio_parent_dir and Path(audio_parent_dir).exists() and str(audio_parent_dir).startswith('/content/drive'):
            base_dir = Path(audio_parent_dir) / OUTPUT_FOLDER_NAME
        else:
            colab_nb = my_drive_root / 'Colab Notebooks'
            base_dir = (colab_nb if colab_nb.exists() else my_drive_root) / OUTPUT_FOLDER_NAME

    safe_stem = re.sub(r'[\\/:*?"<>|]+', '_', (stem_name or '轉錄結果').strip())
    episode_dir = base_dir / safe_stem
    return str(episode_dir)


dict_settings = load_and_sync_ctb_dictionaries()
global_yt_id = extract_youtube_id(youtube_url.split(',')[0]) if youtube_url.strip() else ''


def render_ctb_memory_panel():
    '''在畫面上顯示可展開檢視的 CTB 雲端永久記憶詞庫面板，並支援一鍵下載最新 JSON 回 ctb.52hal.cc'''
    rep_rules = dict_settings['replace_rules']
    pos_t = dict_settings['positive_terms']
    neg_t = dict_settings['negative_terms']
    src_note = '、'.join(dict_settings['loaded_sources']) if dict_settings['loaded_sources'] else '預設詞庫初始化'

    rep_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#fff3e0; border:1px solid #ffb74d; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">{r["original"]} ➔ <b>{r["replacement"]}</b></span>'
        for r in rep_rules
    )
    pos_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#e8f5e9; border:1px solid #81c784; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">🟢 {t}</span>'
        for t in pos_t
    )
    neg_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#ffebee; border:1px solid #e57373; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">🔴 {t}</span>'
        for t in neg_t
    ) or '<span style="color:#757575; font-size:12px;">（無）</span>'

    detail_html = widgets.HTML(
        value=(
            '<div style="padding:8px; line-height:1.8;">'
            f'<div><b>☁️ 雲端永久記憶庫位置：</b><code>{dict_settings["memory_dir"]}/</code>（下次開啟自動載入）</div>'
            f'<div><b>📥 本次載入來源：</b>{src_note}</div>'
            f'<div style="margin-top:6px;"><b>🔄 目前生效的錯字強制替換（共 {len(rep_rules)} 組）：</b><br>{rep_tags_html}</div>'
            f'<div style="margin-top:6px;"><b>🟢 目前生效的正向專有名詞（共 {len(pos_t)} 個）：</b><br>{pos_tags_html}</div>'
            f'<div style="margin-top:6px;"><b>🔴 目前生效的禁用／誤聽參考詞（共 {len(neg_t)} 個）：</b><br>{neg_tags_html}</div>'
            '</div>'
        )
    )
    dl_ctb_btn = widgets.Button(
        description='📤 下載最新詞庫 JSON 設定檔',
        button_style='info',
        layout=widgets.Layout(width='360px', margin='6px 0 0 8px')
    )

    def on_dl_ctb(_):
        from google.colab import files
        mem_d = Path(dict_settings['memory_dir'])
        for fn in ('Batch_Replace_Rules.json', 'AI_Terminology_Rules.json'):
            p = mem_d / fn
            if p.is_file():
                files.download(str(p))

    dl_ctb_btn.on_click(on_dl_ctb)
    acc = widgets.Accordion(children=[widgets.VBox([detail_html, dl_ctb_btn])])
    acc.set_title(0, f'📖 詞庫雲端永久記憶庫（已自動載入：錯字替換 {len(rep_rules)} 組 ｜ 🟢 專有名詞 {len(pos_t)} 個 — 點此展開明細）')
    acc.selected_index = None
    display(acc)


def save_job_queue(mode, jobs_list, auto_save):
    '''儲存待轉錄工作佇列與詞庫設定'''
    queue_data = {
        'mode': mode,
        'auto_save_to_drive': bool(auto_save),
        'drive_output_folder': (drive_output_folder or '').strip(),
        'dictionary': dict_settings,
        'jobs': jobs_list
    }
    (work / 'job_queue.json').write_text(json.dumps(queue_data, ensure_ascii=False, indent=2), encoding='utf-8')
    if jobs_list:
        first_job = jobs_list[0]
        (work / 'output_stem.txt').write_text(first_job['stem'], encoding='utf-8')
        target_media = work / 'input.media'
        src_p = Path(first_job['source_path'])
        if src_p.exists() and src_p != target_media:
            if target_media.exists() or target_media.is_symlink():
                target_media.unlink()
            try:
                target_media.symlink_to(src_p)
            except Exception:
                shutil.copy2(src_p, target_media)


def scan_drive_media(root_dir, max_depth=4):
    found = []
    root_path = Path(root_dir)
    if not root_path.exists():
        return found

    def _walk(curr_path, depth):
        if depth > max_depth:
            return
        try:
            entries = list(os.scandir(curr_path))
        except Exception:
            return
        for entry in entries:
            if entry.name.startswith('.') or entry.name in (OUTPUT_FOLDER_NAME, CTB_MEMORY_FOLDER_NAME, 'node_modules', '__pycache__'):
                continue
            try:
                if entry.is_file(follow_symlinks=False):
                    ext = os.path.splitext(entry.name)[1].lower()
                    if ext in MEDIA_EXTS:
                        st = entry.stat(follow_symlinks=False)
                        if st.st_size > 0:
                            found.append((Path(entry.path), st.st_mtime, st.st_size))
                elif entry.is_dir(follow_symlinks=False):
                    _walk(Path(entry.path), depth + 1)
            except Exception:
                continue

    _walk(root_path, 1)
    found.sort(key=lambda item: ('Colab Notebooks' in str(item[0]), item[1]), reverse=True)
    return found


def scan_drive_texts(root_dir, max_depth=4):
    '''掃描 Google 雲端硬碟中的 .srt / .vtt / .txt / .md 現成文稿檔案'''
    found = []
    root_path = Path(root_dir)
    if not root_path.exists():
        return found

    def _walk(curr_path, depth):
        if depth > max_depth:
            return
        try:
            entries = list(os.scandir(curr_path))
        except Exception:
            return
        for entry in entries:
            if entry.name.startswith('.') or entry.name in (CTB_MEMORY_FOLDER_NAME, 'node_modules', '__pycache__'):
                continue
            try:
                if entry.is_file(follow_symlinks=False):
                    ext = os.path.splitext(entry.name)[1].lower()
                    if ext in TEXT_EXTS and not entry.name.endswith('_原始未校對.srt'):
                        st = entry.stat(follow_symlinks=False)
                        if st.st_size > 0:
                            found.append((Path(entry.path), st.st_mtime, st.st_size))
                elif entry.is_dir(follow_symlinks=False):
                    _walk(Path(entry.path), depth + 1)
            except Exception:
                continue

    _walk(root_path, 1)
    found.sort(key=lambda item: ('Colab Notebooks' in str(item[0]), item[1]), reverse=True)
    return found


def register_direct_text_job(raw_text, filename_or_title, audio_parent_dir=None):
    '''
    將現成 SRT / VTT / TXT 文稿直接解析並建立步驟 4 所需的全部標準工作檔案（raw_segments.json 等），
    讓使用者可以 100% 跳過步驟 1 與步驟 3，直接執行步驟 4！
    '''
    p_obj = Path(filename_or_title)
    ext = p_obj.suffix.lower() if p_obj.suffix.lower() in TEXT_EXTS else '.txt'
    stem = (p_obj.stem if p_obj.suffix.lower() in TEXT_EXTS else filename_or_title).strip() or '現成文稿創作'
    safe_stem = re.sub(r'[\/:*?"<>|]+', '_', stem)

    cues_tuples, converted_segs, rule_hits, has_real_ts, duration_sec = parse_existing_text_or_srt(
        raw_text, ext=ext, replace_rules=dict_settings['replace_rules']
    )
    txt_lines = [t for _, _, t in cues_tuples]
    outputs = build_outputs_from_cues(cues_tuples, converted_segs, intro_music_seconds=0, txt_lines=txt_lines)

    ep_save_dir = resolve_job_drive_save_dir(safe_stem, audio_parent_dir=audio_parent_dir)
    job_id = 'job_001'
    j_dir = work / 'jobs' / job_id
    j_dir.mkdir(parents=True, exist_ok=True)

    for out_name, out_content in outputs.items():
        (j_dir / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (j_dir / out_name).write_text(out_content, encoding='utf-8')
        (work / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (work / out_name).write_text(out_content, encoding='utf-8')

    raw_payload = {
        'job_id': job_id,
        'filename': f'{safe_stem}{ext}',
        'stem': safe_stem,
        'language': 'zh',
        'duration': duration_sec,
        'source_mode': 'direct_text',
        'has_real_timestamps': bool(has_real_ts),
        'youtube_id': global_yt_id,
        'youtube_url': youtube_url.strip(),
        'drive_save_dir': ep_save_dir,
        'deterministic_rule_hits': rule_hits,
        'subtitle_settings': {
            'auto_split': False,
            'auto_punctuation': False,
            'max_subtitle_chars': 22,
            'max_subtitle_seconds': 6.0,
            'intro_music_seconds': 0.0,
            'timing_methods': ['imported_timestamps' if has_real_ts else 'estimated_reading_speed']
        },
        'cues': [
            {'start_ms': s, 'end_ms': e, 'text': t}
            for s, e, t in cues_tuples
        ],
        'segments': converted_segs
    }
    raw_json_str = json.dumps(raw_payload, ensure_ascii=False, indent=2)
    (j_dir / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (j_dir / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'output_stem.txt').write_text(safe_stem, encoding='utf-8')

    queue_data = {
        'mode': 'direct_text',
        'skip_transcription': True,
        'auto_save_to_drive': bool(auto_save_to_drive),
        'drive_output_folder': (drive_output_folder or '').strip(),
        'dictionary': dict_settings,
        'jobs': [{
            'job_id': job_id,
            'filename': f'{safe_stem}{ext}',
            'stem': safe_stem,
            'source_path': str(j_dir / ('raw_transcript.srt' if has_real_ts else 'raw_transcript.txt')),
            'youtube_id': global_yt_id,
            'youtube_url': youtube_url.strip(),
            'drive_save_dir': ep_save_dir,
            'skip_transcription': True,
            'has_real_timestamps': bool(has_real_ts)
        }]
    }
    (work / 'job_queue.json').write_text(json.dumps(queue_data, ensure_ascii=False, indent=2), encoding='utf-8')
    return raw_payload



def execute_selected_source_mode(curr_source_type):
    '''根據目前生效的來源模式，在 dynamic_source_out 中渲染對應的來源選擇器與工作佇列'''
    dynamic_source_out.clear_output(wait=True)
    with dynamic_source_out:
        if 'YouTube' in curr_source_type:
            yt_box_input = widgets.Text(
                value=youtube_url.strip(),
                placeholder='請貼上 YouTube 影片或直播網址（多支網址可用逗號 , 分隔）',
                description='🎬 YT 網址：',
                style={'description_width': '95px'},
                layout=widgets.Layout(width='560px')
            )
            yt_dl_btn = widgets.Button(
                description='🚀 下載 YouTube 音軌並綁定影片 ID',
                button_style='success',
                layout=widgets.Layout(width='260px', margin='0 0 0 8px')
            )
            yt_out_area = widgets.Output()

            def _run_yt_download(raw_u_str):
                global global_yt_id
                yt_out_area.clear_output(wait=True)
                raw_urls = [u.strip() for u in re.split(r'[,，]+', (raw_u_str or '').replace(chr(10), ',')) if u.strip()]
                if not raw_urls:
                    wait_step2.value = '<div style="padding:10px 14px; border:2px solid #e65100; background:#fff3e0; border-radius:8px; color:#bf360c;"><b>⚠️ 請在下方輸入框貼上 YouTube 影片網址後點擊「🚀 下載 YouTube 音軌並綁定影片 ID」！</b></div>'
                    return
                with yt_out_area:
                    if auto_save_to_drive:
                        ensure_drive_mounted_if_needed()
                    yt_dir = work / 'yt_downloads'
                    yt_dir.mkdir(parents=True, exist_ok=True)
                    jobs_list = []
                    for idx, u in enumerate(raw_urls, 1):
                        yt_id = extract_youtube_id(u)
                        full_url = f'https://www.youtube.com/watch?v={yt_id}' if (yt_id and not u.startswith('http')) else u
                        wait_step2.value = f'🎬 正在從 YouTube 取得影片資訊並下載音軌（第 {idx}/{len(raw_urls)} 支：{full_url}）...'
                        print(f'🎬 [{idx}/{len(raw_urls)}] 正在下載 YouTube 音軌：{full_url} ...', flush=True)
                        probe = subprocess.run(
                            ['yt-dlp', '--no-playlist', '--print', '%(id)s|||%(title)s', full_url],
                            capture_output=True, text=True, check=False
                        )
                        vid_id = yt_id or f'yt_{idx:03d}'
                        vid_title = f'YouTube_{vid_id}'
                        if probe.returncode == 0 and '|||' in probe.stdout:
                            first_line = probe.stdout.strip().splitlines()[0]
                            p_id, p_title = first_line.split('|||', 1)
                            if p_id.strip():
                                vid_id = p_id.strip()
                            if p_title.strip():
                                vid_title = re.sub(r'[\/:*?"<>|]+', '_', p_title.strip())
                        out_tmpl = str(yt_dir / f'{vid_id}.%(ext)s')
                        subprocess.run(['yt-dlp', '--no-playlist', '-f', 'bestaudio[ext=m4a]/bestaudio/best', '-o', out_tmpl, full_url], check=True)
                        downloaded_files = list(yt_dir.glob(f'{vid_id}.*'))
                        if not downloaded_files:
                            raise RuntimeError(f'YouTube 音軌下載失敗：{full_url}')
                        media_file = downloaded_files[0]
                        ep_save_dir = resolve_job_drive_save_dir(vid_title)
                        jobs_list.append({
                            'job_id': f'job_{idx:03d}',
                            'filename': f'{vid_title}{media_file.suffix}',
                            'stem': vid_title,
                            'source_path': str(media_file),
                            'youtube_id': vid_id,
                            'youtube_url': f'https://www.youtube.com/watch?v={vid_id}',
                            'drive_save_dir': ep_save_dir
                        })
                        print(f'✅ 已下載完成：{vid_title} (YouTube ID: {vid_id})', flush=True)
                    mode_str = 'batch' if len(jobs_list) > 1 else 'single'
                    save_job_queue(mode_str, jobs_list, auto_save=auto_save_to_drive)
                    wait_step2.value = (
                        '<div style="padding:12px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:8px; color:#1b5e20;">'
                        f'<b>✅ 階段 2/6 完成：已成功下載 {len(jobs_list)} 支 YouTube 音軌！</b><br>'
                        f'<b>🎬 影片標題：</b><code>{jobs_list[0]["stem"]}</code>（已自動綁定 YouTube ID <code>{jobs_list[0]["youtube_id"]}</code>）<br>'
                        f'<b>💾 雲端專屬儲存子資料夾：</b><code>{jobs_list[0]["drive_save_dir"] or "未開啟自動存回雲端"}</code><br>'
                        '👉 <b>請直接執行「步驟 3」開始語音辨識。</b>'
                        '</div>'
                    )

            yt_dl_btn.on_click(lambda _: _run_yt_download(yt_box_input.value))
            display(widgets.HBox([yt_box_input, yt_dl_btn]), yt_out_area)
            if youtube_url.strip():
                _run_yt_download(youtube_url.strip())
            else:
                wait_step2.value = '🎬 <b>請在下方貼上 YouTube 影片或直播網址，並點擊「🚀 下載 YouTube 音軌並綁定影片 ID」：</b>'

        elif '現成 SRT/VTT/TXT 文稿直通（本機上傳' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            from google.colab import files
            wait_step2.value = '📝 <b>請點擊下方「選擇檔案」從電腦上傳您的 .srt、.vtt、.txt 或 .md 文稿檔案：</b>'
            print('📝 請點擊下方「選擇檔案」上傳您的 .srt、.vtt、.txt 或 .md 文稿檔案：', flush=True)
            uploaded = files.upload()
            if not uploaded:
                wait_step2.value = '⚠️ 尚未選取任何文稿檔案，請重新選擇或切換其他文稿來源。'
            else:
                doc_filename = next(iter(uploaded.keys()))
                raw_bytes = uploaded[doc_filename]
                if Path(doc_filename).exists():
                    try:
                        Path(doc_filename).unlink()
                    except Exception:
                        pass
                raw_str = ''
                for enc in ('utf-8-sig', 'utf-8', 'cp950', 'big5', 'utf-16'):
                    try:
                        raw_str = raw_bytes.decode(enc)
                        break
                    except Exception:
                        continue
                if not raw_str:
                    raw_str = raw_bytes.decode('utf-8', errors='replace')
                info = register_direct_text_job(raw_str, doc_filename)
                ts_badge = '🕒 偵測為含時間碼字幕（100% 保留原始時間軸）' if info['has_real_timestamps'] else '📄 偵測為純文字逐字稿（已智慧切分句子）'
                wait_step2.value = (
                    '<div style="padding:14px 16px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
                    f'<b>⚡ 【現成文稿直通模式】準備完成！已載入：<code>{doc_filename}</code></b><br>'
                    f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                    f'💾 <b>雲端自動存檔專屬資料夾：</b><code>{info["drive_save_dir"] or "未開啟"}</code><br>'
                    '<div style="margin-top:8px; padding:8px 12px; background:#fff8e1; border-left:4px solid #ffa000; border-radius:6px; color:#e65100; font-weight:bold;">'
                    '🚀 您無須執行「步驟 1」與「步驟 3（語音辨識）」！請直接往下捲動點擊【步驟 4】開始 AI 全媒體創作與推送 Blogger！'
                    '</div></div>'
                )

        elif '現成 SRT/VTT/TXT 文稿直通（Google 雲端硬碟' in curr_source_type:
            my_drive_root = ensure_drive_mounted_if_needed()
            wait_step2.value = '🔍 正在掃描您的 Google 雲端硬碟中的 .srt / .vtt / .txt / .md 文稿檔案...'
            text_candidates = scan_drive_texts(my_drive_root, max_depth=4)
            if not text_candidates:
                wait_step2.value = '⚠️ 在雲端硬碟中找不到 .srt / .vtt / .txt 文稿檔案，請於上方「📝 軌跡 B 來源」切換為「本機上傳」或「直接貼上純文字」模式。'
            else:
                doc_opts = []
                for f_path, _, f_size in text_candidates[:250]:
                    try:
                        rel = str(f_path.relative_to(my_drive_root))
                    except ValueError:
                        rel = str(f_path)
                    kb = f_size / 1024.0
                    doc_opts.append((f'📝 {rel} ({kb:.1f} KB)', str(f_path)))
                doc_dd = widgets.Dropdown(
                    options=doc_opts,
                    value=doc_opts[0][1],
                    description='選擇雲端文稿：',
                    style={'description_width': '110px'},
                    layout=widgets.Layout(width='95%')
                )
                doc_status = widgets.HTML()

                def apply_doc_selection(path_str):
                    p = Path(path_str)
                    raw_str = ''
                    raw_bytes = p.read_bytes()
                    for enc in ('utf-8-sig', 'utf-8', 'cp950', 'big5', 'utf-16'):
                        try:
                            raw_str = raw_bytes.decode(enc)
                            break
                        except Exception:
                            continue
                    if not raw_str:
                        raw_str = raw_bytes.decode('utf-8', errors='replace')
                    info = register_direct_text_job(raw_str, p.name, audio_parent_dir=p.parent)
                    ts_badge = '🕒 含時間碼字幕（100% 保留原始時間軸）' if info['has_real_timestamps'] else '📄 純文字逐字稿（已智慧切分句子）'
                    doc_status.value = (
                        '<div style="margin-top:8px; padding:14px 16px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
                        f'<b>⚡ 已載入雲端現成文稿：</b><code>{p}</code><br>'
                        f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                        f'💾 <b>雲端成果專屬子資料夾：</b><code>{info["drive_save_dir"]}</code><br>'
                        '<div style="margin-top:8px; padding:8px 12px; background:#fff8e1; border-left:4px solid #ffa000; border-radius:6px; color:#e65100; font-weight:bold;">'
                        '🚀 您無須執行「步驟 3（語音辨識）」！請直接前往【步驟 4】開始 AI 全媒體創作與推送 Blogger！'
                        '</div></div>'
                    )

                doc_dd.observe(lambda ch: apply_doc_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_doc_selection(doc_opts[0][1])
                wait_step2.value = f'✅ 已掃描到 <b>{len(doc_opts)}</b> 份雲端文稿檔案，請由下方選單挑選：'
                display(widgets.VBox([doc_dd, doc_status]))

        elif '直接貼上純文字' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            paste_title_input = widgets.Text(
                value='我的專欄文稿與筆記',
                placeholder='請輸入這篇作品的標題或檔名（將作為雲端專屬子資料夾名稱）',
                description='📌 作品名稱：',
                style={'description_width': '100px'},
                layout=widgets.Layout(width='500px')
            )
            paste_area = widgets.Textarea(
                value='',
                placeholder='請在此直接貼上您的 .srt 字幕內容、會議逐字稿、演講筆記或文章草稿...',
                description='✍️ 貼上文稿：',
                style={'description_width': '100px'},
                layout=widgets.Layout(width='95%', height='220px')
            )
            paste_save_btn = widgets.Button(
                description='⚡ 確認載入文稿並直通【步驟 4】',
                button_style='success',
                layout=widgets.Layout(width='280px', margin='8px 0 0 105px')
            )
            paste_status = widgets.HTML()

            def on_click_paste_save(_):
                body = paste_area.value.strip()
                if not body:
                    paste_status.value = '<span style="color:#d32f2f; font-weight:bold;">❌ 請先在上方文字框貼上文稿內容！</span>'
                    return
                title_str = paste_title_input.value.strip() or '手動貼上文稿'
                info = register_direct_text_job(body, title_str)
                ts_badge = '🕒 偵測為含時間碼 SRT/VTT 字幕' if info['has_real_timestamps'] else '📄 偵測為純文字文稿（已智慧分句）'
                paste_status.value = (
                    '<div style="margin-top:8px; padding:14px 16px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:10px; color:#1b5e20; line-height:1.75;">'
                    f'<b>✅ 文稿已成功載入並完成詞庫清洗！作品名稱：<code>{info["stem"]}</code></b><br>'
                    f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                    f'💾 <b>雲端成果專屬子資料夾：</b><code>{info["drive_save_dir"]}</code><br>'
                    '👉 <b>請直接略過步驟 3，立刻前往【步驟 4】開始 AI 全媒體創作！</b>'
                    '</div>'
                )

            paste_save_btn.on_click(on_click_paste_save)
            wait_step2.value = '✍️ <b>請在下方輸入作品名稱並貼上您的 SRT 字幕或 TXT 純文字逐字稿，再按「⚡ 確認載入文稿並直通【步驟 4】」：</b>'
            display(widgets.VBox([paste_title_input, paste_area, paste_save_btn, paste_status]))

        elif '本機上傳' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            from google.colab import files
            wait_step2.value = '💻 <b>請點擊下方「選擇檔案」從電腦上傳音訊或視訊檔案：</b>'
            print('階段 2/6：請點擊下方「選擇檔案」上傳音訊或視訊檔案（若超過 100MB 建議改用 Google 雲端硬碟模式）：', flush=True)
            uploaded = files.upload()
            if not uploaded:
                wait_step2.value = '⚠️ 尚未選取任何本機檔案，請重新選擇或切換來源模式。'
            else:
                audio_filename = next(iter(uploaded.keys()))
                uploaded_file = Path(audio_filename)
                target_media = work / 'input.media'
                if uploaded_file.exists():
                    if target_media.exists() or target_media.is_symlink():
                        target_media.unlink()
                    shutil.move(str(uploaded_file), str(target_media))
                else:
                    target_media.write_bytes(uploaded[audio_filename])
                del uploaded
                output_stem = Path(audio_filename).stem or '轉錄結果'
                ep_save_dir = resolve_job_drive_save_dir(output_stem)
                save_job_queue('single', [{
                    'job_id': 'job_001',
                    'filename': audio_filename,
                    'stem': output_stem,
                    'source_path': str(target_media),
                    'youtube_id': global_yt_id,
                    'youtube_url': youtube_url.strip(),
                    'drive_save_dir': ep_save_dir
                }], auto_save=bool(ep_save_dir))
                save_msg = f'<br>💾 <b>雲端自動存檔專屬資料夾：</b><code>{ep_save_dir}</code>' if ep_save_dir else ''
                wait_step2.value = f'✅ 階段 2/6 完成：本機音檔已準備（<b>{audio_filename}</b>）。{save_msg}<br>👉 請接續執行步驟 3。'

        elif curr_source_type.startswith('Google 雲端硬碟'):
            my_drive_root = ensure_drive_mounted_if_needed()
            wait_step2.value = '🔍 正在掃描您的 Google 雲端硬碟音訊與視訊檔案...'
            raw_path = drive_path.strip().strip("'").strip('"')
            user_target = None
            if raw_path:
                candidate_p = Path(raw_path)
                if not candidate_p.is_absolute():
                    candidate_p = my_drive_root / candidate_p
                elif str(candidate_p).startswith('/MyDrive/'):
                    candidate_p = Path('/content/drive') / str(candidate_p).lstrip('/')
                if candidate_p.exists():
                    user_target = candidate_p
            all_media = scan_drive_media(my_drive_root, max_depth=4)
            if user_target and user_target.is_dir():
                extra = scan_drive_media(user_target, max_depth=3)
                seen_paths = {p for p, _, _ in all_media}
                for item in extra:
                    if item[0] not in seen_paths:
                        all_media.insert(0, item)
            elif user_target and user_target.is_file():
                st = user_target.stat()
                if all(p != user_target for p, _, _ in all_media):
                    all_media.insert(0, (user_target, st.st_mtime, st.st_size))

            if not all_media:
                wait_step2.value = (
                    '<div style="padding:12px; border:2px solid #f57c00; background:#fff8e1; border-radius:8px;">'
                    '<b>⚠️ 在您的雲端硬碟中尚未掃描到音訊或影片檔案（.m4a / .mp3 / .wav / .mp4 等）。</b>'
                    '</div>'
                )
            elif '單一音檔' in curr_source_type:
                options_list = []
                default_val = None
                for f_path, mtime, f_size in all_media:
                    try:
                        rel_label = str(f_path.relative_to(my_drive_root))
                    except ValueError:
                        rel_label = str(f_path)
                    size_mb = f_size / (1024 * 1024)
                    options_list.append((f'🎵 {rel_label}  ({size_mb:.1f} MB)', str(f_path)))
                    if user_target and user_target.is_file() and f_path.resolve() == user_target.resolve():
                        default_val = str(f_path)
                if default_val is None:
                    default_val = options_list[0][1]

                file_dropdown = widgets.Dropdown(
                    options=options_list,
                    value=default_val,
                    description='選擇雲端音檔：',
                    style={'description_width': '110px'},
                    layout=widgets.Layout(width='95%')
                )
                status_box = widgets.HTML()

                def apply_single_selection(selected_path_str):
                    sel_path = Path(selected_path_str)
                    stem = sel_path.stem or '轉錄結果'
                    ep_out_dir = resolve_job_drive_save_dir(stem, audio_parent_dir=sel_path.parent)
                    save_job_queue('single', [{
                        'job_id': 'job_001',
                        'filename': sel_path.name,
                        'stem': stem,
                        'source_path': str(sel_path),
                        'youtube_id': global_yt_id,
                        'youtube_url': youtube_url.strip(),
                        'drive_save_dir': ep_out_dir
                    }], auto_save=auto_save_to_drive)
                    save_hint = f'開啟（將自動存至專屬同名子資料夾 <code>{ep_out_dir}/</code>）' if auto_save_to_drive else '關閉（僅於步驟 5 下載 ZIP）'
                    yt_hint = f'<br><b>🎬 已綁定 YouTube ID：</b><code>{global_yt_id}</code>' if global_yt_id else ''
                    status_box.value = (
                        '<div style="margin-top:8px; padding:12px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:8px; color:#1b5e20;">'
                        f'<b>✅ 目前已選取音檔：</b><code>{sel_path.name}</code><br>'
                        f'<b>📂 雲端所在位置：</b><code>{sel_path.parent}</code><br>'
                        f'<b>💾 自動存回雲端：</b>{save_hint}{yt_hint}<br>'
                        f'<b>📖 已自動載入雲端永久記憶詞庫：</b>錯字替換 <b>{len(dict_settings["replace_rules"])}</b> 組、🟢 正向專有名詞 <b>{len(dict_settings["positive_terms"])}</b> 個<br>'
                        '👉 <b>確認無誤後請直接點擊下方「步驟 3」開始轉錄！</b>'
                        '</div>'
                    )

                file_dropdown.observe(lambda ch: apply_single_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_single_selection(default_val)
                wait_step2.value = f'✅ 階段 2/6 完成：已在雲端硬碟掃描到 <b>{len(options_list)}</b> 個音訊/視訊檔案，請由下方選單確認或切換：'
                display(widgets.VBox([file_dropdown, status_box]))

            else:
                folder_map = {}
                for f_path, mtime, f_size in all_media:
                    folder_map.setdefault(f_path.parent, []).append((f_path, f_size))
                folder_options = []
                default_folder = None
                for folder_path, files_in_folder in folder_map.items():
                    try:
                        rel_folder = str(folder_path.relative_to(my_drive_root))
                        if rel_folder == '.':
                            rel_folder = '我的雲端硬碟 (根目錄)'
                    except ValueError:
                        rel_folder = str(folder_path)
                    total_mb = sum(sz for _, sz in files_in_folder) / (1024 * 1024)
                    folder_options.append((f'📁 {rel_folder}  （共 {len(files_in_folder)} 個音檔，合計 {total_mb:.1f} MB）', str(folder_path)))
                    if user_target and user_target.is_dir() and folder_path.resolve() == user_target.resolve():
                        default_folder = str(folder_path)
                if default_folder is None:
                    default_folder = folder_options[0][1]

                folder_dropdown = widgets.Dropdown(
                    options=folder_options,
                    value=default_folder,
                    description='選擇批次資料夾：',
                    style={'description_width': '120px'},
                    layout=widgets.Layout(width='95%')
                )
                batch_status_box = widgets.HTML()

                def apply_folder_selection(folder_str):
                    f_dir = Path(folder_str)
                    candidates = folder_map.get(f_dir, [])
                    jobs_list = []
                    skipped_names = []
                    sample_parent_out = ''
                    for idx, (f_path, f_size) in enumerate(candidates, 1):
                        stem = f_path.stem or f'音檔_{idx}'
                        ep_out_dir = resolve_job_drive_save_dir(stem, audio_parent_dir=f_dir)
                        if ep_out_dir and not sample_parent_out:
                            sample_parent_out = str(Path(ep_out_dir).parent)
                        if skip_already_transcribed and ep_out_dir and (Path(ep_out_dir) / f'{stem}.srt').is_file():
                            skipped_names.append(f_path.name)
                            continue
                        jobs_list.append({
                            'job_id': f'job_{idx:03d}',
                            'filename': f_path.name,
                            'stem': stem,
                            'source_path': str(f_path),
                            'youtube_id': '',
                            'youtube_url': '',
                            'drive_save_dir': ep_out_dir
                        })
                    save_job_queue('batch', jobs_list, auto_save=auto_save_to_drive)
                    file_items_html = ''.join(f'<li><code>{j["filename"]}</code> ➔ 存至子資料夾 <code>{j["stem"]}/</code></li>' for j in jobs_list[:15])
                    if len(jobs_list) > 15:
                        file_items_html += f'<li>...及其他 {len(jobs_list) - 15} 個音檔</li>'
                    skip_html = f'<br>⏭️ 已略過雲端既有成果：{len(skipped_names)} 個檔案' if skipped_names else ''
                    batch_status_box.value = (
                        '<div style="margin-top:8px; padding:12px; border:2px solid #1565c0; background:#e3f2fd; border-radius:8px; color:#0d47a1;">'
                        f'<b>📦 已選取整批轉錄資料夾：</b><code>{f_dir}</code><br>'
                        f'<b>🎵 待自動轉錄音檔數：</b>共 <b>{len(jobs_list)}</b> 個音檔{skip_html}<br>'
                        f'<b>💾 自動存回雲端母資料夾：</b><code>{sample_parent_out}/{{音檔同名資料夾}}/</code><br>'
                        f'<ul style="margin:6px 0 6px 20px;">{file_items_html}</ul>'
                        '👉 <b>確認資料夾後，請依序按「步驟 3」與「步驟 4」！</b>'
                        '</div>'
                    )

                folder_dropdown.observe(lambda ch: apply_folder_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_folder_selection(default_folder)
                wait_step2.value = f'✅ 階段 2/6 完成：已掃描到 <b>{len(folder_options)}</b> 個包含音檔的雲端資料夾，請由下方選單確認或切換：'
                display(widgets.VBox([folder_dropdown, batch_status_box]))



dynamic_source_out = widgets.Output()
stepper_box_s2.value = render_stepper_banner(2, '🎧 音檔／YouTube 轉錄軌')
display(dynamic_source_out)
render_ctb_memory_panel()
execute_selected_source_mode(source_type)

## <font color="#00897b"><b>2B. 現成 SRT / TXT 匯入（免開 GPU，完成後直接跳步驟 4）</b></font>

In [ ]:
# @title 🟦 **2B. 現成 SRT / TXT 匯入（免開 GPU，完成後直接跳步驟 4）**
# @markdown ### 📝 現成 SRT / VTT / TXT 文稿來源設定
現成文稿匯入方式 = "📝 現成 SRT/VTT/TXT 文稿直通（本機上傳 — 免跑步驟 1 與 3！）" # @param ["📝 現成 SRT/VTT/TXT 文稿直通（Google 雲端硬碟下拉選單 — 免跑步驟 1 與 3！）", "📝 現成 SRT/VTT/TXT 文稿直通（本機上傳 — 免跑步驟 1 與 3！）", "✍️ 直接貼上純文字／會議筆記／逐字稿（免跑步驟 1 與 3！）"]
嵌入Blog頂部之YouTube網址_選填 = "https://youtu.be/6jDYVh9QFz8" # @param {type:"string"}
自動將成果存回雲端硬碟 = True # @param {type:"boolean"}
雲端成果儲存母資料夾 = "Colab Notebooks/轉錄成果_ㄚ亮笑長助手" # @param {type:"string"}

# @markdown ### 📖 錯字替換與專有名詞詞庫（雲端永久記憶）
從電腦上傳匯入詞庫JSON設定檔 = False # @param {type:"boolean"}
錯字替換規則 = "阿亮校長->ㄚ亮笑長, 阿亮笑長->ㄚ亮笑長, 丫亮校長->ㄚ亮笑長, 阿林校長->ㄚ亮笑長, 思辯->思辨, 居酒屋外->居久屋, 瀑瀑聊聊->噗噗聊聊, 千億科技->謙懿科技, 捷可達->傑可達, 換名->冠名" # @param {type:"string"}
正向專有名詞 = "阿剛, 阿倫,ㄚ亮笑長, 噗噗聊聊, 居久屋微醺夜, 居久屋課程筆記, 謙懿科技, 傑可達數位, T++教育+年華, 數位科技思辨課, NotebookLM, Gemini, 阿剛, 阿倫" # @param {type:"string"}
禁用或常見誤聽參考詞 = "" # @param {type:"string"}

text_source_type = 現成文稿匯入方式
source_type = 現成文稿匯入方式
youtube_url = 嵌入Blog頂部之YouTube網址_選填
drive_path = ""
auto_save_to_drive = 自動將成果存回雲端硬碟
drive_output_folder = 雲端成果儲存母資料夾
skip_already_transcribed = False
upload_dict_json_from_pc = 從電腦上傳匯入詞庫JSON設定檔
typo_replace_rules = 錯字替換規則
positive_terms = 正向專有名詞
negative_terms = 禁用或常見誤聽參考詞

import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

clear_output(wait=True)

work = Path('/content/alta_transcribe_work')
work.mkdir(parents=True, exist_ok=True)

cleanup_names = (
    'input.media', 'output_stem.txt', 'job_queue.json',
    'transcript.txt', 'transcript.srt', 'transcript.vtt', 'segments.json',
    'raw_transcript.txt', 'raw_transcript.srt', 'raw_transcript.vtt', 'raw_segments.json',
    'article_and_summary.md', 'article_and_summary.txt', 'chapters.txt',
    'blog_post.html', 'blog_post.md', 'titles_and_shownotes.md',
    'social_posts.md', 'visual_prompts.md'
)
for name in cleanup_names:
    p = work / name
    if p.exists() or p.is_symlink():
        try:
            p.unlink()
        except Exception:
            pass

for sub_d in ('jobs', 'yt_downloads'):
    if (work / sub_d).is_dir():
        shutil.rmtree(work / sub_d, ignore_errors=True)

for zip_file in work.glob('*.zip'):
    try:
        zip_file.unlink()
    except Exception:
        pass

MEDIA_EXTS = {'.m4a', '.mp3', '.wav', '.mp4', '.aac', '.flac', '.ogg', '.mkv', '.mov', '.webm', '.wma', '.m4v'}
TEXT_EXTS = {'.srt', '.vtt', '.txt', '.md'}
OUTPUT_FOLDER_NAME = '轉錄成果_ㄚ亮笑長助手'
CTB_MEMORY_FOLDER_NAME = 'CTB_詞庫設定_永久記憶'

# 預先寫入共用字幕與文稿解析核心模組 (p0_subtitles.py)，讓使用者就算跳過步驟 1 與步驟 3 也能直接解析 SRT/TXT！
_p0_sub_code = "# Subtitle formatting, Chinese conversion, SRT/VTT/TXT parser, and ctb.52hal.cc deterministic cleaning utilities.\nimport math\nimport re\n\ntry:\n    from opencc import OpenCC\n    _OPENCC_INSTANCE = OpenCC('s2twp')\nexcept Exception:\n    _OPENCC_INSTANCE = None\n\n\ndef to_trad_tw(text):\n    if not text:\n        return ''\n    if _OPENCC_INSTANCE is not None:\n        try:\n            return _OPENCC_INSTANCE.convert(text)\n        except Exception:\n            return text\n    return text\n\n\ndef clean_text_deterministic(text, replace_rules=None):\n    if not text:\n        return '', 0\n    s = to_trad_tw(text.strip())\n    s = re.sub(r'^[\\[\\(【（]?\\s*(?:音樂|music|Music|MUSIC|♪+|♫+)\\s*[\\]\\)】）]?$', '【音樂】', s)\n    s = re.sub(r'\\b([A-Z])(?:\\s+([A-Z]))+\\b', lambda m: re.sub(r'\\s+', '', m.group(0)), s)\n    applied_count = 0\n    for rule in (replace_rules or []):\n        orig = rule.get('original', '')\n        repl = rule.get('replacement', '')\n        if orig and orig in s:\n            applied_count += s.count(orig)\n            s = s.replace(orig, repl)\n    return s.strip(), applied_count\n\n\ndef parse_timestamp_to_ms(ts_str):\n    s = (ts_str or '').strip().replace(',', '.')\n    parts = s.split(':')\n    try:\n        if len(parts) == 3:\n            hh = int(parts[0])\n            mm = int(parts[1])\n            sec_f = float(parts[2])\n        elif len(parts) == 2:\n            hh = 0\n            mm = int(parts[0])\n            sec_f = float(parts[1])\n        else:\n            return 0\n        return max(0, int(round((hh * 3600 + mm * 60 + sec_f) * 1000)))\n    except Exception:\n        return 0\n\n\ndef parse_existing_text_or_srt(raw_content, ext='.srt', replace_rules=None):\n    '''\n    解析現成的 .srt / .vtt / .txt / .md 或是直接貼上的純文字，並自動執行確定性錯字清洗：\n    回傳：(cues_tuples, converted_segments, total_rule_hits, has_real_timestamps, duration_sec)\n    - cues_tuples: [(start_ms, end_ms, text), ...]\n    '''\n    text_body = (raw_content or '').lstrip('\\ufeff').replace('\\r\\n', '\\n').replace('\\r', '\\n').strip()\n    if not text_body:\n        raise ValueError('匯入的文稿內容為空白！')\n\n    # 自動偵測是否包含 SRT / VTT 時間碼 (00:00:00,000 --> 00:00:03,000)\n    ts_pattern = re.compile(\n        r'((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})\\s*-->\\s*((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})'\n    )\n    has_ts_arrows = bool(ts_pattern.search(text_body))\n    cues = []\n    converted_segments = []\n    total_hits = 0\n\n    if has_ts_arrows:\n        blocks = re.split(r'\\n\\s*\\n+', text_body)\n        prev_end = 0\n        for blk in blocks:\n            lines = [ln.strip() for ln in blk.strip().splitlines() if ln.strip()]\n            if not lines:\n                continue\n            if lines[0].upper().startswith('WEBVTT'):\n                lines = lines[1:]\n                if not lines:\n                    continue\n            m_ts = None\n            ts_line_idx = -1\n            for idx_l, ln in enumerate(lines[:3]):\n                m = ts_pattern.search(ln)\n                if m:\n                    m_ts = m\n                    ts_line_idx = idx_l\n                    break\n            if not m_ts:\n                continue\n            start_ms = parse_timestamp_to_ms(m_ts.group(1))\n            end_ms = parse_timestamp_to_ms(m_ts.group(2))\n            if end_ms <= start_ms:\n                end_ms = start_ms + 2000\n            text_lines = lines[ts_line_idx + 1:]\n            # 自動移除既有連帶的台呼水印，避免重複疊加\n            text_lines = [tl for tl in text_lines if tl.strip() != '【ㄚ亮笑長的 Colab轉錄助手】']\n            joined_text = ' '.join(text_lines).strip()\n            if not joined_text:\n                continue\n            cleaned_t, hits = clean_text_deterministic(joined_text, replace_rules=replace_rules)\n            total_hits += hits\n            if cleaned_t:\n                cues.append((start_ms, end_ms, cleaned_t))\n                converted_segments.append({\n                    'start': round(start_ms / 1000.0, 3),\n                    'end': round(end_ms / 1000.0, 3),\n                    'text': joined_text,\n                    'text_zh_tw': cleaned_t\n                })\n                prev_end = max(prev_end, end_ms)\n\n        if cues:\n            duration_sec = max(1.0, round(prev_end / 1000.0, 2))\n            return cues, converted_segments, total_hits, True, duration_sec\n\n    # 純文字 (.txt / .md / 直接貼上文字)：智慧依换行與句號/問號/驚嘆號切分句子，賦予預估閱讀時間軸\n    raw_lines = [ln.strip() for ln in text_body.splitlines() if ln.strip()]\n    sentences = []\n    for ln in raw_lines:\n        # 若單行很長，依全形句號、問號、驚嘆號、分號適度拆句，方便字幕與段落對齊\n        if len(ln) > 45:\n            parts = re.split(r'(?<=[。！？；!?])', ln)\n            for p in parts:\n                p_s = p.strip()\n                if p_s:\n                    sentences.append(p_s)\n        else:\n            sentences.append(ln)\n\n    curr_ms = 0\n    for sent in sentences:\n        if sent == '【ㄚ亮笑長的 Colab轉錄助手】':\n            continue\n        cleaned_t, hits = clean_text_deterministic(sent, replace_rules=replace_rules)\n        total_hits += hits\n        if not cleaned_t:\n            continue\n        # 依字數估算自然語速（約每秒 4.5 字，最短 2 秒，最長 8 秒）\n        dur_ms = max(2000, min(8000, int(round(len(cleaned_t) / 4.5 * 1000))))\n        start_ms = curr_ms\n        end_ms = curr_ms + dur_ms\n        cues.append((start_ms, end_ms, cleaned_t))\n        converted_segments.append({\n            'start': round(start_ms / 1000.0, 3),\n            'end': round(end_ms / 1000.0, 3),\n            'text': sent,\n            'text_zh_tw': cleaned_t\n        })\n        curr_ms = end_ms\n\n    if not cues:\n        raise ValueError('未能從文稿中解析出任何有效句子！')\n    duration_sec = max(1.0, round(curr_ms / 1000.0, 2))\n    return cues, converted_segments, total_hits, False, duration_sec\n\n\ndef subtitle_cues(segments, duration):\n    if not isinstance(duration, (int, float)) or not math.isfinite(duration) or duration <= 0:\n        raise ValueError('無效的音訊長度')\n    max_ms = math.floor(duration * 1000)\n    cues = []\n    previous_ms = 0\n    sorted_segments = sorted(segments, key=lambda s: s['start'])\n    for segment in sorted_segments:\n        start, end = segment['start'], segment['end']\n        if not (isinstance(start, (int, float)) and isinstance(end, (int, float)) and math.isfinite(start) and math.isfinite(end)):\n            continue\n        start_ms = max(previous_ms, round(start * 1000))\n        end_ms = min(round(end * 1000), max_ms)\n        if end_ms <= start_ms:\n            end_ms = min(start_ms + 100, max_ms)\n        if start_ms >= end_ms:\n            continue\n        text = segment['text'].strip()\n        if text:\n            cues.append((start_ms, end_ms, text))\n            previous_ms = end_ms\n    if not cues:\n        raise ValueError('未產生任何有效字幕區間')\n    return cues\n\n\ndef stamp(milliseconds, separator):\n    hours, milliseconds = divmod(milliseconds, 3600000)\n    minutes, milliseconds = divmod(milliseconds, 60000)\n    seconds, milliseconds = divmod(milliseconds, 1000)\n    return f'{hours:02}:{minutes:02}:{seconds:02}{separator}{milliseconds:03}'\n\n\ndef render(cues, kind):\n    if kind not in ('srt', 'vtt'):\n        raise ValueError('不支援的字幕格式')\n    separator = ',' if kind == 'srt' else '.'\n    blocks = []\n    for index, (start, end, text) in enumerate(cues, 1):\n        prefix = (str(index) + chr(10)) if kind == 'srt' else ''\n        time_part = stamp(start, separator) + ' --> ' + stamp(end, separator)\n        blocks.append(prefix + time_part + chr(10) + text)\n    header = ('WEBVTT' + chr(10) + chr(10)) if kind == 'vtt' else ''\n    return header + (chr(10) + chr(10)).join(blocks) + chr(10)\n\n\ndef traditional_segments(segments, replace_rules=None):\n    out = []\n    total_replaced = 0\n    for segment in segments:\n        tw_text = to_trad_tw(segment['text'])\n        cleaned_text, count = clean_text_deterministic(tw_text, replace_rules=replace_rules)\n        total_replaced += count\n        out.append({**segment, 'text_zh_tw': cleaned_text})\n    return out, total_replaced\n\n\ndef _timed_chars(segment):\n    text = segment['text']\n    words = segment.get('words') or []\n    words_clean = re.sub(r'\\s+', '', ''.join(w['word'] for w in words))\n    text_clean = re.sub(r'\\s+', '', text)\n    if words_clean == text_clean and len(words) > 0 and all(\n        isinstance(w.get('start'), (int, float)) and isinstance(w.get('end'), (int, float))\n        and math.isfinite(w['start']) and math.isfinite(w['end'])\n        and segment['start'] <= w['start'] < w['end'] <= segment['end']\n        for w in words\n    ):\n        result = []\n        for word in words:\n            value = word['word'].strip()\n            if not value:\n                continue\n            span = word['end'] - word['start']\n            for i, ch in enumerate(value):\n                c_start = word['start'] + span * (i / len(value))\n                c_end = word['start'] + span * ((i + 1) / len(value))\n                result.append((ch, c_start, c_end))\n        if result:\n            return result, 'word_timestamps'\n\n    span = max(0.01, segment['end'] - segment['start'])\n    return [(ch, segment['start'] + span * i / len(text),\n             segment['start'] + span * (i + 1) / len(text))\n            for i, ch in enumerate(text)], 'segment_interpolated'\n\n\ndef build_outputs_from_cues(cues, converted_segments, intro_music_seconds=0, txt_lines=None):\n    srt_cues = list(cues)\n    vtt_cues = list(cues)\n\n    first_start, first_end, first_text = srt_cues[0]\n    promo_text = '【ㄚ亮笑長的 Colab轉錄助手】'\n\n    if intro_music_seconds > 0:\n        music_ms = round(intro_music_seconds * 1000)\n        if music_ms <= first_start:\n            srt_cues.insert(0, (0, music_ms, '【音樂】' + chr(10) + promo_text))\n            vtt_cues.insert(0, (0, music_ms, '【音樂】'))\n        else:\n            if promo_text not in first_text:\n                srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n    else:\n        if promo_text not in first_text:\n            srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n\n    if txt_lines is None:\n        txt_lines = [row['text_zh_tw'] for row in converted_segments]\n\n    outputs = {\n        'transcript.txt': chr(10).join(txt_lines) + chr(10),\n        'transcript.srt': render(srt_cues, 'srt'),\n        'transcript.vtt': render(vtt_cues, 'vtt')\n    }\n    return outputs\n\n\ndef friendly_exports(segments, duration, split=True, punctuation=False, max_chars=22, max_seconds=6, intro_music_seconds=0, replace_rules=None):\n    if max_chars < 4 or max_seconds <= 0:\n        raise ValueError('無效的字幕長度設定')\n    display = []\n    timing_methods = set()\n    total_rule_hits = 0\n    for index, segment in enumerate(segments):\n        raw_text = segment['text'].strip()\n        if not raw_text:\n            continue\n        chars, method = _timed_chars(segment)\n        if segment['end'] > duration and segment['start'] < duration:\n            scale = (duration - segment['start']) / (segment['end'] - segment['start'])\n            chars = [(char, segment['start'] + (start - segment['start']) * scale,\n                      segment['start'] + (end - segment['start']) * scale)\n                     for char, start, end in chars]\n            method = 'segment_end_adjusted'\n        timing_methods.add(method)\n        start = 0\n        while start < len(chars):\n            end = start + 1\n            limit_break = False\n            while end < len(chars):\n                candidate = ''.join(c[0] for c in chars[start:end + 1])\n                if split and (len(to_trad_tw(candidate)) > max_chars or\n                              chars[end][2] - chars[start][1] > max_seconds):\n                    limit_break = True\n                    break\n                if split and chars[end - 1][0] in '。！？；!?;':\n                    break\n                if split and punctuation and chars[end][1] - chars[end - 1][2] >= 0.65:\n                    break\n                end += 1\n            if (limit_break and end == len(chars) - 1 and end - start > 1\n                    and chars[end - 1][0].isalnum() and chars[end][0].isalnum()\n                    and chars[end][1] - chars[end - 1][2] < 0.65\n                    and chars[end][2] - chars[end - 1][1] <= max_seconds\n                    and len(to_trad_tw(chars[end - 1][0] + chars[end][0])) <= max_chars):\n                end -= 1\n            text = to_trad_tw(''.join(c[0] for c in chars[start:end])).strip()\n            text, hits = clean_text_deterministic(text, replace_rules=replace_rules)\n            total_rule_hits += hits\n            if text:\n                last_chunk = end == len(chars)\n                next_start = segments[index + 1]['start'] if index + 1 < len(segments) else duration\n                gap = next_start - segment['end'] if last_chunk else chars[end][1] - chars[end - 1][2]\n                if punctuation and gap >= 0.65 and text[-1] not in '，。！？；：、,.!?;:':\n                    text += '。'\n                cue_start, cue_end = chars[start][1], chars[end - 1][2]\n                display.append({'start': cue_start, 'end': cue_end, 'text': text})\n            start = end\n\n    cues = subtitle_cues(display, duration)\n    converted, seg_hits = traditional_segments(segments, replace_rules=replace_rules)\n    outputs = build_outputs_from_cues(cues, converted, intro_music_seconds=intro_music_seconds, txt_lines=[t for _, _, t in cues])\n    return converted, outputs, sorted(timing_methods), cues, max(total_rule_hits, seg_hits)\n"
(work / 'p0_subtitles.py').write_text(_p0_sub_code, encoding='utf-8')
if str(work) not in sys.path:
    sys.path.insert(0, str(work))
from p0_subtitles import parse_existing_text_or_srt, build_outputs_from_cues


def render_stepper_banner(active_step=2, mode_badge='語音轉錄軌'):
    '''渲染頂部 Web App 級流程導航狀態橫幅'''
    steps_info = [
        (1, '1. GPU 環境'),
        (2, '2. 來源與詞庫'),
        (3, '3. 語音轉錄'),
        (4, '4. AI 創作工作台'),
        (5, '5. 打包下載')
    ]
    pills = []
    for s_num, s_label in steps_info:
        if s_num == active_step:
            bg, fg, bd = '#ffd54f', '#004d40', '2px solid #ffffff'
            fw = 'bold'
        elif s_num < active_step:
            bg, fg, bd = 'rgba(255,255,255,0.22)', '#e0f2f1', '1px solid rgba(255,255,255,0.35)'
            fw = 'normal'
        else:
            bg, fg, bd = 'rgba(0,0,0,0.18)', '#b2dfdb', '1px solid rgba(255,255,255,0.15)'
            fw = 'normal'
        pills.append(
            f'<span style="display:inline-block; padding:4px 11px; margin:2px 4px; border-radius:999px; background:{bg}; color:{fg}; border:{bd}; font-weight:{fw}; font-size:12.5px;">{s_label}</span>'
        )
    pills_html = ' <span style="color:#80cbc4;">➔</span> '.join(pills)
    return (
        '<div style="background:linear-gradient(135deg, #004d40 0%, #00796b 100%); color:#ffffff; padding:14px 18px; border-radius:12px; margin-bottom:12px; box-shadow:0 3px 10px rgba(0,77,64,0.18);">'
        '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:8px;">'
        '<div style="font-size:17px; font-weight:bold; letter-spacing:0.5px;">🎙️ ㄚ亮笑長的 Colab 全媒體創作助手 </div>'
        f'<div style="font-size:12.5px; background:rgba(255,255,255,0.16); padding:3px 10px; border-radius:6px;">當前模式：<b>{mode_badge}</b></div>'
        '</div>'
        f'<div style="margin-top:4px;">{pills_html}</div>'
        '</div>'
    )


stepper_box_s2 = widgets.HTML(value=render_stepper_banner(2, '📝 現成文稿直通軌（免跑步驟 3）' if ('現成' in source_type or '直接貼上' in source_type) else '🎧 語音轉錄軌'))
wait_step2 = widgets.HTML(value='<b>階段 2/6：準備來源與同步雲端永久記憶詞庫...</b> <progress style="accent-color:green"></progress>')
display(stepper_box_s2, wait_step2)

def extract_youtube_id(url_or_id):
    '''從各種 YouTube 網址格式提取 11 碼影片 ID'''
    s = (url_or_id or '').strip()
    if not s:
        return ''
    if re.fullmatch(r'[A-Za-z0-9_-]{11}', s):
        return s
    m = re.search(r'(?:v=|youtu\.be/|youtube\.com/(?:embed|v|shorts|live)/)([A-Za-z0-9_-]{11})', s)
    return m.group(1) if m else ''


def ensure_drive_mounted_if_needed():
    '''確保 Google 雲端硬碟已掛載'''
    my_drive_root = Path('/content/drive/MyDrive')
    if not my_drive_root.exists():
        from google.colab import drive
        print('階段 2/6：正在掛載 Google 雲端硬碟；如出現授權畫面，請點選確認...', flush=True)
        try:
            drive.mount('/content/drive', timeout_ms=120000)
        except Exception as error:
            wait_step2.value = '❌ 雲端硬碟掛載失敗或逾時，請確認授權畫面後重試。'
            raise RuntimeError('雲端硬碟掛載失敗或逾時。') from error
    return my_drive_root


def get_ctb_memory_dir():
    '''取得雲端硬碟中的 CTB 詞庫永久記憶資料夾路徑'''
    my_drive_root = Path('/content/drive/MyDrive')
    if not my_drive_root.exists():
        return work / CTB_MEMORY_FOLDER_NAME
    colab_nb = my_drive_root / 'Colab Notebooks'
    base_d = colab_nb if colab_nb.exists() else my_drive_root
    mem_dir = base_d / CTB_MEMORY_FOLDER_NAME
    mem_dir.mkdir(parents=True, exist_ok=True)
    return mem_dir


def ingest_ctb_json_content(raw_json_str, rep_map, pos_list, neg_list):
    '''
    解析來自 ctb.52hal.cc 的 JSON 設定檔（自動識別 Batch_Replace_Rules 或 AI_Terminology_Rules 格式）
    回傳：(新增/更新的替換規則數, 新增的專有名詞數)
    '''
    added_rep = 0
    added_term = 0
    try:
        data = json.loads(raw_json_str)
    except Exception:
        return 0, 0

    items = []
    if isinstance(data, list):
        items = data
    elif isinstance(data, dict):
        for k in ('rules', 'replace_rules', 'terms', 'terminology', 'items', 'data'):
            if isinstance(data.get(k), list):
                items.extend(data[k])

    for item in items:
        if not isinstance(item, dict):
            continue
        # 情況 1：ctb.52hal.cc 錯字替換規則 {"original": "...", "replacement": "..."}
        if 'original' in item and 'replacement' in item:
            orig = str(item.get('original') or '').strip()
            repl = str(item.get('replacement') or '').strip()
            if orig:
                rep_map[orig] = repl
                added_rep += 1
        # 情況 2：ctb.52hal.cc 專有名詞規則 {"type": "positive" | "negative", "term": "..."}
        elif 'term' in item:
            term_val = str(item.get('term') or '').strip()
            term_type = str(item.get('type') or 'positive').strip().lower()
            if term_val:
                if term_type == 'negative':
                    if term_val not in neg_list:
                        neg_list.append(term_val)
                else:
                    if term_val not in pos_list:
                        pos_list.append(term_val)
                added_term += 1
    return added_rep, added_term


def load_and_sync_ctb_dictionaries():
    '''
    完整執行「雲端永久記憶自動載入 ➔ 電腦上傳新 JSON 匯入 ➔ 表單文字合併 ➔ 自動存回雲端永久記憶」
    '''
    if auto_save_to_drive or ('Google 雲端硬碟' in source_type) or upload_dict_json_from_pc:
        ensure_drive_mounted_if_needed()

    mem_dir = get_ctb_memory_dir()
    mem_dir.mkdir(parents=True, exist_ok=True)
    rep_json_path = mem_dir / 'Batch_Replace_Rules.json'
    term_json_path = mem_dir / 'AI_Terminology_Rules.json'

    rep_map = {}
    pos_list = []
    neg_list = []
    loaded_sources = []

    # 1. 自動掃描並載入雲端硬碟永久記憶資料夾 (以及 Colab Notebooks 根目錄下任何 CTB JSON 檔案)
    candidate_json_files = []
    if rep_json_path.is_file():
        candidate_json_files.append(rep_json_path)
    if term_json_path.is_file():
        candidate_json_files.append(term_json_path)
    for extra_j in sorted(mem_dir.glob('*.json')):
        if extra_j not in candidate_json_files:
            candidate_json_files.append(extra_j)
    if mem_dir.parent.exists():
        for root_j in sorted(mem_dir.parent.glob('*Rules*.json')):
            if root_j not in candidate_json_files:
                candidate_json_files.append(root_j)

    for jf in candidate_json_files:
        try:
            r_cnt, t_cnt = ingest_ctb_json_content(jf.read_text(encoding='utf-8'), rep_map, pos_list, neg_list)
            if r_cnt > 0 or t_cnt > 0:
                loaded_sources.append(f'{jf.name} (替換:{r_cnt}/名詞:{t_cnt})')
        except Exception:
            pass

    # 2. 若使用者勾選「從電腦上傳匯入 ctb.52hal.cc 的 JSON 設定檔」
    if upload_dict_json_from_pc:
        from google.colab import files
        print('📥 請點擊下方「選擇檔案」從電腦選取詞庫 JSON 設定檔（可同時多選 Batch_Replace_Rules*.json 與 AI_Terminology_Rules*.json）：', flush=True)
        uploaded_jsons = files.upload()
        for u_name, u_bytes in (uploaded_jsons or {}).items():
            try:
                u_text = u_bytes.decode('utf-8', errors='replace')
                r_cnt, t_cnt = ingest_ctb_json_content(u_text, rep_map, pos_list, neg_list)
                # 清理當前目錄下的上傳暫存檔
                if Path(u_name).is_file():
                    Path(u_name).unlink()
                if r_cnt > 0 or t_cnt > 0:
                    loaded_sources.append(f'電腦上傳:{u_name} (替換:{r_cnt}/名詞:{t_cnt})')
                    print(f'✅ 已成功匯入 {u_name}：新增/更新 {r_cnt} 組錯字替換、{t_cnt} 個專有名詞！', flush=True)
                else:
                    print(f'⚠️ {u_name} 未偵測到相容的詞庫欄位（original/replacement 或 type/term）。', flush=True)
            except Exception as u_err:
                print(f'⚠️ 解析 {u_name} 時發生錯誤：{u_err}', flush=True)

    # 3. 合併上方表單文字框填寫的規則（表單文字可隨時補充新詞）
    for item in re.split(r'[,，;；]+', (typo_replace_rules or '').replace(chr(10), ',')):
        item = item.strip()
        if '->' in item:
            orig, repl = item.split('->', 1)
        elif '=>' in item:
            orig, repl = item.split('=>', 1)
        else:
            continue
        if orig.strip() and orig.strip() not in rep_map:
            rep_map[orig.strip()] = repl.strip()

    for x in re.split(r'[,，;；]+', (positive_terms or '').replace(chr(10), ',')):
        val = x.strip()
        if val and val not in pos_list:
            pos_list.append(val)

    for x in re.split(r'[,，;；]+', (negative_terms or '').replace(chr(10), ',')):
        val = x.strip()
        if val and val not in neg_list:
            neg_list.append(val)

    # 4. 自動將合併後的完整詞庫以 100% ctb.52hal.cc 原生 JSON 格式寫回雲端硬碟永久記憶！
    ctb_replace_array = [{'original': k, 'replacement': v} for k, v in rep_map.items()]
    ctb_term_array = (
        [{'type': 'positive', 'term': t} for t in pos_list] +
        [{'type': 'negative', 'term': t} for t in neg_list]
    )
    try:
        rep_json_path.write_text(json.dumps(ctb_replace_array, ensure_ascii=False, indent=2), encoding='utf-8')
        term_json_path.write_text(json.dumps(ctb_term_array, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as w_err:
        print(f'⚠️ 寫入雲端詞庫永久記憶時發生警告：{w_err}', flush=True)

    return {
        'replace_rules': ctb_replace_array,
        'positive_terms': pos_list,
        'negative_terms': neg_list,
        'memory_dir': str(mem_dir),
        'loaded_sources': loaded_sources
    }


def resolve_job_drive_save_dir(stem_name, audio_parent_dir=None):
    '''
    計算該音檔在雲端硬碟的專屬儲存路徑：
    會在指定的「轉錄成果_ㄚ亮笑長助手」母資料夾下，自動建立「與音檔同名（stem_name）」的子資料夾！
    '''
    if not auto_save_to_drive:
        return ''
    my_drive_root = ensure_drive_mounted_if_needed()
    custom_out = (drive_output_folder or '').strip().strip("'").strip('"')
    if custom_out:
        p = Path(custom_out)
        if not p.is_absolute():
            base_dir = my_drive_root / p
        elif str(p).startswith('/MyDrive/'):
            base_dir = Path('/content/drive') / str(p).lstrip('/')
        else:
            base_dir = p
    else:
        if audio_parent_dir and Path(audio_parent_dir).exists() and str(audio_parent_dir).startswith('/content/drive'):
            base_dir = Path(audio_parent_dir) / OUTPUT_FOLDER_NAME
        else:
            colab_nb = my_drive_root / 'Colab Notebooks'
            base_dir = (colab_nb if colab_nb.exists() else my_drive_root) / OUTPUT_FOLDER_NAME

    safe_stem = re.sub(r'[\\/:*?"<>|]+', '_', (stem_name or '轉錄結果').strip())
    episode_dir = base_dir / safe_stem
    return str(episode_dir)


dict_settings = load_and_sync_ctb_dictionaries()
global_yt_id = extract_youtube_id(youtube_url.split(',')[0]) if youtube_url.strip() else ''


def render_ctb_memory_panel():
    '''在畫面上顯示可展開檢視的 CTB 雲端永久記憶詞庫面板，並支援一鍵下載最新 JSON 回 ctb.52hal.cc'''
    rep_rules = dict_settings['replace_rules']
    pos_t = dict_settings['positive_terms']
    neg_t = dict_settings['negative_terms']
    src_note = '、'.join(dict_settings['loaded_sources']) if dict_settings['loaded_sources'] else '預設詞庫初始化'

    rep_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#fff3e0; border:1px solid #ffb74d; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">{r["original"]} ➔ <b>{r["replacement"]}</b></span>'
        for r in rep_rules
    )
    pos_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#e8f5e9; border:1px solid #81c784; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">🟢 {t}</span>'
        for t in pos_t
    )
    neg_tags_html = ' '.join(
        f'<span style="display:inline-block; background:#ffebee; border:1px solid #e57373; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;">🔴 {t}</span>'
        for t in neg_t
    ) or '<span style="color:#757575; font-size:12px;">（無）</span>'

    detail_html = widgets.HTML(
        value=(
            '<div style="padding:8px; line-height:1.8;">'
            f'<div><b>☁️ 雲端永久記憶庫位置：</b><code>{dict_settings["memory_dir"]}/</code>（下次開啟自動載入）</div>'
            f'<div><b>📥 本次載入來源：</b>{src_note}</div>'
            f'<div style="margin-top:6px;"><b>🔄 目前生效的錯字強制替換（共 {len(rep_rules)} 組）：</b><br>{rep_tags_html}</div>'
            f'<div style="margin-top:6px;"><b>🟢 目前生效的正向專有名詞（共 {len(pos_t)} 個）：</b><br>{pos_tags_html}</div>'
            f'<div style="margin-top:6px;"><b>🔴 目前生效的禁用／誤聽參考詞（共 {len(neg_t)} 個）：</b><br>{neg_tags_html}</div>'
            '</div>'
        )
    )
    dl_ctb_btn = widgets.Button(
        description='📤 下載最新詞庫 JSON 設定檔',
        button_style='info',
        layout=widgets.Layout(width='360px', margin='6px 0 0 8px')
    )

    def on_dl_ctb(_):
        from google.colab import files
        mem_d = Path(dict_settings['memory_dir'])
        for fn in ('Batch_Replace_Rules.json', 'AI_Terminology_Rules.json'):
            p = mem_d / fn
            if p.is_file():
                files.download(str(p))

    dl_ctb_btn.on_click(on_dl_ctb)
    acc = widgets.Accordion(children=[widgets.VBox([detail_html, dl_ctb_btn])])
    acc.set_title(0, f'📖 詞庫雲端永久記憶庫（已自動載入：錯字替換 {len(rep_rules)} 組 ｜ 🟢 專有名詞 {len(pos_t)} 個 — 點此展開明細）')
    acc.selected_index = None
    display(acc)


def save_job_queue(mode, jobs_list, auto_save):
    '''儲存待轉錄工作佇列與詞庫設定'''
    queue_data = {
        'mode': mode,
        'auto_save_to_drive': bool(auto_save),
        'drive_output_folder': (drive_output_folder or '').strip(),
        'dictionary': dict_settings,
        'jobs': jobs_list
    }
    (work / 'job_queue.json').write_text(json.dumps(queue_data, ensure_ascii=False, indent=2), encoding='utf-8')
    if jobs_list:
        first_job = jobs_list[0]
        (work / 'output_stem.txt').write_text(first_job['stem'], encoding='utf-8')
        target_media = work / 'input.media'
        src_p = Path(first_job['source_path'])
        if src_p.exists() and src_p != target_media:
            if target_media.exists() or target_media.is_symlink():
                target_media.unlink()
            try:
                target_media.symlink_to(src_p)
            except Exception:
                shutil.copy2(src_p, target_media)


def scan_drive_media(root_dir, max_depth=4):
    found = []
    root_path = Path(root_dir)
    if not root_path.exists():
        return found

    def _walk(curr_path, depth):
        if depth > max_depth:
            return
        try:
            entries = list(os.scandir(curr_path))
        except Exception:
            return
        for entry in entries:
            if entry.name.startswith('.') or entry.name in (OUTPUT_FOLDER_NAME, CTB_MEMORY_FOLDER_NAME, 'node_modules', '__pycache__'):
                continue
            try:
                if entry.is_file(follow_symlinks=False):
                    ext = os.path.splitext(entry.name)[1].lower()
                    if ext in MEDIA_EXTS:
                        st = entry.stat(follow_symlinks=False)
                        if st.st_size > 0:
                            found.append((Path(entry.path), st.st_mtime, st.st_size))
                elif entry.is_dir(follow_symlinks=False):
                    _walk(Path(entry.path), depth + 1)
            except Exception:
                continue

    _walk(root_path, 1)
    found.sort(key=lambda item: ('Colab Notebooks' in str(item[0]), item[1]), reverse=True)
    return found


def scan_drive_texts(root_dir, max_depth=4):
    '''掃描 Google 雲端硬碟中的 .srt / .vtt / .txt / .md 現成文稿檔案'''
    found = []
    root_path = Path(root_dir)
    if not root_path.exists():
        return found

    def _walk(curr_path, depth):
        if depth > max_depth:
            return
        try:
            entries = list(os.scandir(curr_path))
        except Exception:
            return
        for entry in entries:
            if entry.name.startswith('.') or entry.name in (CTB_MEMORY_FOLDER_NAME, 'node_modules', '__pycache__'):
                continue
            try:
                if entry.is_file(follow_symlinks=False):
                    ext = os.path.splitext(entry.name)[1].lower()
                    if ext in TEXT_EXTS and not entry.name.endswith('_原始未校對.srt'):
                        st = entry.stat(follow_symlinks=False)
                        if st.st_size > 0:
                            found.append((Path(entry.path), st.st_mtime, st.st_size))
                elif entry.is_dir(follow_symlinks=False):
                    _walk(Path(entry.path), depth + 1)
            except Exception:
                continue

    _walk(root_path, 1)
    found.sort(key=lambda item: ('Colab Notebooks' in str(item[0]), item[1]), reverse=True)
    return found


def register_direct_text_job(raw_text, filename_or_title, audio_parent_dir=None):
    '''
    將現成 SRT / VTT / TXT 文稿直接解析並建立步驟 4 所需的全部標準工作檔案（raw_segments.json 等），
    讓使用者可以 100% 跳過步驟 1 與步驟 3，直接執行步驟 4！
    '''
    p_obj = Path(filename_or_title)
    ext = p_obj.suffix.lower() if p_obj.suffix.lower() in TEXT_EXTS else '.txt'
    stem = (p_obj.stem if p_obj.suffix.lower() in TEXT_EXTS else filename_or_title).strip() or '現成文稿創作'
    safe_stem = re.sub(r'[\/:*?"<>|]+', '_', stem)

    cues_tuples, converted_segs, rule_hits, has_real_ts, duration_sec = parse_existing_text_or_srt(
        raw_text, ext=ext, replace_rules=dict_settings['replace_rules']
    )
    txt_lines = [t for _, _, t in cues_tuples]
    outputs = build_outputs_from_cues(cues_tuples, converted_segs, intro_music_seconds=0, txt_lines=txt_lines)

    ep_save_dir = resolve_job_drive_save_dir(safe_stem, audio_parent_dir=audio_parent_dir)
    job_id = 'job_001'
    j_dir = work / 'jobs' / job_id
    j_dir.mkdir(parents=True, exist_ok=True)

    for out_name, out_content in outputs.items():
        (j_dir / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (j_dir / out_name).write_text(out_content, encoding='utf-8')
        (work / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (work / out_name).write_text(out_content, encoding='utf-8')

    raw_payload = {
        'job_id': job_id,
        'filename': f'{safe_stem}{ext}',
        'stem': safe_stem,
        'language': 'zh',
        'duration': duration_sec,
        'source_mode': 'direct_text',
        'has_real_timestamps': bool(has_real_ts),
        'youtube_id': global_yt_id,
        'youtube_url': youtube_url.strip(),
        'drive_save_dir': ep_save_dir,
        'deterministic_rule_hits': rule_hits,
        'subtitle_settings': {
            'auto_split': False,
            'auto_punctuation': False,
            'max_subtitle_chars': 22,
            'max_subtitle_seconds': 6.0,
            'intro_music_seconds': 0.0,
            'timing_methods': ['imported_timestamps' if has_real_ts else 'estimated_reading_speed']
        },
        'cues': [
            {'start_ms': s, 'end_ms': e, 'text': t}
            for s, e, t in cues_tuples
        ],
        'segments': converted_segs
    }
    raw_json_str = json.dumps(raw_payload, ensure_ascii=False, indent=2)
    (j_dir / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (j_dir / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'output_stem.txt').write_text(safe_stem, encoding='utf-8')

    queue_data = {
        'mode': 'direct_text',
        'skip_transcription': True,
        'auto_save_to_drive': bool(auto_save_to_drive),
        'drive_output_folder': (drive_output_folder or '').strip(),
        'dictionary': dict_settings,
        'jobs': [{
            'job_id': job_id,
            'filename': f'{safe_stem}{ext}',
            'stem': safe_stem,
            'source_path': str(j_dir / ('raw_transcript.srt' if has_real_ts else 'raw_transcript.txt')),
            'youtube_id': global_yt_id,
            'youtube_url': youtube_url.strip(),
            'drive_save_dir': ep_save_dir,
            'skip_transcription': True,
            'has_real_timestamps': bool(has_real_ts)
        }]
    }
    (work / 'job_queue.json').write_text(json.dumps(queue_data, ensure_ascii=False, indent=2), encoding='utf-8')
    return raw_payload



def execute_selected_source_mode(curr_source_type):
    '''根據目前生效的來源模式，在 dynamic_source_out 中渲染對應的來源選擇器與工作佇列'''
    dynamic_source_out.clear_output(wait=True)
    with dynamic_source_out:
        if 'YouTube' in curr_source_type:
            yt_box_input = widgets.Text(
                value=youtube_url.strip(),
                placeholder='請貼上 YouTube 影片或直播網址（多支網址可用逗號 , 分隔）',
                description='🎬 YT 網址：',
                style={'description_width': '95px'},
                layout=widgets.Layout(width='560px')
            )
            yt_dl_btn = widgets.Button(
                description='🚀 下載 YouTube 音軌並綁定影片 ID',
                button_style='success',
                layout=widgets.Layout(width='260px', margin='0 0 0 8px')
            )
            yt_out_area = widgets.Output()

            def _run_yt_download(raw_u_str):
                global global_yt_id
                yt_out_area.clear_output(wait=True)
                raw_urls = [u.strip() for u in re.split(r'[,，]+', (raw_u_str or '').replace(chr(10), ',')) if u.strip()]
                if not raw_urls:
                    wait_step2.value = '<div style="padding:10px 14px; border:2px solid #e65100; background:#fff3e0; border-radius:8px; color:#bf360c;"><b>⚠️ 請在下方輸入框貼上 YouTube 影片網址後點擊「🚀 下載 YouTube 音軌並綁定影片 ID」！</b></div>'
                    return
                with yt_out_area:
                    if auto_save_to_drive:
                        ensure_drive_mounted_if_needed()
                    yt_dir = work / 'yt_downloads'
                    yt_dir.mkdir(parents=True, exist_ok=True)
                    jobs_list = []
                    for idx, u in enumerate(raw_urls, 1):
                        yt_id = extract_youtube_id(u)
                        full_url = f'https://www.youtube.com/watch?v={yt_id}' if (yt_id and not u.startswith('http')) else u
                        wait_step2.value = f'🎬 正在從 YouTube 取得影片資訊並下載音軌（第 {idx}/{len(raw_urls)} 支：{full_url}）...'
                        print(f'🎬 [{idx}/{len(raw_urls)}] 正在下載 YouTube 音軌：{full_url} ...', flush=True)
                        probe = subprocess.run(
                            ['yt-dlp', '--no-playlist', '--print', '%(id)s|||%(title)s', full_url],
                            capture_output=True, text=True, check=False
                        )
                        vid_id = yt_id or f'yt_{idx:03d}'
                        vid_title = f'YouTube_{vid_id}'
                        if probe.returncode == 0 and '|||' in probe.stdout:
                            first_line = probe.stdout.strip().splitlines()[0]
                            p_id, p_title = first_line.split('|||', 1)
                            if p_id.strip():
                                vid_id = p_id.strip()
                            if p_title.strip():
                                vid_title = re.sub(r'[\/:*?"<>|]+', '_', p_title.strip())
                        out_tmpl = str(yt_dir / f'{vid_id}.%(ext)s')
                        subprocess.run(['yt-dlp', '--no-playlist', '-f', 'bestaudio[ext=m4a]/bestaudio/best', '-o', out_tmpl, full_url], check=True)
                        downloaded_files = list(yt_dir.glob(f'{vid_id}.*'))
                        if not downloaded_files:
                            raise RuntimeError(f'YouTube 音軌下載失敗：{full_url}')
                        media_file = downloaded_files[0]
                        ep_save_dir = resolve_job_drive_save_dir(vid_title)
                        jobs_list.append({
                            'job_id': f'job_{idx:03d}',
                            'filename': f'{vid_title}{media_file.suffix}',
                            'stem': vid_title,
                            'source_path': str(media_file),
                            'youtube_id': vid_id,
                            'youtube_url': f'https://www.youtube.com/watch?v={vid_id}',
                            'drive_save_dir': ep_save_dir
                        })
                        print(f'✅ 已下載完成：{vid_title} (YouTube ID: {vid_id})', flush=True)
                    mode_str = 'batch' if len(jobs_list) > 1 else 'single'
                    save_job_queue(mode_str, jobs_list, auto_save=auto_save_to_drive)
                    wait_step2.value = (
                        '<div style="padding:12px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:8px; color:#1b5e20;">'
                        f'<b>✅ 階段 2/6 完成：已成功下載 {len(jobs_list)} 支 YouTube 音軌！</b><br>'
                        f'<b>🎬 影片標題：</b><code>{jobs_list[0]["stem"]}</code>（已自動綁定 YouTube ID <code>{jobs_list[0]["youtube_id"]}</code>）<br>'
                        f'<b>💾 雲端專屬儲存子資料夾：</b><code>{jobs_list[0]["drive_save_dir"] or "未開啟自動存回雲端"}</code><br>'
                        '👉 <b>請直接執行「步驟 3」開始語音辨識。</b>'
                        '</div>'
                    )

            yt_dl_btn.on_click(lambda _: _run_yt_download(yt_box_input.value))
            display(widgets.HBox([yt_box_input, yt_dl_btn]), yt_out_area)
            if youtube_url.strip():
                _run_yt_download(youtube_url.strip())
            else:
                wait_step2.value = '🎬 <b>請在下方貼上 YouTube 影片或直播網址，並點擊「🚀 下載 YouTube 音軌並綁定影片 ID」：</b>'

        elif '現成 SRT/VTT/TXT 文稿直通（本機上傳' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            from google.colab import files
            wait_step2.value = '📝 <b>請點擊下方「選擇檔案」從電腦上傳您的 .srt、.vtt、.txt 或 .md 文稿檔案：</b>'
            print('📝 請點擊下方「選擇檔案」上傳您的 .srt、.vtt、.txt 或 .md 文稿檔案：', flush=True)
            uploaded = files.upload()
            if not uploaded:
                wait_step2.value = '⚠️ 尚未選取任何文稿檔案，請重新選擇或切換其他文稿來源。'
            else:
                doc_filename = next(iter(uploaded.keys()))
                raw_bytes = uploaded[doc_filename]
                if Path(doc_filename).exists():
                    try:
                        Path(doc_filename).unlink()
                    except Exception:
                        pass
                raw_str = ''
                for enc in ('utf-8-sig', 'utf-8', 'cp950', 'big5', 'utf-16'):
                    try:
                        raw_str = raw_bytes.decode(enc)
                        break
                    except Exception:
                        continue
                if not raw_str:
                    raw_str = raw_bytes.decode('utf-8', errors='replace')
                info = register_direct_text_job(raw_str, doc_filename)
                ts_badge = '🕒 偵測為含時間碼字幕（100% 保留原始時間軸）' if info['has_real_timestamps'] else '📄 偵測為純文字逐字稿（已智慧切分句子）'
                wait_step2.value = (
                    '<div style="padding:14px 16px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
                    f'<b>⚡ 【現成文稿直通模式】準備完成！已載入：<code>{doc_filename}</code></b><br>'
                    f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                    f'💾 <b>雲端自動存檔專屬資料夾：</b><code>{info["drive_save_dir"] or "未開啟"}</code><br>'
                    '<div style="margin-top:8px; padding:8px 12px; background:#fff8e1; border-left:4px solid #ffa000; border-radius:6px; color:#e65100; font-weight:bold;">'
                    '🚀 您無須執行「步驟 1」與「步驟 3（語音辨識）」！請直接往下捲動點擊【步驟 4】開始 AI 全媒體創作與推送 Blogger！'
                    '</div></div>'
                )

        elif '現成 SRT/VTT/TXT 文稿直通（Google 雲端硬碟' in curr_source_type:
            my_drive_root = ensure_drive_mounted_if_needed()
            wait_step2.value = '🔍 正在掃描您的 Google 雲端硬碟中的 .srt / .vtt / .txt / .md 文稿檔案...'
            text_candidates = scan_drive_texts(my_drive_root, max_depth=4)
            if not text_candidates:
                wait_step2.value = '⚠️ 在雲端硬碟中找不到 .srt / .vtt / .txt 文稿檔案，請於上方「📝 軌跡 B 來源」切換為「本機上傳」或「直接貼上純文字」模式。'
            else:
                doc_opts = []
                for f_path, _, f_size in text_candidates[:250]:
                    try:
                        rel = str(f_path.relative_to(my_drive_root))
                    except ValueError:
                        rel = str(f_path)
                    kb = f_size / 1024.0
                    doc_opts.append((f'📝 {rel} ({kb:.1f} KB)', str(f_path)))
                doc_dd = widgets.Dropdown(
                    options=doc_opts,
                    value=doc_opts[0][1],
                    description='選擇雲端文稿：',
                    style={'description_width': '110px'},
                    layout=widgets.Layout(width='95%')
                )
                doc_status = widgets.HTML()

                def apply_doc_selection(path_str):
                    p = Path(path_str)
                    raw_str = ''
                    raw_bytes = p.read_bytes()
                    for enc in ('utf-8-sig', 'utf-8', 'cp950', 'big5', 'utf-16'):
                        try:
                            raw_str = raw_bytes.decode(enc)
                            break
                        except Exception:
                            continue
                    if not raw_str:
                        raw_str = raw_bytes.decode('utf-8', errors='replace')
                    info = register_direct_text_job(raw_str, p.name, audio_parent_dir=p.parent)
                    ts_badge = '🕒 含時間碼字幕（100% 保留原始時間軸）' if info['has_real_timestamps'] else '📄 純文字逐字稿（已智慧切分句子）'
                    doc_status.value = (
                        '<div style="margin-top:8px; padding:14px 16px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
                        f'<b>⚡ 已載入雲端現成文稿：</b><code>{p}</code><br>'
                        f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                        f'💾 <b>雲端成果專屬子資料夾：</b><code>{info["drive_save_dir"]}</code><br>'
                        '<div style="margin-top:8px; padding:8px 12px; background:#fff8e1; border-left:4px solid #ffa000; border-radius:6px; color:#e65100; font-weight:bold;">'
                        '🚀 您無須執行「步驟 3（語音辨識）」！請直接前往【步驟 4】開始 AI 全媒體創作與推送 Blogger！'
                        '</div></div>'
                    )

                doc_dd.observe(lambda ch: apply_doc_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_doc_selection(doc_opts[0][1])
                wait_step2.value = f'✅ 已掃描到 <b>{len(doc_opts)}</b> 份雲端文稿檔案，請由下方選單挑選：'
                display(widgets.VBox([doc_dd, doc_status]))

        elif '直接貼上純文字' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            paste_title_input = widgets.Text(
                value='我的專欄文稿與筆記',
                placeholder='請輸入這篇作品的標題或檔名（將作為雲端專屬子資料夾名稱）',
                description='📌 作品名稱：',
                style={'description_width': '100px'},
                layout=widgets.Layout(width='500px')
            )
            paste_area = widgets.Textarea(
                value='',
                placeholder='請在此直接貼上您的 .srt 字幕內容、會議逐字稿、演講筆記或文章草稿...',
                description='✍️ 貼上文稿：',
                style={'description_width': '100px'},
                layout=widgets.Layout(width='95%', height='220px')
            )
            paste_save_btn = widgets.Button(
                description='⚡ 確認載入文稿並直通【步驟 4】',
                button_style='success',
                layout=widgets.Layout(width='280px', margin='8px 0 0 105px')
            )
            paste_status = widgets.HTML()

            def on_click_paste_save(_):
                body = paste_area.value.strip()
                if not body:
                    paste_status.value = '<span style="color:#d32f2f; font-weight:bold;">❌ 請先在上方文字框貼上文稿內容！</span>'
                    return
                title_str = paste_title_input.value.strip() or '手動貼上文稿'
                info = register_direct_text_job(body, title_str)
                ts_badge = '🕒 偵測為含時間碼 SRT/VTT 字幕' if info['has_real_timestamps'] else '📄 偵測為純文字文稿（已智慧分句）'
                paste_status.value = (
                    '<div style="margin-top:8px; padding:14px 16px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:10px; color:#1b5e20; line-height:1.75;">'
                    f'<b>✅ 文稿已成功載入並完成詞庫清洗！作品名稱：<code>{info["stem"]}</code></b><br>'
                    f'📌 <b>解析狀態：</b>{ts_badge} ｜ 共 <b>{len(info["cues"])}</b> 句 ｜ 已自動套用錯字替換 <b>{info["deterministic_rule_hits"]}</b> 處<br>'
                    f'💾 <b>雲端成果專屬子資料夾：</b><code>{info["drive_save_dir"]}</code><br>'
                    '👉 <b>請直接略過步驟 3，立刻前往【步驟 4】開始 AI 全媒體創作！</b>'
                    '</div>'
                )

            paste_save_btn.on_click(on_click_paste_save)
            wait_step2.value = '✍️ <b>請在下方輸入作品名稱並貼上您的 SRT 字幕或 TXT 純文字逐字稿，再按「⚡ 確認載入文稿並直通【步驟 4】」：</b>'
            display(widgets.VBox([paste_title_input, paste_area, paste_save_btn, paste_status]))

        elif '本機上傳' in curr_source_type:
            if auto_save_to_drive:
                ensure_drive_mounted_if_needed()
            from google.colab import files
            wait_step2.value = '💻 <b>請點擊下方「選擇檔案」從電腦上傳音訊或視訊檔案：</b>'
            print('階段 2/6：請點擊下方「選擇檔案」上傳音訊或視訊檔案（若超過 100MB 建議改用 Google 雲端硬碟模式）：', flush=True)
            uploaded = files.upload()
            if not uploaded:
                wait_step2.value = '⚠️ 尚未選取任何本機檔案，請重新選擇或切換來源模式。'
            else:
                audio_filename = next(iter(uploaded.keys()))
                uploaded_file = Path(audio_filename)
                target_media = work / 'input.media'
                if uploaded_file.exists():
                    if target_media.exists() or target_media.is_symlink():
                        target_media.unlink()
                    shutil.move(str(uploaded_file), str(target_media))
                else:
                    target_media.write_bytes(uploaded[audio_filename])
                del uploaded
                output_stem = Path(audio_filename).stem or '轉錄結果'
                ep_save_dir = resolve_job_drive_save_dir(output_stem)
                save_job_queue('single', [{
                    'job_id': 'job_001',
                    'filename': audio_filename,
                    'stem': output_stem,
                    'source_path': str(target_media),
                    'youtube_id': global_yt_id,
                    'youtube_url': youtube_url.strip(),
                    'drive_save_dir': ep_save_dir
                }], auto_save=bool(ep_save_dir))
                save_msg = f'<br>💾 <b>雲端自動存檔專屬資料夾：</b><code>{ep_save_dir}</code>' if ep_save_dir else ''
                wait_step2.value = f'✅ 階段 2/6 完成：本機音檔已準備（<b>{audio_filename}</b>）。{save_msg}<br>👉 請接續執行步驟 3。'

        elif curr_source_type.startswith('Google 雲端硬碟'):
            my_drive_root = ensure_drive_mounted_if_needed()
            wait_step2.value = '🔍 正在掃描您的 Google 雲端硬碟音訊與視訊檔案...'
            raw_path = drive_path.strip().strip("'").strip('"')
            user_target = None
            if raw_path:
                candidate_p = Path(raw_path)
                if not candidate_p.is_absolute():
                    candidate_p = my_drive_root / candidate_p
                elif str(candidate_p).startswith('/MyDrive/'):
                    candidate_p = Path('/content/drive') / str(candidate_p).lstrip('/')
                if candidate_p.exists():
                    user_target = candidate_p
            all_media = scan_drive_media(my_drive_root, max_depth=4)
            if user_target and user_target.is_dir():
                extra = scan_drive_media(user_target, max_depth=3)
                seen_paths = {p for p, _, _ in all_media}
                for item in extra:
                    if item[0] not in seen_paths:
                        all_media.insert(0, item)
            elif user_target and user_target.is_file():
                st = user_target.stat()
                if all(p != user_target for p, _, _ in all_media):
                    all_media.insert(0, (user_target, st.st_mtime, st.st_size))

            if not all_media:
                wait_step2.value = (
                    '<div style="padding:12px; border:2px solid #f57c00; background:#fff8e1; border-radius:8px;">'
                    '<b>⚠️ 在您的雲端硬碟中尚未掃描到音訊或影片檔案（.m4a / .mp3 / .wav / .mp4 等）。</b>'
                    '</div>'
                )
            elif '單一音檔' in curr_source_type:
                options_list = []
                default_val = None
                for f_path, mtime, f_size in all_media:
                    try:
                        rel_label = str(f_path.relative_to(my_drive_root))
                    except ValueError:
                        rel_label = str(f_path)
                    size_mb = f_size / (1024 * 1024)
                    options_list.append((f'🎵 {rel_label}  ({size_mb:.1f} MB)', str(f_path)))
                    if user_target and user_target.is_file() and f_path.resolve() == user_target.resolve():
                        default_val = str(f_path)
                if default_val is None:
                    default_val = options_list[0][1]

                file_dropdown = widgets.Dropdown(
                    options=options_list,
                    value=default_val,
                    description='選擇雲端音檔：',
                    style={'description_width': '110px'},
                    layout=widgets.Layout(width='95%')
                )
                status_box = widgets.HTML()

                def apply_single_selection(selected_path_str):
                    sel_path = Path(selected_path_str)
                    stem = sel_path.stem or '轉錄結果'
                    ep_out_dir = resolve_job_drive_save_dir(stem, audio_parent_dir=sel_path.parent)
                    save_job_queue('single', [{
                        'job_id': 'job_001',
                        'filename': sel_path.name,
                        'stem': stem,
                        'source_path': str(sel_path),
                        'youtube_id': global_yt_id,
                        'youtube_url': youtube_url.strip(),
                        'drive_save_dir': ep_out_dir
                    }], auto_save=auto_save_to_drive)
                    save_hint = f'開啟（將自動存至專屬同名子資料夾 <code>{ep_out_dir}/</code>）' if auto_save_to_drive else '關閉（僅於步驟 5 下載 ZIP）'
                    yt_hint = f'<br><b>🎬 已綁定 YouTube ID：</b><code>{global_yt_id}</code>' if global_yt_id else ''
                    status_box.value = (
                        '<div style="margin-top:8px; padding:12px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:8px; color:#1b5e20;">'
                        f'<b>✅ 目前已選取音檔：</b><code>{sel_path.name}</code><br>'
                        f'<b>📂 雲端所在位置：</b><code>{sel_path.parent}</code><br>'
                        f'<b>💾 自動存回雲端：</b>{save_hint}{yt_hint}<br>'
                        f'<b>📖 已自動載入雲端永久記憶詞庫：</b>錯字替換 <b>{len(dict_settings["replace_rules"])}</b> 組、🟢 正向專有名詞 <b>{len(dict_settings["positive_terms"])}</b> 個<br>'
                        '👉 <b>確認無誤後請直接點擊下方「步驟 3」開始轉錄！</b>'
                        '</div>'
                    )

                file_dropdown.observe(lambda ch: apply_single_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_single_selection(default_val)
                wait_step2.value = f'✅ 階段 2/6 完成：已在雲端硬碟掃描到 <b>{len(options_list)}</b> 個音訊/視訊檔案，請由下方選單確認或切換：'
                display(widgets.VBox([file_dropdown, status_box]))

            else:
                folder_map = {}
                for f_path, mtime, f_size in all_media:
                    folder_map.setdefault(f_path.parent, []).append((f_path, f_size))
                folder_options = []
                default_folder = None
                for folder_path, files_in_folder in folder_map.items():
                    try:
                        rel_folder = str(folder_path.relative_to(my_drive_root))
                        if rel_folder == '.':
                            rel_folder = '我的雲端硬碟 (根目錄)'
                    except ValueError:
                        rel_folder = str(folder_path)
                    total_mb = sum(sz for _, sz in files_in_folder) / (1024 * 1024)
                    folder_options.append((f'📁 {rel_folder}  （共 {len(files_in_folder)} 個音檔，合計 {total_mb:.1f} MB）', str(folder_path)))
                    if user_target and user_target.is_dir() and folder_path.resolve() == user_target.resolve():
                        default_folder = str(folder_path)
                if default_folder is None:
                    default_folder = folder_options[0][1]

                folder_dropdown = widgets.Dropdown(
                    options=folder_options,
                    value=default_folder,
                    description='選擇批次資料夾：',
                    style={'description_width': '120px'},
                    layout=widgets.Layout(width='95%')
                )
                batch_status_box = widgets.HTML()

                def apply_folder_selection(folder_str):
                    f_dir = Path(folder_str)
                    candidates = folder_map.get(f_dir, [])
                    jobs_list = []
                    skipped_names = []
                    sample_parent_out = ''
                    for idx, (f_path, f_size) in enumerate(candidates, 1):
                        stem = f_path.stem or f'音檔_{idx}'
                        ep_out_dir = resolve_job_drive_save_dir(stem, audio_parent_dir=f_dir)
                        if ep_out_dir and not sample_parent_out:
                            sample_parent_out = str(Path(ep_out_dir).parent)
                        if skip_already_transcribed and ep_out_dir and (Path(ep_out_dir) / f'{stem}.srt').is_file():
                            skipped_names.append(f_path.name)
                            continue
                        jobs_list.append({
                            'job_id': f'job_{idx:03d}',
                            'filename': f_path.name,
                            'stem': stem,
                            'source_path': str(f_path),
                            'youtube_id': '',
                            'youtube_url': '',
                            'drive_save_dir': ep_out_dir
                        })
                    save_job_queue('batch', jobs_list, auto_save=auto_save_to_drive)
                    file_items_html = ''.join(f'<li><code>{j["filename"]}</code> ➔ 存至子資料夾 <code>{j["stem"]}/</code></li>' for j in jobs_list[:15])
                    if len(jobs_list) > 15:
                        file_items_html += f'<li>...及其他 {len(jobs_list) - 15} 個音檔</li>'
                    skip_html = f'<br>⏭️ 已略過雲端既有成果：{len(skipped_names)} 個檔案' if skipped_names else ''
                    batch_status_box.value = (
                        '<div style="margin-top:8px; padding:12px; border:2px solid #1565c0; background:#e3f2fd; border-radius:8px; color:#0d47a1;">'
                        f'<b>📦 已選取整批轉錄資料夾：</b><code>{f_dir}</code><br>'
                        f'<b>🎵 待自動轉錄音檔數：</b>共 <b>{len(jobs_list)}</b> 個音檔{skip_html}<br>'
                        f'<b>💾 自動存回雲端母資料夾：</b><code>{sample_parent_out}/{{音檔同名資料夾}}/</code><br>'
                        f'<ul style="margin:6px 0 6px 20px;">{file_items_html}</ul>'
                        '👉 <b>確認資料夾後，請依序按「步驟 3」與「步驟 4」！</b>'
                        '</div>'
                    )

                folder_dropdown.observe(lambda ch: apply_folder_selection(ch['new']) if ch.get('new') else None, names='value')
                apply_folder_selection(default_folder)
                wait_step2.value = f'✅ 階段 2/6 完成：已掃描到 <b>{len(folder_options)}</b> 個包含音檔的雲端資料夾，請由下方選單確認或切換：'
                display(widgets.VBox([folder_dropdown, batch_status_box]))



dynamic_source_out = widgets.Output()
stepper_box_s2.value = render_stepper_banner(2, '📝 支援現成 SRT / TXT 匯入（免開 GPU）')
display(dynamic_source_out)
render_ctb_memory_panel()
execute_selected_source_mode(source_type)

## <font color="#2e7d32"><b>3. 開始語音辨識與字幕清洗</b></font>

In [ ]:
# @title 🟨 **3. 開始語音辨識與字幕清洗（支援 Faster-Whisper ／ Qwen3-ASR 台華雙語切換）**
# @markdown ### 🎙️ 語音辨識引擎與模型大小（選好後點擊「🚀 載入所選模型並開始辨識」才會載入該模型，絕不浪費記憶體！）
語音辨識引擎 = "⚡ Faster-Whisper（國語／中英夾雜首選）" # @param ["⚡ Faster-Whisper（國語／中英夾雜首選）", "🇹🇼 Qwen3-ASR（台語／國台語夾雜專用）"]
FasterWhisper模型大小 = "large-v3-turbo" # @param ["large-v3-turbo", "large-v3", "medium", "small"]
Qwen3ASR模型大小 = "Qwen/Qwen3-ASR-1.7B" # @param ["Qwen/Qwen3-ASR-1.7B", "Qwen/Qwen3-ASR-0.6B"]
啟用Qwen3精準時間軸對齊模型 = True # @param {type:"boolean"}

# @markdown ### ✂️ 字幕斷句與標點設定
自動依字數與秒數斷句 = True # @param {type:"boolean"}
停頓處自動補上句號 = False # @param {type:"boolean"}
每段字幕最大字數 = 22 # @param {type:"integer"}
每段字幕最長秒數 = 6 # @param {type:"number"}
開場純音樂秒數_0為不加入 = 0 # @param {type:"number"}

asr_engine_choice = 語音辨識引擎
whisper_model_size = FasterWhisper模型大小
qwen_model_size = Qwen3ASR模型大小
qwen_use_aligner = 啟用Qwen3精準時間軸對齊模型
auto_split = 自動依字數與秒數斷句
auto_punctuation = 停頓處自動補上句號
max_subtitle_chars = 每段字幕最大字數
max_subtitle_seconds = 每段字幕最長秒數
intro_music_seconds = 開場純音樂秒數_0為不加入

import json, os, sys, subprocess
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

clear_output(wait=True)
work = Path('/content/alta_transcribe_work')
queue_path = work / 'job_queue.json'

if not queue_path.is_file() and not (work / 'input.media').exists():
    raise RuntimeError('找不到待轉錄音檔，請先執行「步驟 2A」或「步驟 2B」選擇來源！')

if not queue_path.is_file():
    stem = (work / 'output_stem.txt').read_text(encoding='utf-8').strip() if (work / 'output_stem.txt').is_file() else '轉錄結果'
    fallback_queue = {
        'mode': 'single',
        'auto_save_to_drive': False,
        'dictionary': {'replace_rules': [], 'positive_terms': [], 'negative_terms': []},
        'jobs': [{
            'job_id': 'job_001',
            'filename': stem,
            'stem': stem,
            'source_path': str(work / 'input.media'),
            'youtube_id': '',
            'youtube_url': '',
            'drive_save_dir': ''
        }]
    }
    queue_path.write_text(json.dumps(fallback_queue, ensure_ascii=False, indent=2), encoding='utf-8')

queue_info = json.loads(queue_path.read_text(encoding='utf-8'))
jobs_list = queue_info.get('jobs') or []
if not jobs_list:
    raise RuntimeError('目前工作佇列中沒有待處理的音檔，請在「步驟 2A」確認選取音檔、資料夾或 YouTube 網址！')

# 準備字幕處理與確定性清洗函式庫 (p0_subtitles.py)
if not (work / 'p0_subtitles.py').is_file():
    raise RuntimeError('找不到 p0_subtitles.py，請先執行一次步驟 2A 或步驟 2B！')

# 準備支援雙引擎 (Faster-Whisper + Qwen3-ASR 台語/國台語夾雜) 的核心執行腳本 (transcribe_worker.py)
worker_code = r'''# Transcribe single or batch audio files with Faster-Whisper or Qwen3-ASR (Mandarin/Taiwanese Hokkien).
import json
import os
import shutil
import subprocess
import sys
import wave
from pathlib import Path

try:
    import av
    _orig_av_open = av.open
    def _safe_av_open(*args, **kwargs):
        try:
            return _orig_av_open(*args, **kwargs)
        except TypeError as e:
            if 'metadata_errors' in str(e):
                kwargs.pop('metadata_errors', None)
                return _orig_av_open(*args, **kwargs)
            raise
    av.open = _safe_av_open
except Exception:
    pass

from p0_subtitles import friendly_exports

work = Path(os.environ['ALTA_WORK_DIR'])
engine_id = os.environ.get('ALTA_ENGINE', 'faster_whisper')
model_id = os.environ['ALTA_MODEL']
use_aligner = os.environ.get('ALTA_QWEN_ALIGNER', '1') == '1'
queue_data = json.loads((work / 'job_queue.json').read_text(encoding='utf-8'))
jobs = queue_data.get('jobs') or []
dict_cfg = queue_data.get('dictionary') or {}
replace_rules = dict_cfg.get('replace_rules') or []
positive_terms = dict_cfg.get('positive_terms') or []

if not jobs:
    raise RuntimeError('沒有待處理的音檔工作！')

limit = float(os.environ['ALTA_MAX_SECONDS'])
intro_seconds = float(os.environ.get('ALTA_INTRO_MUSIC_SECONDS', '0'))
terms_hint = ('專有名詞參考：' + '、'.join(positive_terms[:25]) + '。') if positive_terms else ''
initial_prompt = '以下是臺灣繁體中文內容，請使用繁體中文輸出，包含常見標點符號。' + terms_hint

import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ 目前 Colab 執行階段未啟用 GPU（可能為 CPU 模式）！請點選上方「執行階段 ➔ 變更執行階段類型 ➔ 選擇 T4 GPU」後再重跑步驟 1 與步驟 3。')
print(f'✅ 已確認使用 GPU 加速：{torch.cuda.get_device_name(0)}', flush=True)

if engine_id == 'qwen3_asr':
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['USE_TF'] = '0'
    import soundfile as sf
    import numpy as np
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
    except Exception:
        pass
    from qwen_asr import Qwen3ASRModel

    aligner_desc = ' ＋ Qwen3-ForcedAligner-0.6B 時間軸對齊' if use_aligner else ''
    print(f'階段 3/6：正在按需載入台華雙語模型（{model_id}{aligner_desc}），首次使用會下載權重...', flush=True)
    load_kwargs = {
        'dtype': torch.float16,
        'device_map': 'cuda:0',
        'max_new_tokens': 512,
    }
    if use_aligner:
        load_kwargs['forced_aligner'] = 'Qwen/Qwen3-ForcedAligner-0.6B'
        load_kwargs['forced_aligner_kwargs'] = {'dtype': torch.float16, 'device_map': 'cuda:0'}
    qwen_model = Qwen3ASRModel.from_pretrained(model_id, **load_kwargs)
else:
    from faster_whisper import WhisperModel
    print(f'階段 3/6：正在按需載入 Faster-Whisper 模型（{model_id}），首次使用會下載權重...', flush=True)
    whisper_model = WhisperModel(model_id, device='cuda', compute_type='float16')


def split_audio_for_qwen(media_path, tmp_dir):
    # 將長音檔轉為 16kHz 單聲道 WAV，並依 VAD 靜音點切分為 <=22 秒的語音片段，避免切斷詞彙
    tmp_dir.mkdir(parents=True, exist_ok=True)
    wav_16k = tmp_dir / 'full_16k.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', str(media_path), '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', str(wav_16k)],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    data, sr = sf.read(str(wav_16k), dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)
    total_samples = len(data)
    duration = max(1.0, total_samples / float(sr))

    # 使用 Silero VAD 尋找自然靜音斷點，合併為每段最長 22 秒的語音塊
    speech_spans = []
    try:
        from faster_whisper.vad import VadOptions, get_speech_timestamps
        ts_list = get_speech_timestamps(data, VadOptions(min_silence_duration_ms=450, speech_pad_ms=200), sampling_rate=sr)
        for t in ts_list:
            speech_spans.append((t['start'], t['end']))
    except Exception:
        pass

    chunks_meta = []
    max_chunk_samples = int(22 * sr)
    if speech_spans:
        cur_s, cur_e = speech_spans[0]
        for s, e in speech_spans[1:]:
            if (e - cur_s) <= max_chunk_samples:
                cur_e = e
            else:
                chunks_meta.append((cur_s, cur_e))
                cur_s, cur_e = s, e
        chunks_meta.append((cur_s, cur_e))
    else:
        for s in range(0, total_samples, max_chunk_samples):
            chunks_meta.append((s, min(total_samples, s + max_chunk_samples)))

    # 若單一語音段仍超過 25 秒，再強制切分
    final_chunks = []
    for s, e in chunks_meta:
        if (e - s) > int(25 * sr):
            for sub_s in range(s, e, max_chunk_samples):
                final_chunks.append((sub_s, min(e, sub_s + max_chunk_samples)))
        elif (e - s) >= int(0.2 * sr):
            final_chunks.append((s, e))

    chunk_files = []
    for idx, (s, e) in enumerate(final_chunks):
        c_path = tmp_dir / f'chunk_{idx:04d}.wav'
        sf.write(str(c_path), data[s:e], sr)
        chunk_files.append((c_path, s / float(sr), e / float(sr)))
    return duration, chunk_files


total_jobs = len(jobs)
jobs_root = work / 'jobs'
jobs_root.mkdir(parents=True, exist_ok=True)

for j_idx, job in enumerate(jobs, 1):
    job_id = job['job_id']
    filename = job['filename']
    stem = job['stem']
    media_path = Path(job['source_path'])
    youtube_id = job.get('youtube_id') or ''
    youtube_url = job.get('youtube_url') or ''
    drive_save_dir = job.get('drive_save_dir') or ''
    job_dir = jobs_root / job_id
    job_dir.mkdir(parents=True, exist_ok=True)

    print(f'ALTA_JOB:{j_idx}/{total_jobs}:{filename}', flush=True)
    print(chr(10) + f'▶ [{j_idx}/{total_jobs}] 開始辨識音檔：{filename}（引擎：{engine_id} / {model_id}）...', flush=True)

    if not media_path.exists():
        print(f'⚠️ 找不到音檔 {media_path}，跳過此檔案。', flush=True)
        continue

    rows = []
    duration = 1.0

    if engine_id == 'qwen3_asr':
        tmp_chunk_dir = job_dir / 'qwen_chunks'
        duration, chunk_files = split_audio_for_qwen(media_path, tmp_chunk_dir)
        total_chunks = max(1, len(chunk_files))
        for c_idx, (c_wav, c_start, c_end) in enumerate(chunk_files, 1):
            try:
                res_list = qwen_model.transcribe(
                    audio=str(c_wav),
                    language=None,
                    return_time_stamps=use_aligner
                )
            except TypeError:
                res_list = qwen_model.transcribe(audio=str(c_wav), language=None)
            if res_list:
                r = res_list[0]
                txt = (getattr(r, 'text', '') or '').strip()
                if txt:
                    words = []
                    raw_ts = getattr(r, 'time_stamps', None) or []
                    for w_item in raw_ts:
                        w_txt = str(getattr(w_item, 'text', '') or (w_item.get('text', '') if isinstance(w_item, dict) else ''))
                        w_s = float(getattr(w_item, 'start_time', 0.0) if not isinstance(w_item, dict) else w_item.get('start_time', 0.0))
                        w_e = float(getattr(w_item, 'end_time', 0.0) if not isinstance(w_item, dict) else w_item.get('end_time', 0.0))
                        abs_s = max(c_start, min(c_end, c_start + w_s))
                        abs_e = max(abs_s + 0.01, min(c_end, c_start + w_e))
                        if w_txt.strip():
                            words.append({'start': abs_s, 'end': abs_e, 'word': w_txt})
                    rows.append({
                        'start': c_start,
                        'end': c_end,
                        'text': txt,
                        'words': words
                    })
            pct = min(99, int((c_idx / total_chunks) * 100))
            print(f'ALTA_PROGRESS:{pct}', flush=True)
        shutil.rmtree(tmp_chunk_dir, ignore_errors=True)
    else:
        segments_gen, info = whisper_model.transcribe(
            str(media_path),
            language='zh',
            beam_size=5,
            word_timestamps=True,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=500),
            initial_prompt=initial_prompt
        )
        duration = info.duration or 1.0
        for item in segments_gen:
            start = max(0.0, float(item.start))
            end = min(duration, max(start, float(item.end)))
            if start >= end or not item.text.strip():
                continue
            words = []
            for w in (getattr(item, 'words', None) or []):
                w_start = max(start, min(end, float(w.start)))
                w_end = max(start, min(end, float(w.end)))
                if w_start < w_end:
                    words.append({'start': w_start, 'end': w_end, 'word': w.word})
            rows.append({
                'start': start,
                'end': end,
                'avg_logprob': getattr(item, 'avg_logprob', None),
                'no_speech_prob': getattr(item, 'no_speech_prob', None),
                'text': item.text.strip(),
                'words': words
            })
            pct = min(99, int((end / duration) * 100))
            print(f'ALTA_PROGRESS:{pct}', flush=True)

    if not rows:
        print(f'⚠️ [{filename}] 未辨識出任何語音內容，略過匯出。', flush=True)
        continue

    converted, outputs, timing_methods, cues, rule_hits = friendly_exports(
        rows, duration,
        split=os.environ['ALTA_SPLIT'] == '1',
        punctuation=os.environ['ALTA_PUNCTUATION'] == '1',
        max_chars=int(os.environ['ALTA_MAX_CHARS']),
        max_seconds=limit,
        intro_music_seconds=intro_seconds,
        replace_rules=replace_rules
    )

    print(f'✅ 確定性字幕清洗報告：由 {len(rows)} 個原始語音段切分為 {len(cues)} 句標準字幕，自動套用錯字替換規則 {rule_hits} 處。', flush=True)

    for name, content in outputs.items():
        (job_dir / name).write_text(content, encoding='utf-8')
        (job_dir / ('raw_' + name)).write_text(content, encoding='utf-8')
        (work / name).write_text(content, encoding='utf-8')
        (work / ('raw_' + name)).write_text(content, encoding='utf-8')

    segments_payload = {
        'job_id': job_id,
        'filename': filename,
        'stem': stem,
        'youtube_id': youtube_id,
        'youtube_url': youtube_url,
        'drive_save_dir': drive_save_dir,
        'asr_engine': engine_id,
        'model_id': model_id,
        'language': 'zh',
        'duration': duration,
        'ai_proofread': False,
        'deterministic_rule_hits': rule_hits,
        'segments': converted,
        'cues': [{'start_ms': s, 'end_ms': e, 'text': t} for s, e, t in cues],
        'subtitle_timing_methods': timing_methods,
        'subtitle_settings': {
            'split': os.environ['ALTA_SPLIT'] == '1',
            'punctuation': os.environ['ALTA_PUNCTUATION'] == '1',
            'max_chars': int(os.environ['ALTA_MAX_CHARS']),
            'max_seconds': limit,
            'intro_music_seconds': intro_seconds
        }
    }
    segments_json_text = json.dumps(segments_payload, ensure_ascii=False, indent=2)
    (job_dir / 'segments.json').write_text(segments_json_text, encoding='utf-8')
    (job_dir / 'raw_segments.json').write_text(segments_json_text, encoding='utf-8')
    (work / 'segments.json').write_text(segments_json_text, encoding='utf-8')
    (work / 'raw_segments.json').write_text(segments_json_text, encoding='utf-8')
    (work / 'output_stem.txt').write_text(stem, encoding='utf-8')

    if drive_save_dir:
        try:
            d_dir = Path(drive_save_dir)
            d_dir.mkdir(parents=True, exist_ok=True)
            (d_dir / f'{stem}.txt').write_text(outputs['transcript.txt'], encoding='utf-8')
            (d_dir / f'{stem}.srt').write_text(outputs['transcript.srt'], encoding='utf-8')
            print(f'💾 [{j_idx}/{total_jobs}] 初步字幕已自動存回雲端：{d_dir}/{stem}.srt 與 {stem}.txt', flush=True)
        except Exception as save_err:
            print(f'⚠️ 自動存回雲端時發生警告：{save_err}', flush=True)

    print(f'ALTA_PROGRESS:100', flush=True)

print(chr(10) + '🎉 階段 3/6 全部完成！請接著執行「步驟 4：雙引擎 AI 全媒體創作工作台」。', flush=True)
'''
(work / 'transcribe_worker.py').write_text(worker_code, encoding='utf-8')

if queue_info.get('skip_transcription') or queue_info.get('mode') == 'direct_text':
    display(HTML(
        '<div style="padding:14px 18px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
        '<b>⚡ 偵測到您目前已載入「現成 SRT / VTT / TXT 文稿直通模式」！</b><br>'
        f'系統已在步驟 2B 完成文稿解析與錯字清洗（作品：<code>{jobs_list[0].get("stem", "")}</code>），<b>完全無須耗時跑 GPU 語音辨識</b>！<br>'
        '👉 <b>請直接執行下方的「步驟 4：雙引擎 AI 全媒體創作工作台」即可立即產出所有作品！</b>'
        '</div>'
    ))
else:
    # ==========================================================================
    # 🎛️ 雙引擎互斥選擇面板（選好引擎與模型大小後再按開始，才會按需載入該模型！）
    # ==========================================================================
    init_is_qwen = ('Qwen3' in asr_engine_choice)
    engine_toggle = widgets.ToggleButtons(
        options=[
            ('⚡ Faster-Whisper（國語／中英夾雜最快最穩）', 'faster_whisper'),
            ('🇹🇼 Qwen3-ASR（台語／國台語夾雜專用）', 'qwen3_asr')
        ],
        value=('qwen3_asr' if init_is_qwen else 'faster_whisper'),
        description='🎙️ 辨識引擎：',
        button_style='info',
        style={'description_width': '95px', 'button_width': '335px'}
    )
    dd_whisper_size = widgets.Dropdown(
        options=['large-v3-turbo', 'large-v3', 'medium', 'small'],
        value=(whisper_model_size if whisper_model_size in ('large-v3-turbo', 'large-v3', 'medium', 'small') else 'large-v3-turbo'),
        description='⚡ Whisper 模型：',
        disabled=init_is_qwen,
        style={'description_width': '125px'},
        layout=widgets.Layout(width='380px')
    )
    dd_qwen_size = widgets.Dropdown(
        options=[
            ('Qwen3-ASR-1.7B（高精度主力，約 4~7GB VRAM）', 'Qwen/Qwen3-ASR-1.7B'),
            ('Qwen3-ASR-0.6B（極速輕量，約 2~4GB VRAM）', 'Qwen/Qwen3-ASR-0.6B')
        ],
        value=(qwen_model_size if qwen_model_size in ('Qwen/Qwen3-ASR-1.7B', 'Qwen/Qwen3-ASR-0.6B') else 'Qwen/Qwen3-ASR-1.7B'),
        description='🇹🇼 Qwen3 模型：',
        disabled=(not init_is_qwen),
        style={'description_width': '125px'},
        layout=widgets.Layout(width='440px')
    )
    cb_qwen_aligner = widgets.Checkbox(
        value=bool(qwen_use_aligner),
        description='啟用 Qwen3-ForcedAligner-0.6B 精準時間軸對齊（推薦製作 SRT 字幕）',
        disabled=(not init_is_qwen),
        indent=False,
        layout=widgets.Layout(width='500px')
    )
    badge_w = widgets.HTML()
    badge_q = widgets.HTML()
    row_w = widgets.HBox([dd_whisper_size, badge_w], layout=widgets.Layout(padding='6px 10px', border_radius='8px', margin='4px 0', align_items='center'))
    row_q = widgets.HBox([dd_qwen_size, cb_qwen_aligner, badge_q], layout=widgets.Layout(padding='6px 10px', border_radius='8px', margin='4px 0', align_items='center', flex_flow='row wrap'))

    start_asr_btn = widgets.Button(
        description='🚀 載入所選模型並開始語音辨識',
        button_style='success',
        layout=widgets.Layout(width='280px', margin='6px 0 0 0')
    )
    asr_run_out = widgets.Output()
    _asr_running = [False]

    def update_asr_lock_ui(eng):
        if eng == 'faster_whisper':
            dd_whisper_size.disabled = False
            dd_qwen_size.disabled = True
            cb_qwen_aligner.disabled = True
            row_w.layout.border = '2px solid #00897b'
            row_q.layout.border = '1px dashed #cfd8dc'
            badge_w.value = '<span style="background:#e0f2f1; color:#00695c; padding:2px 9px; border-radius:99px; font-size:12px; font-weight:bold; margin-left:8px;">🟢 生效中</span>'
            badge_q.value = '<span style="background:#eceff1; color:#90a4ae; padding:2px 9px; border-radius:99px; font-size:12px; margin-left:8px;">🔒 已鎖定</span>'
        else:
            dd_whisper_size.disabled = True
            dd_qwen_size.disabled = False
            cb_qwen_aligner.disabled = False
            row_w.layout.border = '1px dashed #cfd8dc'
            row_q.layout.border = '2px solid #1565c0'
            badge_w.value = '<span style="background:#eceff1; color:#90a4ae; padding:2px 9px; border-radius:99px; font-size:12px; margin-left:8px;">🔒 已鎖定</span>'
            badge_q.value = '<span style="background:#e3f2fd; color:#1565c0; padding:2px 9px; border-radius:99px; font-size:12px; font-weight:bold; margin-left:8px;">🟢 生效中（台語／國台語夾雜專用）</span>'

    engine_toggle.observe(lambda ch: update_asr_lock_ui(ch['new']) if ch.get('new') else None, names='value')
    update_asr_lock_ui(engine_toggle.value)

    def stream_command(command, env, cwd, on_line=None):
        '''即時傳遞子行程輸出到 Notebook 介面'''
        with subprocess.Popen(command, env=env, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            for line in process.stdout:
                if on_line is None:
                    print(line, end='', flush=True)
                else:
                    on_line(line)
            status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)

    def on_click_start_asr(_):
        if _asr_running[0]:
            return
        _asr_running[0] = True
        start_asr_btn.disabled = True
        start_asr_btn.description = '⏳ 正在載入模型與辨識中...'
        asr_run_out.clear_output(wait=True)
        try:
            with asr_run_out:
                sel_engine = engine_toggle.value
                sel_model = dd_whisper_size.value if sel_engine == 'faster_whisper' else dd_qwen_size.value

                # 若使用者選擇 Qwen3-ASR 且環境尚未安裝 qwen-asr，現場自動按需安裝！
                if sel_engine == 'qwen3_asr':
                    try:
                        import qwen_asr, soundfile
                    except ImportError:
                        print('📦 首次使用 Qwen3-ASR，正在按需安裝 qwen-asr 與 soundfile 套件（約 20~40 秒）...', flush=True)
                        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-asr', 'soundfile'], check=True)

                library_dirs = []
                for mod_name in ('nvidia.cublas', 'nvidia.cudnn'):
                    try:
                        mod = __import__(mod_name, fromlist=[''])
                        lib_dir = Path(next(iter(mod.__path__))) / 'lib'
                        if lib_dir.is_dir():
                            library_dirs.append(str(lib_dir))
                    except Exception:
                        pass

                env = os.environ.copy()
                if library_dirs:
                    env['LD_LIBRARY_PATH'] = ':'.join(library_dirs + ([env['LD_LIBRARY_PATH']] if env.get('LD_LIBRARY_PATH') else []))
                env['HF_HUB_DISABLE_TELEMETRY'] = '1'
                env['ALTA_WORK_DIR'] = str(work)
                env['ALTA_ENGINE'] = sel_engine
                env['ALTA_MODEL'] = sel_model
                env['ALTA_QWEN_ALIGNER'] = '1' if cb_qwen_aligner.value else '0'
                env['ALTA_SPLIT'] = '1' if auto_split else '0'
                env['ALTA_PUNCTUATION'] = '1' if auto_punctuation else '0'
                env['ALTA_MAX_CHARS'] = str(max_subtitle_chars)
                env['ALTA_MAX_SECONDS'] = str(max_subtitle_seconds)
                env['ALTA_INTRO_MUSIC_SECONDS'] = str(intro_music_seconds)

                batch_bar = widgets.IntProgress(value=0, min=0, max=len(jobs_list), description='總批次進度', bar_style='info')
                batch_label = widgets.HTML(value=f'準備中（共 {len(jobs_list)} 個音檔）')
                progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='單檔進度', bar_style='success')
                progress_label = widgets.HTML(value='0%')

                if len(jobs_list) > 1:
                    display(widgets.VBox([
                        widgets.HBox([batch_bar, batch_label]),
                        widgets.HBox([progress_bar, progress_label])
                    ]))
                else:
                    display(widgets.HBox([progress_bar, progress_label]))

                def show_transcription_output(line):
                    if line.startswith('ALTA_JOB:'):
                        parts = line.strip()[9:].split(':', 1)
                        if len(parts) == 2 and '/' in parts[0]:
                            curr_str, _ = parts[0].split('/', 1)
                            if curr_str.isdigit():
                                batch_bar.value = int(curr_str) - 1
                                batch_label.value = f'正在處理第 {parts[0]} 個音檔：<b>{parts[1]}</b>'
                                progress_bar.value = 0
                                progress_label.value = '0%'
                    elif line.startswith('ALTA_PROGRESS:') and line.strip()[14:].isdigit():
                        percent = max(0, min(100, int(line.strip()[14:])))
                        progress_bar.value = percent
                        progress_label.value = f'{percent}%'
                        if percent == 100:
                            batch_bar.value = min(batch_bar.max, batch_bar.value + 1)
                    else:
                        if 'pad_token_id' in line or 'open-end generation' in line:
                            continue
                        print(line, end='', flush=True)

                stream_command([sys.executable, '-u', str(work / 'transcribe_worker.py')], env=env, cwd=work, on_line=show_transcription_output)
                batch_bar.value = len(jobs_list)
                batch_label.value = f'✅ 已完成全部 {len(jobs_list)} 個音檔轉錄（已自動釋放 GPU 顯存，可隨時切換另一引擎重測）！'
        finally:
            _asr_running[0] = False
            start_asr_btn.disabled = False
            start_asr_btn.description = '🚀 載入所選模型並開始語音辨識'

    start_asr_btn.on_click(on_click_start_asr)

    asr_panel = widgets.VBox([
        widgets.HTML('<div style="font-size:13.5px; font-weight:bold; color:#004d40; margin-bottom:6px;">🎙️ 雙引擎語音辨識選擇台（可先選好引擎與模型大小，按下綠色按鈕才會下載並載入該模型）：</div>'),
        engine_toggle,
        row_w,
        row_q,
        start_asr_btn
    ], layout=widgets.Layout(padding='12px 14px', border='1px solid #b2dfdb', border_radius='10px', margin='0 0 10px 0'))

    display(asr_panel, asr_run_out)

## <font color="#e65100"><b>4. 雙引擎 AI 全媒體創作工作台</b></font>

In [ ]:
# @title 🟧 **4. 雙引擎 AI 全媒體創作工作台**
# @markdown ### 🔐 模型與頻率設定
第一棒_字幕校對模型 = "gemini-3.5-flash-lite" # @param ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash-lite", "gemini-2.0-flash-lite", "gemini-flash-lite-latest"]
第二棒_深度創作模型 = "gemini-4-argon" # @param ["gemini-4-argon", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-pro", "gemini-2.5-flash"]
API安全請求間隔秒數 = 6 # @param {type:"integer"}

# @markdown ### ☑️ 產出模組勾選
啟用模組A_AI字幕精準校對 = True # @param {type:"boolean"}
自動補上智慧標點 = True # @param {type:"boolean"}
自動過濾口語贅字 = True # @param {type:"boolean"}
啟用模組B_9組爆款標題與時間軸章節 = True # @param {type:"boolean"}
啟用模組C_升級版Blog專欄文章 = True # @param {type:"boolean"}
啟用模組D_FB與IG與LINE社群貼文 = True # @param {type:"boolean"}
啟用模組E_四大視覺生圖提示詞 = True # @param {type:"boolean"}

# @markdown ### ✍️ Blog 專欄與 Blogger 推送設定
專欄系列模板 = "【T++教育+年華課程筆記】系列" # @param ["自動偵測（依音檔名稱自動判斷系列）", "【居久屋課程筆記】系列", "《噗噗聊聊》Podcast 系列", "【T++教育+年華課程筆記】系列", "一般教育思辨／科技專欄"]
嵌入Blog頂部之YouTube網址 = "https://youtu.be/wrO6-9tG0fw" # @param {type:"string"}
文章敘事人稱 = "第一人稱（講者「我」溫暖現身說法 — 推薦）" # @param ["第一人稱（講者「我」溫暖現身說法 — 推薦）", "第三人稱（客觀專欄報導與側寫）"]
文章字數深度 = "約 1500 字（深度完整專欄）" # @param ["約 1500 字（深度完整專欄）", "約 1200 字（標準好讀專欄）", "約 800 字（輕量精華筆記）"]
段落H2標題風格 = "雙段式痛點／亮點標題（預設）" # @param ["雙段式痛點／亮點標題（預設）", "帶有引發好奇的疑問句", "活潑有趣（加入適度 Emoji）"]
加入文末重點回顧表格 = True # @param {type:"boolean"}
Blogger自動推送模式 = "推送至所選 Google Blogger 後台【草稿夾】（推薦：亦可於產出後手動點擊按鈕推送）" # @param ["推送至所選 Google Blogger 後台【草稿夾】（推薦：亦可於產出後手動點擊按鈕推送）", "不自動推送（僅產出檔案，可於預覽後點擊下方按鈕手動推送至 Blogger）", "直接公開發布至所選 Google Blogger"]

# @markdown ### 🎨 社群與視覺生圖設定
社群貼文開頭鉤子風格 = "自動選擇最適鉤子" # @param ["自動選擇最適鉤子", "犀利提問開場 (Question)", "痛點共鳴開場 (Painpoint)", "真實故事開場 (Story)"]
視覺提示詞主角設定 = "講師：陳香吟老師" # @param {type:"string"}
畫面包含品牌Logo圖示 = False # @param {type:"boolean"}
視覺藝術風格 = "自動智慧匹配（依內容主題推薦）" # @param ["自動智慧匹配（依內容主題推薦）", "溫馨可愛 Q 版 (warm-cute-chibi)", "現代極簡扁平 (modern-minimalist-flat)", "3D 黏土風格 (3d-clay)", "新粗野主義 (neo-brutalism)", "高飽和 3D 渲染 (saturated-3d)", "電影級寫實 (cinematic-realistic)", "手繪塗鴉風 (doodle)"]

proofread_model = 第一棒_字幕校對模型
creative_model = 第二棒_深度創作模型
api_safe_interval_seconds = API安全請求間隔秒數
enable_ai_proofread = 啟用模組A_AI字幕精準校對
smart_punctuation = 自動補上智慧標點
remove_filler_words = 自動過濾口語贅字
generate_titles_and_chapters = 啟用模組B_9組爆款標題與時間軸章節
generate_blog_post = 啟用模組C_升級版Blog專欄文章
generate_social_posts = 啟用模組D_FB與IG與LINE社群貼文
generate_visual_prompts = 啟用模組E_四大視覺生圖提示詞
blog_series_template = 專欄系列模板
blog_youtube_url = 嵌入Blog頂部之YouTube網址
blog_pov = 文章敘事人稱
blog_length = 文章字數深度
blog_h2_style = 段落H2標題風格
blog_include_summary_table = 加入文末重點回顧表格
blogger_publish_mode = Blogger自動推送模式
social_hook_style = 社群貼文開頭鉤子風格
visual_character_desc = 視覺提示詞主角設定
visual_include_logo = 畫面包含品牌Logo圖示
visual_style_preset = 視覺藝術風格

import json
import os
import re
import shutil
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path
from IPython.display import display, clear_output, Markdown, HTML
import ipywidgets as widgets

clear_output(wait=True)
import base64
work = Path('/content/alta_transcribe_work')
work.mkdir(parents=True, exist_ok=True)

# 若使用者完全跳過步驟 1/2/3 直接點開步驟 4，自動建立 p0_subtitles.py 核心解析庫！
if not (work / 'p0_subtitles.py').is_file():
    (work / 'p0_subtitles.py').write_text("# Subtitle formatting, Chinese conversion, SRT/VTT/TXT parser, and ctb.52hal.cc deterministic cleaning utilities.\nimport math\nimport re\n\ntry:\n    from opencc import OpenCC\n    _OPENCC_INSTANCE = OpenCC('s2twp')\nexcept Exception:\n    _OPENCC_INSTANCE = None\n\n\ndef to_trad_tw(text):\n    if not text:\n        return ''\n    if _OPENCC_INSTANCE is not None:\n        try:\n            return _OPENCC_INSTANCE.convert(text)\n        except Exception:\n            return text\n    return text\n\n\ndef clean_text_deterministic(text, replace_rules=None):\n    if not text:\n        return '', 0\n    s = to_trad_tw(text.strip())\n    s = re.sub(r'^[\\[\\(【（]?\\s*(?:音樂|music|Music|MUSIC|♪+|♫+)\\s*[\\]\\)】）]?$', '【音樂】', s)\n    s = re.sub(r'\\b([A-Z])(?:\\s+([A-Z]))+\\b', lambda m: re.sub(r'\\s+', '', m.group(0)), s)\n    applied_count = 0\n    for rule in (replace_rules or []):\n        orig = rule.get('original', '')\n        repl = rule.get('replacement', '')\n        if orig and orig in s:\n            applied_count += s.count(orig)\n            s = s.replace(orig, repl)\n    return s.strip(), applied_count\n\n\ndef parse_timestamp_to_ms(ts_str):\n    s = (ts_str or '').strip().replace(',', '.')\n    parts = s.split(':')\n    try:\n        if len(parts) == 3:\n            hh = int(parts[0])\n            mm = int(parts[1])\n            sec_f = float(parts[2])\n        elif len(parts) == 2:\n            hh = 0\n            mm = int(parts[0])\n            sec_f = float(parts[1])\n        else:\n            return 0\n        return max(0, int(round((hh * 3600 + mm * 60 + sec_f) * 1000)))\n    except Exception:\n        return 0\n\n\ndef parse_existing_text_or_srt(raw_content, ext='.srt', replace_rules=None):\n    '''\n    解析現成的 .srt / .vtt / .txt / .md 或是直接貼上的純文字，並自動執行確定性錯字清洗：\n    回傳：(cues_tuples, converted_segments, total_rule_hits, has_real_timestamps, duration_sec)\n    - cues_tuples: [(start_ms, end_ms, text), ...]\n    '''\n    text_body = (raw_content or '').lstrip('\\ufeff').replace('\\r\\n', '\\n').replace('\\r', '\\n').strip()\n    if not text_body:\n        raise ValueError('匯入的文稿內容為空白！')\n\n    # 自動偵測是否包含 SRT / VTT 時間碼 (00:00:00,000 --> 00:00:03,000)\n    ts_pattern = re.compile(\n        r'((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})\\s*-->\\s*((?:\\d{1,2}:)?\\d{2}:\\d{2}[\\.,]\\d{1,3})'\n    )\n    has_ts_arrows = bool(ts_pattern.search(text_body))\n    cues = []\n    converted_segments = []\n    total_hits = 0\n\n    if has_ts_arrows:\n        blocks = re.split(r'\\n\\s*\\n+', text_body)\n        prev_end = 0\n        for blk in blocks:\n            lines = [ln.strip() for ln in blk.strip().splitlines() if ln.strip()]\n            if not lines:\n                continue\n            if lines[0].upper().startswith('WEBVTT'):\n                lines = lines[1:]\n                if not lines:\n                    continue\n            m_ts = None\n            ts_line_idx = -1\n            for idx_l, ln in enumerate(lines[:3]):\n                m = ts_pattern.search(ln)\n                if m:\n                    m_ts = m\n                    ts_line_idx = idx_l\n                    break\n            if not m_ts:\n                continue\n            start_ms = parse_timestamp_to_ms(m_ts.group(1))\n            end_ms = parse_timestamp_to_ms(m_ts.group(2))\n            if end_ms <= start_ms:\n                end_ms = start_ms + 2000\n            text_lines = lines[ts_line_idx + 1:]\n            # 自動移除既有連帶的台呼水印，避免重複疊加\n            text_lines = [tl for tl in text_lines if tl.strip() != '【ㄚ亮笑長的 Colab轉錄助手】']\n            joined_text = ' '.join(text_lines).strip()\n            if not joined_text:\n                continue\n            cleaned_t, hits = clean_text_deterministic(joined_text, replace_rules=replace_rules)\n            total_hits += hits\n            if cleaned_t:\n                cues.append((start_ms, end_ms, cleaned_t))\n                converted_segments.append({\n                    'start': round(start_ms / 1000.0, 3),\n                    'end': round(end_ms / 1000.0, 3),\n                    'text': joined_text,\n                    'text_zh_tw': cleaned_t\n                })\n                prev_end = max(prev_end, end_ms)\n\n        if cues:\n            duration_sec = max(1.0, round(prev_end / 1000.0, 2))\n            return cues, converted_segments, total_hits, True, duration_sec\n\n    # 純文字 (.txt / .md / 直接貼上文字)：智慧依换行與句號/問號/驚嘆號切分句子，賦予預估閱讀時間軸\n    raw_lines = [ln.strip() for ln in text_body.splitlines() if ln.strip()]\n    sentences = []\n    for ln in raw_lines:\n        # 若單行很長，依全形句號、問號、驚嘆號、分號適度拆句，方便字幕與段落對齊\n        if len(ln) > 45:\n            parts = re.split(r'(?<=[。！？；!?])', ln)\n            for p in parts:\n                p_s = p.strip()\n                if p_s:\n                    sentences.append(p_s)\n        else:\n            sentences.append(ln)\n\n    curr_ms = 0\n    for sent in sentences:\n        if sent == '【ㄚ亮笑長的 Colab轉錄助手】':\n            continue\n        cleaned_t, hits = clean_text_deterministic(sent, replace_rules=replace_rules)\n        total_hits += hits\n        if not cleaned_t:\n            continue\n        # 依字數估算自然語速（約每秒 4.5 字，最短 2 秒，最長 8 秒）\n        dur_ms = max(2000, min(8000, int(round(len(cleaned_t) / 4.5 * 1000))))\n        start_ms = curr_ms\n        end_ms = curr_ms + dur_ms\n        cues.append((start_ms, end_ms, cleaned_t))\n        converted_segments.append({\n            'start': round(start_ms / 1000.0, 3),\n            'end': round(end_ms / 1000.0, 3),\n            'text': sent,\n            'text_zh_tw': cleaned_t\n        })\n        curr_ms = end_ms\n\n    if not cues:\n        raise ValueError('未能從文稿中解析出任何有效句子！')\n    duration_sec = max(1.0, round(curr_ms / 1000.0, 2))\n    return cues, converted_segments, total_hits, False, duration_sec\n\n\ndef subtitle_cues(segments, duration):\n    if not isinstance(duration, (int, float)) or not math.isfinite(duration) or duration <= 0:\n        raise ValueError('無效的音訊長度')\n    max_ms = math.floor(duration * 1000)\n    cues = []\n    previous_ms = 0\n    sorted_segments = sorted(segments, key=lambda s: s['start'])\n    for segment in sorted_segments:\n        start, end = segment['start'], segment['end']\n        if not (isinstance(start, (int, float)) and isinstance(end, (int, float)) and math.isfinite(start) and math.isfinite(end)):\n            continue\n        start_ms = max(previous_ms, round(start * 1000))\n        end_ms = min(round(end * 1000), max_ms)\n        if end_ms <= start_ms:\n            end_ms = min(start_ms + 100, max_ms)\n        if start_ms >= end_ms:\n            continue\n        text = segment['text'].strip()\n        if text:\n            cues.append((start_ms, end_ms, text))\n            previous_ms = end_ms\n    if not cues:\n        raise ValueError('未產生任何有效字幕區間')\n    return cues\n\n\ndef stamp(milliseconds, separator):\n    hours, milliseconds = divmod(milliseconds, 3600000)\n    minutes, milliseconds = divmod(milliseconds, 60000)\n    seconds, milliseconds = divmod(milliseconds, 1000)\n    return f'{hours:02}:{minutes:02}:{seconds:02}{separator}{milliseconds:03}'\n\n\ndef render(cues, kind):\n    if kind not in ('srt', 'vtt'):\n        raise ValueError('不支援的字幕格式')\n    separator = ',' if kind == 'srt' else '.'\n    blocks = []\n    for index, (start, end, text) in enumerate(cues, 1):\n        prefix = (str(index) + chr(10)) if kind == 'srt' else ''\n        time_part = stamp(start, separator) + ' --> ' + stamp(end, separator)\n        blocks.append(prefix + time_part + chr(10) + text)\n    header = ('WEBVTT' + chr(10) + chr(10)) if kind == 'vtt' else ''\n    return header + (chr(10) + chr(10)).join(blocks) + chr(10)\n\n\ndef traditional_segments(segments, replace_rules=None):\n    out = []\n    total_replaced = 0\n    for segment in segments:\n        tw_text = to_trad_tw(segment['text'])\n        cleaned_text, count = clean_text_deterministic(tw_text, replace_rules=replace_rules)\n        total_replaced += count\n        out.append({**segment, 'text_zh_tw': cleaned_text})\n    return out, total_replaced\n\n\ndef _timed_chars(segment):\n    text = segment['text']\n    words = segment.get('words') or []\n    words_clean = re.sub(r'\\s+', '', ''.join(w['word'] for w in words))\n    text_clean = re.sub(r'\\s+', '', text)\n    if words_clean == text_clean and len(words) > 0 and all(\n        isinstance(w.get('start'), (int, float)) and isinstance(w.get('end'), (int, float))\n        and math.isfinite(w['start']) and math.isfinite(w['end'])\n        and segment['start'] <= w['start'] < w['end'] <= segment['end']\n        for w in words\n    ):\n        result = []\n        for word in words:\n            value = word['word'].strip()\n            if not value:\n                continue\n            span = word['end'] - word['start']\n            for i, ch in enumerate(value):\n                c_start = word['start'] + span * (i / len(value))\n                c_end = word['start'] + span * ((i + 1) / len(value))\n                result.append((ch, c_start, c_end))\n        if result:\n            return result, 'word_timestamps'\n\n    span = max(0.01, segment['end'] - segment['start'])\n    return [(ch, segment['start'] + span * i / len(text),\n             segment['start'] + span * (i + 1) / len(text))\n            for i, ch in enumerate(text)], 'segment_interpolated'\n\n\ndef build_outputs_from_cues(cues, converted_segments, intro_music_seconds=0, txt_lines=None):\n    srt_cues = list(cues)\n    vtt_cues = list(cues)\n\n    first_start, first_end, first_text = srt_cues[0]\n    promo_text = '【ㄚ亮笑長的 Colab轉錄助手】'\n\n    if intro_music_seconds > 0:\n        music_ms = round(intro_music_seconds * 1000)\n        if music_ms <= first_start:\n            srt_cues.insert(0, (0, music_ms, '【音樂】' + chr(10) + promo_text))\n            vtt_cues.insert(0, (0, music_ms, '【音樂】'))\n        else:\n            if promo_text not in first_text:\n                srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n    else:\n        if promo_text not in first_text:\n            srt_cues[0] = (first_start, first_end, first_text + chr(10) + promo_text)\n\n    if txt_lines is None:\n        txt_lines = [row['text_zh_tw'] for row in converted_segments]\n\n    outputs = {\n        'transcript.txt': chr(10).join(txt_lines) + chr(10),\n        'transcript.srt': render(srt_cues, 'srt'),\n        'transcript.vtt': render(vtt_cues, 'vtt')\n    }\n    return outputs\n\n\ndef friendly_exports(segments, duration, split=True, punctuation=False, max_chars=22, max_seconds=6, intro_music_seconds=0, replace_rules=None):\n    if max_chars < 4 or max_seconds <= 0:\n        raise ValueError('無效的字幕長度設定')\n    display = []\n    timing_methods = set()\n    total_rule_hits = 0\n    for index, segment in enumerate(segments):\n        raw_text = segment['text'].strip()\n        if not raw_text:\n            continue\n        chars, method = _timed_chars(segment)\n        if segment['end'] > duration and segment['start'] < duration:\n            scale = (duration - segment['start']) / (segment['end'] - segment['start'])\n            chars = [(char, segment['start'] + (start - segment['start']) * scale,\n                      segment['start'] + (end - segment['start']) * scale)\n                     for char, start, end in chars]\n            method = 'segment_end_adjusted'\n        timing_methods.add(method)\n        start = 0\n        while start < len(chars):\n            end = start + 1\n            limit_break = False\n            while end < len(chars):\n                candidate = ''.join(c[0] for c in chars[start:end + 1])\n                if split and (len(to_trad_tw(candidate)) > max_chars or\n                              chars[end][2] - chars[start][1] > max_seconds):\n                    limit_break = True\n                    break\n                if split and chars[end - 1][0] in '。！？；!?;':\n                    break\n                if split and punctuation and chars[end][1] - chars[end - 1][2] >= 0.65:\n                    break\n                end += 1\n            if (limit_break and end == len(chars) - 1 and end - start > 1\n                    and chars[end - 1][0].isalnum() and chars[end][0].isalnum()\n                    and chars[end][1] - chars[end - 1][2] < 0.65\n                    and chars[end][2] - chars[end - 1][1] <= max_seconds\n                    and len(to_trad_tw(chars[end - 1][0] + chars[end][0])) <= max_chars):\n                end -= 1\n            text = to_trad_tw(''.join(c[0] for c in chars[start:end])).strip()\n            text, hits = clean_text_deterministic(text, replace_rules=replace_rules)\n            total_rule_hits += hits\n            if text:\n                last_chunk = end == len(chars)\n                next_start = segments[index + 1]['start'] if index + 1 < len(segments) else duration\n                gap = next_start - segment['end'] if last_chunk else chars[end][1] - chars[end - 1][2]\n                if punctuation and gap >= 0.65 and text[-1] not in '，。！？；：、,.!?;:':\n                    text += '。'\n                cue_start, cue_end = chars[start][1], chars[end - 1][2]\n                display.append({'start': cue_start, 'end': cue_end, 'text': text})\n            start = end\n\n    cues = subtitle_cues(display, duration)\n    converted, seg_hits = traditional_segments(segments, replace_rules=replace_rules)\n    outputs = build_outputs_from_cues(cues, converted, intro_music_seconds=intro_music_seconds, txt_lines=[t for _, _, t in cues])\n    return converted, outputs, sorted(timing_methods), cues, max(total_rule_hits, seg_hits)\n", encoding='utf-8')

if str(work) not in sys.path:
    sys.path.insert(0, str(work))
from p0_subtitles import build_outputs_from_cues, clean_text_deterministic, parse_existing_text_or_srt

queue_path = work / 'job_queue.json'
queue_data = json.loads(queue_path.read_text(encoding='utf-8')) if queue_path.is_file() else {}
dict_cfg = queue_data.get('dictionary') or {}
replace_rules = dict_cfg.get('replace_rules') or []
positive_terms = dict_cfg.get('positive_terms') or ['ㄚ亮笑長', '噗噗聊聊', '居久屋微醺夜', '居久屋課程筆記', '謙懿科技', '傑可達數位', 'T++教育+年華']
negative_terms = dict_cfg.get('negative_terms') or []
auto_save_flag = queue_data.get('auto_save_to_drive', True)
custom_drive_folder = queue_data.get('drive_output_folder') or 'Colab Notebooks/轉錄成果_ㄚ亮笑長助手'


def scan_ready_job_dirs():
    dirs = []
    j_root = work / 'jobs'
    if j_root.is_dir():
        for d in sorted(j_root.iterdir()):
            if d.is_dir() and (d / 'raw_segments.json').is_file():
                dirs.append(d)
    if not dirs and (work / 'raw_segments.json').is_file():
        dirs = [work]
    return dirs


job_dirs = scan_ready_job_dirs()


def extract_yt_id(url_or_id):
    s = (url_or_id or '').strip()
    if not s:
        return ''
    if re.fullmatch(r'[A-Za-z0-9_-]{11}', s):
        return s
    m = re.search(r'(?:v=|youtu\.be/|youtube\.com/(?:embed|v|shorts|live)/)([A-Za-z0-9_-]{11})', s)
    return m.group(1) if m else ''


def get_series_cta_and_tags(series_choice, stem_name):
    '''根據系列選單或檔名自動匹配「丫亮笑長練功坊」三大系列的標準結尾 CTA HTML 與預設標籤'''
    s = series_choice
    if s.startswith('自動偵測'):
        if '噗噗聊聊' in stem_name or (re.search(r'EP\d+', stem_name, re.I) and '居久屋' not in stem_name):
            s = '《噗噗聊聊》Podcast 系列'
        elif 'T++' in stem_name or '年華' in stem_name:
            s = '【T++教育+年華課程筆記】系列'
        else:
            s = '【居久屋課程筆記】系列'

    if '噗噗聊聊' in s:
        cta_html = (
            '<h2>喜歡噗噗聊聊嗎？</h2>' + chr(10) +
            '<p>如果你想要了解更多關於教育及 <a href="https://bit.ly/PuChatPodcast" rel="noopener noreferrer" target="_blank">Podcast</a> 的內容，歡迎追蹤我們的節目，一起探索教育的無限可能。</p>' + chr(10) +
            '<ul>' + chr(10) +
            '  <li><a href="https://bit.ly/PuChatFB" rel="noopener noreferrer" target="_blank">噗噗聊聊粉絲專頁</a></li>' + chr(10) +
            '  <li><a href="https://bit.ly/PuChatYT" rel="noopener noreferrer" target="_blank">噗噗聊聊Youtube頻道</a></li>' + chr(10) +
            '  <li><a href="https://bit.ly/PuChatPodcast" rel="noopener noreferrer" target="_blank">噗噗聊聊Podcast</a></li>' + chr(10) +
            '  <li><a href="https://bit.ly/aliangblog" rel="noopener noreferrer" target="_blank">ㄚ亮笑長練功坊Blog</a></li>' + chr(10) +
            '</ul>'
        )
        base_tags = ['噗噗聊聊', '教育現場', '數位教學']
        series_prefix = '噗噗聊聊 '
    elif 'T++' in s:
        cta_html = (
            '<h2>🏃‍♂️ 手刀追蹤我們，精采不錯過~</h2>' + chr(10) +
            '<ul>' + chr(10) +
            '  <li>✅ 官方LINE：<a href="https://lin.ee/f6w421d" rel="noopener noreferrer" target="_blank">https://lin.ee/f6w421d</a></li>' + chr(10) +
            '  <li>✅ IG追蹤：<a href="https://www.instagram.com/t.plus.plus/" rel="noopener noreferrer" target="_blank">https://www.instagram.com/t.plus.plus/</a></li>' + chr(10) +
            '  <li>✅ FB粉專：<a href="https://www.facebook.com/tplusplus.tw" rel="noopener noreferrer" target="_blank">https://www.facebook.com/tplusplus.tw</a></li>' + chr(10) +
            '</ul>'
        )
        base_tags = ['T++', 'T++教育+年華', '教育', '教學創新']
        series_prefix = '【T++教育+年華課程筆記】'
    else:
        cta_html = (
            '<h2>🎁 喜歡我們的課程嗎？</h2>' + chr(10) +
            '<p>如果你想要學習更多學科教學知識與科技應用，歡迎訂閱謙懿科技 Youtube 頻道，記得按讚追蹤我們的節目，一起探索教育的無限可能。</p>' + chr(10) +
            '<ul>' + chr(10) +
            '  <li>謙懿科技Youtube：<a href="https://www.youtube.com/@morganfang0905" rel="noopener noreferrer" target="_blank">http://www.youtube.com/@morganfang0905</a></li>' + chr(10) +
            '  <li>ㄚ亮笑長練功坊Blog：<a href="https://bit.ly/aliangblog" rel="noopener noreferrer" target="_blank">https://bit.ly/aliangblog</a></li>' + chr(10) +
            '</ul>'
        )
        base_tags = ['居久屋微醺夜', 'AI教學應用', '數位教學']
        series_prefix = '【居久屋課程筆記】'

    return cta_html, base_tags, series_prefix


def ensure_episode_drive_dir(seg_data):
    '''
    確保取得雲端硬碟「轉錄成果_ㄚ亮笑長助手/{音檔同名資料夾}」路徑：
    若先前步驟 2 因本機上傳未掛載雲端導致 drive_save_dir 為空，此處會自動補掛載並建立同名子資料夾！
    '''
    stem = seg_data.get('stem') or '轉錄結果'
    safe_stem = re.sub(r'[\\/:*?"<>|]+', '_', stem.strip())
    drive_save_dir = seg_data.get('drive_save_dir') or ''

    if not drive_save_dir and auto_save_flag:
        my_drive_root = Path('/content/drive/MyDrive')
        if not my_drive_root.exists():
            try:
                from google.colab import drive
                print('💾 正在掛載 Google 雲端硬碟以自動保存全媒體成果...', flush=True)
                drive.mount('/content/drive', timeout_ms=120000)
            except Exception as m_err:
                print(f'⚠️ 無法掛載雲端硬碟（請透過步驟 5 下載 ZIP）：{m_err}', flush=True)
                return ''
        p = Path(custom_drive_folder)
        base_dir = (my_drive_root / p) if not p.is_absolute() else p
        drive_save_dir = str(base_dir / safe_stem)

    if drive_save_dir:
        d_path = Path(drive_save_dir)
        # 確保最後一層是與音檔同名的子資料夾
        if d_path.name != safe_stem:
            d_path = d_path / safe_stem
        return str(d_path)
    return ''


def strip_md_to_plain(text):
    '''徹底清除 Markdown 語法符號（**、###、`、---），保留乾淨的純文字與空行排版，直貼 YT 簡介或 FB/IG/LINE 免修字！'''
    if not text:
        return ''
    s = text.replace('\r\n', '\n')
    # 移除程式碼區塊標記 ```text 或 ```
    s = re.sub(r'^\s*```[a-zA-Z]*\s*$', '', s, flags=re.M)
    # 移除標題井號 ###
    s = re.sub(r'^\s*#{1,6}\s*', '', s, flags=re.M)
    # 移除粗體 **text** 與 __text__ 與行內反引號 `text`
    s = re.sub(r'\*\*(.+?)\*\*', r'\1', s)
    s = re.sub(r'__(.+?)__', r'\1', s)
    s = re.sub(r'`([^`]+)`', r'\1', s)
    # 將開頭的 - 或 * 清單符號統一轉為清爽的 ▸
    s = re.sub(r'^(\s*)[\-\*]\s+', r'\1▸ ', s, flags=re.M)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()


def extract_between_tags(raw_text, start_tag, end_tag):
    m = re.search(re.escape(start_tag) + r'([\s\S]*?)' + re.escape(end_tag), raw_text or '', re.I)
    return m.group(1).strip() if m else ''


def build_copy_btn_html(text_to_copy, label='📋 一鍵複製', bg='#00796b', small=False):
    '''產生獨立 HTML 內嵌的一鍵複製按鈕（採用 base64 UTF-8 解碼，無論在 Colab 內或下載回電腦點開 HTML 皆永久有效！）'''
    b64 = base64.b64encode((text_to_copy or '').encode('utf-8')).decode('ascii')
    pad = '4px 10px' if small else '7px 14px'
    fs = '12px' if small else '13px'
    js = (
        "const b=this.getAttribute('data-b64');"
        "const bytes=Uint8Array.from(atob(b),c=>c.charCodeAt(0));"
        "const txt=new TextDecoder('utf-8').decode(bytes);"
        "navigator.clipboard.writeText(txt).then(()=>{"
        "const orig=this.innerHTML;const obg=this.style.background;"
        "this.innerHTML='✅ 已複製！';this.style.background='#2e7d32';"
        "setTimeout(()=>{this.innerHTML=orig;this.style.background=obg;},2000);});"
    )
    return (
        f'<button data-b64="{b64}" onclick="{js}" '
        f'style="background:{bg}; color:#ffffff; border:none; border-radius:6px; padding:{pad}; font-size:{fs}; font-weight:bold; cursor:pointer; box-shadow:0 1px 3px rgba(0,0,0,0.16); white-space:nowrap;">'
        f'{label}</button>'
    )


def html_escape(s):
    return (s or '').replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')


def build_all_in_one_warroom_html(
    stem,
    readable_txt='',
    srt_txt='',
    mod_b_raw='',
    yt_desc_plain='',
    yt_chapters_plain='',
    blog_title='',
    blog_html='',
    seo_data=None,
    social_raw='',
    visual_raw=''
):
    '''
    將所有產出依序整理為一份高質感一頁式「🌟 全媒體創作戰情室.html」！
    每一個區塊、每一組標題、每一篇社群貼文、每一張生圖提示詞皆附有獨立「📋 一鍵複製」按鈕！
    '''
    seo_data = seo_data or {}
    # 1. 解析爆款標題區塊
    titles_sec = extract_between_tags(mod_b_raw, '[TITLES_START]', '[TITLES_END]') or mod_b_raw
    titles_plain = strip_md_to_plain(titles_sec)
    # 擷取每一行含主副標題的句子提供單行快速複製
    individual_titles = []
    for ln in titles_plain.splitlines():
        ln_s = ln.strip()
        if any(k in ln_s for k in ('正選', '備選', '推薦首選標題')) and ('：' in ln_s or ':' in ln_s):
            parts = re.split(r'[：:]', ln_s, maxsplit=1)
            if len(parts) == 2 and len(parts[1].strip()) >= 5:
                t_val = re.sub(r'[（(]\s*設計概念[：:].*$', '', parts[1].strip()).strip()
                if t_val:
                    individual_titles.append((parts[0].strip(), t_val))

    title_rows_html = ''
    for badge_lbl, t_val in individual_titles[:12]:
        btn_h = build_copy_btn_html(t_val, '📋 複製此標題', bg='#00897b', small=True)
        title_rows_html += (
            '<div style="display:flex; justify-content:space-between; align-items:center; gap:10px; padding:8px 12px; margin:5px 0; background:#f4fbfa; border:1px solid #b2dfdb; border-radius:8px;">'
            f'<div style="font-size:14px; color:#004d40;"><span style="background:#00796b; color:#fff; padding:2px 7px; border-radius:4px; font-size:12px; margin-right:8px;">{html_escape(badge_lbl)}</span><b>{html_escape(t_val)}</b></div>'
            f'{btn_h}</div>'
        )

    # 2. 解析三平台社群貼文（已完全去 Markdown 化）
    fb_plain = strip_md_to_plain(extract_between_tags(social_raw, '[FB_POST_START]', '[FB_POST_END]'))
    ig_plain = strip_md_to_plain(extract_between_tags(social_raw, '[IG_POST_START]', '[IG_POST_END]'))
    line_plain = strip_md_to_plain(extract_between_tags(social_raw, '[LINE_POST_START]', '[LINE_POST_END]'))
    if not (fb_plain or ig_plain or line_plain) and social_raw:
        fb_plain = strip_md_to_plain(social_raw)

    # 3. 解析四大視覺生圖提示詞（輪播圖 4 張合為單一複製鈕、Reels 5 張合為單一複製鈕）
    carousel_all_raw = extract_between_tags(visual_raw, '[CAROUSEL_ALL_START]', '[CAROUSEL_ALL_END]')
    reels_all_raw = extract_between_tags(visual_raw, '[REELS_ALL_START]', '[REELS_ALL_END]')
    reels_caption_raw = extract_between_tags(visual_raw, '[REELS_CAPTION_START]', '[REELS_CAPTION_END]')
    infographic_zh_raw = extract_between_tags(visual_raw, '[INFOGRAPHIC_ZH_START]', '[INFOGRAPHIC_ZH_END]')
    infographic_en_raw = extract_between_tags(visual_raw, '[INFOGRAPHIC_EN_START]', '[INFOGRAPHIC_EN_END]')
    yt_thumb_raw = extract_between_tags(visual_raw, '[YT_THUMB_START]', '[YT_THUMB_END]')

    visual_cards_defs = [
        ('🎠 社群輪播圖 1:1（共 4 張獨立生圖任務 — 一鍵複製全部 4 張指令）', '📋 一鍵複製輪播圖 4 張提示詞', carousel_all_raw, False),
        ('📽️ Reels 9:16 直式分鏡（共 5 張獨立生圖任務 — 一鍵複製全部 5 張指令）', '📋 一鍵複製 Reels 5 張提示詞', reels_all_raw, False),
        ('📝 Reels 短影音上架文案 & Hashtags', '📋 複製 Reels 上架文案', reels_caption_raw, True),
        ('📊 資訊圖表 Infographic【繁體中文生圖提示詞 + 推薦佈局】', '📋 複製資訊圖表(中文)提示詞', infographic_zh_raw, True),
        ('📊 資訊圖表 Infographic【英文生圖提示詞 (English Prompt)】', '📋 複製資訊圖表(英文)提示詞', infographic_en_raw, True),
        ('🎬 YouTube 16:9 封面六段式生圖提示詞', '📋 複製 YT 封面提示詞', yt_thumb_raw, True),
    ]
    visual_cards_html = ''
    has_split_vis = any(bool(v_txt) for _, _, v_txt, _ in visual_cards_defs)
    if has_split_vis:
        for v_title, btn_lbl, v_txt, strip_flag in visual_cards_defs:
            if not v_txt:
                continue
            v_clean = strip_md_to_plain(v_txt) if strip_flag else v_txt.strip()
            v_btn = build_copy_btn_html(v_clean, btn_lbl, bg='#6a1b9a', small=False)
            visual_cards_html += (
                '<div style="background:#faf5ff; border:1px solid #e1bee7; border-radius:10px; padding:12px 14px; margin-bottom:12px;">'
                '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:8px;">'
                f'<b style="color:#4a148c; font-size:14.5px;">{html_escape(v_title)}</b>{v_btn}</div>'
                f'<pre style="margin:0; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:13.5px; color:#263238; background:#ffffff; padding:12px; border-radius:6px; border:1px solid #f3e5f5; max-height:380px; overflow:auto; line-height:1.68;">{html_escape(v_clean)}</pre>'
                '</div>'
            )
    elif visual_raw:
        v_clean_all = strip_md_to_plain(visual_raw)
        v_btn_all = build_copy_btn_html(v_clean_all, '📋 複製全部視覺提示詞', bg='#6a1b9a')
        visual_cards_html = (
            '<div style="background:#faf5ff; border:1px solid #e1bee7; border-radius:10px; padding:12px 14px;">'
            f'<div style="margin-bottom:8px;">{v_btn_all}</div>'
            f'<pre style="margin:0; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:13.5px; color:#263238; line-height:1.65;">{html_escape(v_clean_all)}</pre>'
            '</div>'
        )

    labels_str = ', '.join(seo_data.get('labels') or [])
    meta_desc_str = seo_data.get('meta_description') or ''
    slug_str = seo_data.get('slug') or ''

    sections_html = []

    # 區塊 1：🔥 9 組爆款主題與標題
    if titles_plain:
        sec1 = (
            '<div id="sec-titles" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #e65100; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:12px;">'
            '<h2 style="margin:0; font-size:18px; color:#bf360c;">🔥 一、9 組爆款主題與標題建議（點右側按鈕直接複製單一標題）</h2>'
            + build_copy_btn_html(titles_plain, '📋 複製全部 9 組標題分析', bg='#e65100') +
            '</div>'
            + (f'<div style="margin-bottom:12px;">{title_rows_html}</div>' if title_rows_html else '') +
            '<details style="margin-top:8px;"><summary style="cursor:pointer; color:#bf360c; font-weight:bold; font-size:13.5px;">📂 展開檢視 3 大方案完整設計概念與分析</summary>'
            f'<pre style="margin-top:8px; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:13.5px; background:#fff8f5; padding:12px; border-radius:8px; border:1px solid #ffccbc; line-height:1.7;">{html_escape(titles_plain)}</pre>'
            '</details></div>'
        )
        sections_html.append(sec1)

    # 區塊 2：🎬 YouTube / Podcast 簡介欄（無 Markdown 純文字直貼版！）
    if yt_desc_plain:
        sec2 = (
            '<div id="sec-yt" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #d32f2f; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:10px;">'
            '<div><h2 style="margin:0; font-size:18px; color:#b71c1c;">🎬 二、YouTube / Podcast 資訊欄簡介 & 時間軸章節（純文字免刪符號・直貼 YT！）</h2>'
            '<div style="font-size:12.5px; color:#546e7a; margin-top:2px;">✨ 已徹底清除 Markdown 星號與井號，複製後可直接貼入 YouTube 影片說明欄，00:00 時間軸將自動生效！</div></div>'
            '<div style="display:flex; gap:8px; flex-wrap:wrap;">'
            + build_copy_btn_html(yt_desc_plain, '📋 一鍵複製完整 YT 簡介欄（含摘要＋時間軸）', bg='#d32f2f')
            + (build_copy_btn_html(yt_chapters_plain, '⏱️ 僅複製 00:00 時間軸', bg='#455a64') if yt_chapters_plain else '') +
            '</div></div>'
            f'<pre style="margin:0; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:14px; color:#263238; background:#fffafa; padding:14px; border-radius:8px; border:1px solid #ffcdd2; line-height:1.75;">{html_escape(yt_desc_plain)}</pre>'
            '</div>'
        )
        sections_html.append(sec2)

    # 區塊 3：✍️ 升級版 Blog 專欄文章 & Blogger SEO 設定包（含自訂永久連結按鈕）
    if blog_html:
        sec3 = (
            '<div id="sec-blog" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #00796b; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:12px;">'
            '<h2 style="margin:0; font-size:18px; color:#004d40;">✍️ 三、升級版 Blog 專欄文章 & Blogger 後台欄位一鍵複製</h2>'
            '<div style="display:flex; gap:6px; flex-wrap:wrap;">'
            + build_copy_btn_html(blog_title, '📋 複製文章標題', bg='#00897b', small=True)
            + (build_copy_btn_html(labels_str, '🏷️ 複製分類標籤', bg='#00897b', small=True) if labels_str else '')
            + (build_copy_btn_html(slug_str, '🔗 複製自訂永久連結', bg='#00897b', small=True) if slug_str else '')
            + (build_copy_btn_html(meta_desc_str, '🔍 複製搜尋描述 (150字)', bg='#00897b', small=True) if meta_desc_str else '')
            + build_copy_btn_html(blog_html, '📋 一鍵複製完整 Blog HTML 原始碼', bg='#004d40') +
            '</div></div>'
            '<div style="background:#f4fbfa; border:1px solid #b2dfdb; border-radius:8px; padding:10px 14px; margin-bottom:14px; font-size:13px; color:#004d40; line-height:1.75;">'
            f'<div><b>📌 建議文章標題：</b>{html_escape(blog_title)}</div>'
            + (f'<div><b>🏷️ Blogger 分類標籤（Labels）：</b><code>{html_escape(labels_str)}</code></div>' if labels_str else '')
            + (f'<div><b>🔗 自訂永久連結（Permalink Slug）：</b><code>{html_escape(slug_str)}</code></div>' if slug_str else '')
            + (f'<div><b>🔍 Meta 搜尋描述（150字內）：</b>{html_escape(meta_desc_str)}</div>' if meta_desc_str else '') +
            '</div>'
            '<div style="padding:18px; background:#ffffff; border:1px solid #e0e0e0; border-radius:10px;">'
            + blog_html +
            '</div></div>'
        )
        sections_html.append(sec3)

    # 區塊 4：📱 FB / IG / LINE 三平台社群宣傳貼文（獨立複製按鈕）
    if fb_plain or ig_plain or line_plain:
        soc_cards = ''
        for p_name, p_color, p_bg, p_txt in [
            ('📘 Facebook 專頁貼文（直貼免改字）', '#1565c0', '#f0f7ff', fb_plain),
            ('📸 Instagram 貼文（含高觸及 Hashtags）', '#ad1457', '#fdf2f8', ig_plain),
            ('💬 LINE 社群／教師群組推播短文案', '#2e7d32', '#f2fbf4', line_plain),
        ]:
            if not p_txt:
                continue
            soc_cards += (
                f'<div style="background:{p_bg}; border:1px solid #cfd8dc; border-radius:10px; padding:12px 14px; margin-bottom:10px;">'
                '<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">'
                f'<b style="color:{p_color}; font-size:14.5px;">{p_name}</b>'
                + build_copy_btn_html(p_txt, f'📋 複製 {p_name.split()[1]} 文案', bg=p_color, small=True) +
                '</div>'
                f'<pre style="margin:0; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:13.5px; color:#263238; background:#ffffff; padding:12px; border-radius:8px; border:1px solid #e0e0e0; line-height:1.7;">{html_escape(p_txt)}</pre>'
                '</div>'
            )
        sec4 = (
            '<div id="sec-social" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #1565c0; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<h2 style="margin:0 0 12px 0; font-size:18px; color:#0d47a1;">📱 四、FB / IG / LINE 三平台社群貼文（已去 Markdown 符號，一鍵複製直貼）</h2>'
            + soc_cards +
            '</div>'
        )
        sections_html.append(sec4)

    # 區塊 5：🎨 四大視覺生圖提示詞大禮包（每張圖獨立複製卡）
    if visual_cards_html:
        sec5 = (
            '<div id="sec-visual" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #6a1b9a; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<h2 style="margin:0 0 12px 0; font-size:18px; color:#4a148c;">🎨 五、四大視覺生圖提示詞大禮包（每張圖卡皆可獨立一鍵複製生圖！）</h2>'
            + visual_cards_html +
            '</div>'
        )
        sections_html.append(sec5)

    # 區塊 6：📄 高可讀性全本順稿文稿 (.txt) & 校對後 SRT 字幕 (.srt)
    if readable_txt:
        sec6 = (
            '<div id="sec-transcript" style="background:#ffffff; border:1px solid #cfd8dc; border-top:4px solid #37474f; border-radius:12px; padding:18px 20px; margin-bottom:18px; box-shadow:0 2px 8px rgba(0,0,0,0.04);">'
            '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:10px;">'
            '<div><h2 style="margin:0; font-size:18px; color:#263238;">📄 六、高可讀性全本順稿文稿（同步存為 <code>' + html_escape(stem) + '.txt</code>）</h2>'
            '<div style="font-size:12.5px; color:#546e7a; margin-top:2px;">完整保留原音檔敘述脈絡與細節，已分段並加上標點與段落小標，適合閱讀、發講義或匯入 NotebookLM。</div></div>'
            '<div style="display:flex; gap:8px; flex-wrap:wrap;">'
            + build_copy_btn_html(readable_txt, '📋 一鍵複製全本順稿 (.txt)', bg='#37474f')
            + (build_copy_btn_html(srt_txt, '🔤 複製修正後 SRT 字幕全文', bg='#00796b') if srt_txt else '') +
            '</div></div>'
            f'<pre style="margin:0; white-space:pre-wrap; word-break:break-word; font-family:inherit; font-size:14px; color:#263238; background:#f8fafb; padding:16px; border-radius:8px; border:1px solid #cfd8dc; max-height:480px; overflow:auto; line-height:1.8;">{html_escape(readable_txt)}</pre>'
            '</div>'
        )
        sections_html.append(sec6)

    nav_bar = (
        '<div style="position:sticky; top:0; z-index:99; background:linear-gradient(135deg, #004d40 0%, #00796b 100%); color:#fff; padding:12px 18px; border-radius:10px; margin-bottom:16px; box-shadow:0 3px 10px rgba(0,0,0,0.15); display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">'
        f'<div style="font-size:16px; font-weight:bold;">🌟 {html_escape(stem)} — 全媒體創作戰情室</div>'
        '<div style="display:flex; gap:6px; flex-wrap:wrap; font-size:12.5px;">'
        '<a href="#sec-titles" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">🔥 1.爆款標題</a>'
        '<a href="#sec-yt" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">🎬 2.YT簡介&時間軸</a>'
        '<a href="#sec-blog" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">✍️ 3.Blog專欄</a>'
        '<a href="#sec-social" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">📱 4.社群貼文</a>'
        '<a href="#sec-visual" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">🎨 5.生圖提示詞</a>'
        '<a href="#sec-transcript" style="color:#fff; background:rgba(255,255,255,0.18); padding:4px 10px; border-radius:99px; text-decoration:none;">📄 6.全本順稿</a>'
        '</div></div>'
    )

    body_combined = nav_bar + chr(10).join(sections_html)
    standalone_html = (
        '<!DOCTYPE html><html lang="zh-TW"><head><meta charset="utf-8">'
        f'<title>{html_escape(stem)} - 全媒體創作戰情室</title>'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        '<style>body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI","Noto Sans TC","PingFang TC","Microsoft JhengHei",sans-serif; background:#f0f4f8; margin:0; padding:20px; color:#263238;} .container{max-width:1020px; margin:0 auto;}</style>'
        f'</head><body><div class="container">{body_combined}</div></body></html>'
    )
    return standalone_html, body_combined


def sync_job_to_drive(job_dir, seg_data):
    '''
    極簡黃金 3 檔歸檔：
    將單一作品的成果存回 Google 雲端硬碟「轉錄成果_ㄚ亮笑長助手/{音檔同名資料夾}/」，
    只保留最實用的 3 個核心檔案（外加一個打包好的 ZIP）：
      1. {音檔名}.srt（修正後最終字幕）
      2. {音檔名}.txt（高可讀性、保留原敘述脈絡的完整順稿文稿）
      3. {音檔名}_全媒體創作戰情室.html（一頁式含所有複製按鈕的總覽網頁）
    '''
    drive_save_dir = ensure_episode_drive_dir(seg_data)
    stem = seg_data.get('stem') or '轉錄結果'
    if not drive_save_dir:
        return
    try:
        d_dir = Path(drive_save_dir)
        d_dir.mkdir(parents=True, exist_ok=True)

        # 自動清理舊版曾留下的零碎檔案，讓雲端資料夾保持極致清爽
        legacy_patterns = [
            f'{stem}.vtt', f'{stem}.segments.json', f'{stem}_原始未校對.srt',
            f'{stem}_部落格專欄文章.html', f'{stem}_部落格專欄文章_含SEO.md',
            f'{stem}_部落格SEO與標籤設定.json', f'{stem}_爆款標題與ShowNotes章節.md',
            f'{stem}_YouTube與Podcast時間軸章節.txt', f'{stem}_FB_IG_LINE社群貼文.md',
            f'{stem}_四大視覺生圖提示詞大禮包.md', f'{stem}.zip'
        ]
        for lp in legacy_patterns:
            old_p = d_dir / lp
            if old_p.is_file():
                try:
                    old_p.unlink()
                except Exception:
                    pass

        file_map = [
            ('transcript.srt', f'{stem}.srt'),
            ('transcript.txt', f'{stem}.txt'),
            ('all_in_one_hub.html', f'{stem}_全媒體創作戰情室.html'),
        ]
        packed_pairs = []
        for local_name, remote_name in file_map:
            src_f = job_dir / local_name
            if src_f.is_file():
                shutil.copy2(src_f, d_dir / remote_name)
                packed_pairs.append((src_f, remote_name))

        zip_p = d_dir / f'{stem}_全媒體創作包.zip'
        with zipfile.ZipFile(zip_p, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            for src_f, remote_name in packed_pairs:
                zf.write(src_f, arcname=remote_name)
        print(f'💾 已同步精簡版 3 大核心成果至雲端專屬子資料夾：{d_dir}/ （{stem}.srt ＋ {stem}.txt ＋ {stem}_全媒體創作戰情室.html）', flush=True)
    except Exception as err:
        print(f'⚠️ 同步存回雲端時發生警告：{err}', flush=True)


# ==============================================================================
# 🔗 Google Blogger 一鍵連結 ＋ 自動抓取個人網誌下拉選單 ＋ 雲端永久記憶管理員
# ==============================================================================
OAUTH_PLAYGROUND_BLOGGER_URL = 'https://developers.google.com/oauthplayground/'
BLOGGER_OAUTH_SCOPE = 'https://www.googleapis.com/auth/blogger'


def get_blogger_memory_dir():
    '''取得使用者個人 Google 雲端硬碟中的永久記憶資料夾（每位使用者各自獨立，分享 Colab 絕不外洩帳號）'''
    mem_dir_str = dict_cfg.get('memory_dir') or ''
    if mem_dir_str and Path(mem_dir_str).exists():
        return Path(mem_dir_str)
    my_drive_root = Path('/content/drive/MyDrive')
    if not my_drive_root.exists() and auto_save_flag:
        try:
            from google.colab import drive
            drive.mount('/content/drive', timeout_ms=120000)
        except Exception:
            pass
    if my_drive_root.exists():
        d = my_drive_root / 'Colab Notebooks' / 'CTB_詞庫設定_永久記憶'
        d.mkdir(parents=True, exist_ok=True)
        return d
    local_mem = work / 'blogger_memory'
    local_mem.mkdir(parents=True, exist_ok=True)
    return local_mem


def get_blogger_auth_file():
    return get_blogger_memory_dir() / 'blogger_auth.json'


def load_blogger_auth():
    auth_f = get_blogger_auth_file()
    if auth_f.is_file():
        try:
            data = json.loads(auth_f.read_text(encoding='utf-8'))
            if isinstance(data, dict):
                return data
        except Exception:
            pass
    cached_env = os.environ.get('ALTA_CACHED_BLOGGER_AUTH')
    if cached_env:
        try:
            return json.loads(cached_env)
        except Exception:
            pass
    return {}


def save_blogger_auth(data):
    os.environ['ALTA_CACHED_BLOGGER_AUTH'] = json.dumps(data, ensure_ascii=False)
    try:
        auth_f = get_blogger_auth_file()
        auth_f.parent.mkdir(parents=True, exist_ok=True)
        auth_f.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as e:
        print(f'⚠️ 儲存 Blogger 授權記憶至雲端時發生警告：{e}', flush=True)


def clear_blogger_auth():
    os.environ.pop('ALTA_CACHED_BLOGGER_AUTH', None)
    try:
        auth_f = get_blogger_auth_file()
        if auth_f.is_file():
            auth_f.unlink()
    except Exception:
        pass


def load_oauth_client_credentials():
    '''
    檢查使用者個人雲端永久記憶資料夾（CTB_詞庫設定_永久記憶）中是否存有 OAuth Desktop Client (client_secret*.json)，
    或 blogger_auth.json 中已記住的 client_id / client_secret。
    一旦有此設定，即可啟用「終身自動續約 Refresh Token（永遠不必重新登入）」！
    '''
    auth_data = load_blogger_auth()
    if auth_data.get('client_id') and auth_data.get('client_secret'):
        return auth_data['client_id'], auth_data['client_secret']
    mem_dir = get_blogger_memory_dir()
    for p in sorted(mem_dir.glob('*client_secret*.json')) + [mem_dir / 'blogger_oauth_client.json']:
        if p.is_file():
            try:
                raw = json.loads(p.read_text(encoding='utf-8'))
                block = raw.get('installed') or raw.get('web') or raw
                cid = (block.get('client_id') or '').strip()
                csec = (block.get('client_secret') or '').strip()
                if cid and csec:
                    auth_data['client_id'] = cid
                    auth_data['client_secret'] = csec
                    save_blogger_auth(auth_data)
                    return cid, csec
            except Exception:
                pass
    return '', ''


def refresh_blogger_access_token_if_needed(auth_data):
    '''
    自動無感續約 Access Token：
    - 情況 A：若有自備 client_id + client_secret + refresh_token，直接向 oauth2.googleapis.com/token 永久續約。
    - 情況 B：若僅貼入 OAuth Playground 的 Refresh Token (1// 開頭)，自動透過 Playground 官方端點換發最新 Access Token！
    '''
    refresh_tok = (auth_data.get('refresh_token') or '').strip()
    if not refresh_tok:
        return auth_data.get('access_token') or ''
    client_id = (auth_data.get('client_id') or '').strip()
    client_secret = (auth_data.get('client_secret') or '').strip()

    if client_id and client_secret:
        token_url = 'https://oauth2.googleapis.com/token'
        payload = urllib.parse.urlencode({
            'client_id': client_id,
            'client_secret': client_secret,
            'refresh_token': refresh_tok,
            'grant_type': 'refresh_token'
        }).encode('utf-8')
        req = urllib.request.Request(token_url, data=payload, method='POST')
        with urllib.request.urlopen(req, timeout=25) as resp:
            res = json.loads(resp.read().decode('utf-8'))
            new_acc = (res.get('access_token') or '').strip()
            if new_acc:
                auth_data['access_token'] = new_acc
                auth_data['updated_at'] = int(time.time())
                save_blogger_auth(auth_data)
                return new_acc
    else:
        # 使用 OAuth Playground 端點自動換發
        pg_refresh_url = 'https://developers.google.com/oauthplayground/refreshAccessToken'
        pg_body = json.dumps({
            'token_uri': 'https://oauth2.googleapis.com/token',
            'refresh_token': refresh_tok
        }).encode('utf-8')
        req = urllib.request.Request(
            pg_refresh_url,
            data=pg_body,
            headers={'Content-Type': 'application/json', 'User-Agent': 'Mozilla/5.0'},
            method='POST'
        )
        with urllib.request.urlopen(req, timeout=25) as resp:
            res = json.loads(resp.read().decode('utf-8'))
            new_acc = (res.get('access_token') or '').strip()
            if new_acc:
                auth_data['access_token'] = new_acc
                if res.get('refresh_token'):
                    auth_data['refresh_token'] = res['refresh_token'].strip()
                auth_data['updated_at'] = int(time.time())
                save_blogger_auth(auth_data)
                return new_acc
    return auth_data.get('access_token') or ''


def fetch_user_blogs_with_auth(auth_data):
    '''
    呼叫 Blogger API v3 (GET /users/self/blogs) 自動抓取當前登入者名下的所有 Blogger 網誌清單。
    若 access_token 過期（HTTP 401）且具備 refresh_token，會自動續約並重試！
    '''
    acc_token = (auth_data.get('access_token') or '').strip()
    if not acc_token and auth_data.get('refresh_token'):
        acc_token = refresh_blogger_access_token_if_needed(auth_data)

    if not acc_token:
        raise RuntimeError('尚未提供 Blogger 授權金鑰（Access Token）。')

    url = 'https://www.googleapis.com/blogger/v3/users/self/blogs'
    for attempt in range(2):
        try:
            req = urllib.request.Request(url, headers={'Authorization': f'Bearer {acc_token}'})
            with urllib.request.urlopen(req, timeout=25) as resp:
                body = json.loads(resp.read().decode('utf-8'))
                items = body.get('items') or []
                blogs = []
                for it in items:
                    if it.get('id'):
                        blogs.append({
                            'id': str(it['id']),
                            'name': it.get('name') or '未命名網誌',
                            'url': it.get('url') or ''
                        })
                auth_data['blogs'] = blogs
                if blogs:
                    saved_id = str(auth_data.get('selected_blog_id') or '')
                    if not any(b['id'] == saved_id for b in blogs):
                        auth_data['selected_blog_id'] = blogs[0]['id']
                        auth_data['selected_blog_name'] = blogs[0]['name']
                        auth_data['selected_blog_url'] = blogs[0]['url']
                save_blogger_auth(auth_data)
                return blogs, acc_token
        except urllib.error.HTTPError as he:
            if he.code == 401 and attempt == 0 and auth_data.get('refresh_token'):
                acc_token = refresh_blogger_access_token_if_needed(auth_data)
                continue
            err_detail = he.read().decode('utf-8', errors='replace')
            raise RuntimeError(f'HTTP {he.code}: {err_detail[:250]}') from he


def parse_user_blogger_token_input(raw_text, existing_auth=None):
    '''
    智慧解析使用者貼入的授權資訊：
    支援 1) 直接貼上 Access Token Access Token
    支援 2) 直接貼上 OAuth Playground 的 JSON 回應（含 access_token / refresh_token）
    支援 3) 貼上「access_token | refresh_token | client_id | client_secret」組合
    '''
    auth_obj = dict(existing_auth or {})
    s = (raw_text or '').strip()
    if not s:
        return auth_obj
    if s.startswith('{') and s.endswith('}'):
        try:
            j = json.loads(s)
            block = j.get('installed') or j.get('web') or j
            if block.get('access_token'):
                auth_obj['access_token'] = block['access_token'].strip()
            if block.get('refresh_token'):
                auth_obj['refresh_token'] = block['refresh_token'].strip()
            if block.get('client_id'):
                auth_obj['client_id'] = block['client_id'].strip()
            if block.get('client_secret'):
                auth_obj['client_secret'] = block['client_secret'].strip()
            return auth_obj
        except Exception:
            pass
    if '|' in s:
        parts = [p.strip() for p in s.split('|')]
        if len(parts) >= 1 and parts[0]:
            auth_obj['access_token'] = parts[0]
        if len(parts) >= 2 and parts[1]:
            auth_obj['refresh_token'] = parts[1]
        if len(parts) >= 4 and parts[2] and parts[3]:
            auth_obj['client_id'] = parts[2]
            auth_obj['client_secret'] = parts[3]
        return auth_obj
    if s.lower().startswith('bearer '):
        s = s[7:].strip()
    # 若使用者直接貼上 1// 開頭的 Refresh Token，自動存入 refresh_token 並立即換發 access_token！
    if s.startswith('1//'):
        auth_obj['refresh_token'] = s
        auth_obj['access_token'] = ''
        return auth_obj
    # 若使用者把整段含有 access_token 與 refresh_token 的文字一起貼進來，智慧正則提取
    m_acc = re.search(r'"access_token"\s*:\s*"([^"]+)"', s)
    m_ref = re.search(r'"refresh_token"\s*:\s*"([^"]+)"', s)
    if m_acc or m_ref:
        if m_acc:
            auth_obj['access_token'] = m_acc.group(1).strip()
        if m_ref:
            auth_obj['refresh_token'] = m_ref.group(1).strip()
        return auth_obj
    auth_obj['access_token'] = s
    return auth_obj


def push_or_update_post_to_blogger(j_dir, post_title, post_html, post_labels, is_draft=True, out_widget=None):
    '''
    將產出的升級版 Blog 文章推送至下拉選單所選的個人 Google Blogger 網誌！
    - 智慧防呆：若該集音檔已經推送過（`j_dir / blogger_post_meta.json` 存有 `post_id` 且為同一 `blog_id`），
      再次點擊會自動以 `PUT` 更新同一篇草稿／文章，絕不重複產生多篇垃圾草稿！
    '''
    def _log(msg, is_html=False):
        if out_widget is not None:
            with out_widget:
                if is_html:
                    display(HTML(msg))
                else:
                    print(msg, flush=True)
        else:
            if is_html:
                display(HTML(msg))
            else:
                print(msg, flush=True)

    auth_data = load_blogger_auth()
    if not auth_data.get('access_token') and not auth_data.get('refresh_token'):
        _log('⚠️ 尚未連結 Google Blogger 帳號！請先在上方「🔗 Google Blogger 一鍵連結與個人網誌選單」貼上授權金鑰完成連結（您亦可於連結後隨時點擊按鈕補推送，無須重跑 AI！）。')
        return False

    try:
        blogs, valid_token = fetch_user_blogs_with_auth(auth_data)
    except Exception as e:
        _log(f'⚠️ Blogger 授權金鑰已過期或無效，請於上方 Blogger 控制列重新貼上最新 Access Token（錯誤：{e}）')
        return False

    if not blogs:
        _log('⚠️ 您的 Google 帳號名下尚未建立任何 Blogger 網誌！請先至 blogger.com 建立網誌後再試。')
        return False

    selected_id = str(auth_data.get('selected_blog_id') or blogs[0]['id'])
    target_blog = next((b for b in blogs if b['id'] == selected_id), blogs[0])
    blog_id = target_blog['id']
    blog_name = target_blog['name']
    blog_url = target_blog['url']

    meta_file = j_dir / 'blogger_post_meta.json'
    existing_post_id = ''
    if meta_file.is_file():
        try:
            m = json.loads(meta_file.read_text(encoding='utf-8'))
            if str(m.get('blog_id')) == blog_id and m.get('post_id'):
                existing_post_id = str(m['post_id'])
        except Exception:
            pass

    post_payload = {
        'kind': 'blogger#post',
        'blog': {'id': blog_id},
        'title': post_title,
        'content': post_html,
        'labels': (post_labels or [])[:15]
    }

    try:
        headers = {
            'Authorization': f'Bearer {valid_token}',
            'Content-Type': 'application/json; charset=utf-8'
        }
        action_verb = '新建推送'
        if existing_post_id:
            # 更新同一篇已存在的文章或草稿
            update_url = f'https://www.googleapis.com/blogger/v3/blogs/{blog_id}/posts/{existing_post_id}'
            post_payload['id'] = existing_post_id
            req_up = urllib.request.Request(
                update_url,
                data=json.dumps(post_payload, ensure_ascii=False).encode('utf-8'),
                headers=headers,
                method='PUT'
            )
            try:
                with urllib.request.urlopen(req_up, timeout=30) as resp:
                    post_res = json.loads(resp.read().decode('utf-8'))
                    action_verb = '同步更新既有文章'
            except urllib.error.HTTPError as he_up:
                if he_up.code == 404:
                    existing_post_id = ''
                else:
                    raise

        if not existing_post_id:
            post_url = f'https://www.googleapis.com/blogger/v3/blogs/{blog_id}/posts?isDraft={"true" if is_draft else "false"}'
            req_post = urllib.request.Request(
                post_url,
                data=json.dumps(post_payload, ensure_ascii=False).encode('utf-8'),
                headers=headers,
                method='POST'
            )
            with urllib.request.urlopen(req_post, timeout=30) as resp:
                post_res = json.loads(resp.read().decode('utf-8'))

        post_id = str(post_res.get('id') or '')
        published_url = post_res.get('url') or blog_url
        edit_url = f'https://www.blogger.com/blog/post/edit/{blog_id}/{post_id}'
        status_str = '【草稿夾】' if is_draft else '【正式公開發布】'

        meta_file.write_text(json.dumps({
            'blog_id': blog_id,
            'blog_name': blog_name,
            'post_id': post_id,
            'post_title': post_title,
            'edit_url': edit_url,
            'published_url': published_url,
            'is_draft': is_draft,
            'updated_at': int(time.time())
        }, ensure_ascii=False, indent=2), encoding='utf-8')

        _log(
            '<div style="background:#e8f5e9; border-left:5px solid #2e7d32; padding:12px 16px; margin:12px 0; border-radius:8px;">'
            f'<b>🎉 已成功{action_verb}至您的 Google Blogger「{blog_name}」{status_str}！</b><br>'
            f'📌 <b>文章標題</b>：{post_title}<br>'
            f'🏷️ <b>分類標籤</b>：{", ".join((post_labels or [])[:10])}<br>'
            f'👉 <a href="{edit_url}" target="_blank" style="font-weight:bold; color:#00695c; text-decoration:underline;">點此直接開啟 Blogger 後台預覽／編輯這篇文章</a>'
            + (f' ｜ 🌐 <a href="{published_url}" target="_blank" style="color:#1565c0;">查看公開網址</a>' if not is_draft and published_url else '')
            + '</div>',
            is_html=True
        )
        return True
    except Exception as b_err:
        _log(f'⚠️ 推送至 Google Blogger 時發生錯誤（您仍可直接複製已生成的 HTML 檔案貼上 Blogger）：{b_err}')
        return False


def render_step4_banner(curr_jobs):
    '''步驟 4 頂部 Web App 級流程導航橫幅與當前素材狀態'''
    steps_info = [
        (1, '1. GPU 環境'),
        (2, '2. 來源與詞庫'),
        (3, '3. 語音轉錄'),
        (4, '4. AI 創作工作台'),
        (5, '5. 打包下載')
    ]
    pills = []
    for s_num, s_label in steps_info:
        if s_num == 4:
            bg, fg, bd, fw = '#ffd54f', '#004d40', '2px solid #ffffff', 'bold'
        elif s_num < 4:
            bg, fg, bd, fw = 'rgba(255,255,255,0.22)', '#e0f2f1', '1px solid rgba(255,255,255,0.35)', 'normal'
        else:
            bg, fg, bd, fw = 'rgba(0,0,0,0.18)', '#b2dfdb', '1px solid rgba(255,255,255,0.15)', 'normal'
        pills.append(
            f'<span style="display:inline-block; padding:4px 11px; margin:2px 4px; border-radius:999px; background:{bg}; color:{fg}; border:{bd}; font-weight:{fw}; font-size:12.5px;">{s_label}</span>'
        )
    pills_html = ' <span style="color:#80cbc4;">➔</span> '.join(pills)

    if curr_jobs:
        first_raw = json.loads((curr_jobs[0] / 'raw_segments.json').read_text(encoding='utf-8'))
        st_name = first_raw.get('stem') or '未命名'
        c_count = len(first_raw.get('cues') or [])
        is_direct = first_raw.get('source_mode') == 'direct_text'
        badge_txt = f'📝 現成文稿直通：{st_name}（共 {c_count} 句）' if is_direct else f'🎧 語音轉錄就緒：{st_name}（共 {len(curr_jobs)} 集 / {c_count} 句）'
    else:
        badge_txt = '⚠️ 尚未載入文稿（請直接於下方【分頁 ③】上傳或貼上 SRT/TXT！）'

    return (
        '<div style="background:linear-gradient(135deg, #004d40 0%, #00796b 100%); color:#ffffff; padding:14px 18px; border-radius:12px; margin-bottom:10px; box-shadow:0 3px 10px rgba(0,77,64,0.18);">'
        '<div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:8px;">'
        '<div style="font-size:17px; font-weight:bold; letter-spacing:0.5px;">🎛️ 雙引擎 AI 全媒體創作工作台 <span style="font-size:12px; background:#26a69a; padding:2px 8px; border-radius:6px; font-weight:normal; margin-left:6px;">Web App 互動儀表板</span></div>'
        f'<div style="font-size:12.5px; background:rgba(255,255,255,0.18); padding:3px 10px; border-radius:6px;">{badge_txt}</div>'
        '</div>'
        f'<div style="margin-top:4px;">{pills_html}</div>'
        '</div>'
    )


def render_five_lights_html(states_dict, stem_title=''):
    '''
    渲染動態「五燈獎任務進度儀表板」卡片
    states_dict[key] = ('wait'|'run'|'done'|'skip', 'detail text')
    '''
    items_meta = [
        ('A', '🔤 模組 A：AI 字幕精準校對'),
        ('B', '🔥 模組 B：9 組爆款標題 & 時間軸章節'),
        ('C', '✍️ 模組 C：升級版 Blog 專欄 & Blogger 草稿'),
        ('D', '📱 模組 D：FB / IG / LINE 三平台社群貼文'),
        ('E', '🎨 模組 E：四大視覺生圖提示詞大禮包'),
    ]
    cards = []
    for k, title in items_meta:
        st, detail = states_dict.get(k, ('wait', '等待中'))
        if st == 'done':
            bg, bd, fg, icon = '#e8f5e9', '#66bb6a', '#1b5e20', '🟢'
        elif st == 'run':
            bg, bd, fg, icon = '#e3f2fd', '#1e88e5', '#0d47a1', '🔄'
        elif st == 'skip':
            bg, bd, fg, icon = '#f5f5f5', '#e0e0e0', '#9e9e9e', '⚪'
        else:
            bg, bd, fg, icon = '#fff8e1', '#ffe082', '#795548', '⏳'
        cards.append(
            f'<div style="flex:1 1 175px; background:{bg}; border:1.5px solid {bd}; border-radius:8px; padding:8px 10px; color:{fg};">'
            f'<div style="font-weight:bold; font-size:13px;">{icon} {title}</div>'
            f'<div style="font-size:12px; margin-top:3px; opacity:0.92;">{detail}</div>'
            '</div>'
        )
    header_t = f'<div style="font-weight:bold; color:#004d40; margin-bottom:6px; font-size:13.5px;">📊 【{stem_title}】五燈獎即時創作進度儀表板（內建 6 秒安全間隔護欄）：</div>' if stem_title else ''
    return (
        '<div style="background:#fafafa; border:1px solid #cfd8dc; border-radius:10px; padding:10px 12px; margin:8px 0;">'
        + header_t +
        '<div style="display:flex; flex-wrap:wrap; gap:8px;">' + ''.join(cards) + '</div>'
        '</div>'
    )


def build_copy_and_download_bar(text_to_copy, file_path, btn_label='複製內容'):
    '''建立內建瀏覽器 Clipboard API 一鍵複製按鈕 ＋ Colab 單檔直接下載按鈕列'''
    b64_str = base64.b64encode((text_to_copy or '').encode('utf-8')).decode('ascii')
    js_click = (
        "const b=this.getAttribute('data-b64');"
        "const bytes=Uint8Array.from(atob(b),c=>c.charCodeAt(0));"
        "const txt=new TextDecoder('utf-8').decode(bytes);"
        "navigator.clipboard.writeText(txt).then(()=>{"
        "const orig=this.innerHTML;this.innerHTML='✅ 已成功複製到剪貼簿！';this.style.background='#2e7d32';"
        "setTimeout(()=>{this.innerHTML=orig;this.style.background='#00796b';},2200);});"
    )
    copy_html_widget = widgets.HTML(
        value=(
            f'<button data-b64="{b64_str}" onclick="{js_click}" '
            'style="background:#00796b; color:#ffffff; border:none; border-radius:6px; padding:6px 14px; font-size:13px; font-weight:bold; cursor:pointer; box-shadow:0 1px 3px rgba(0,0,0,0.15);">'
            f'📋 一鍵{btn_label}</button>'
        )
    )
    dl_btn = widgets.Button(
        description=f'⬇️ 直接下載 {Path(file_path).name}',
        button_style='info',
        layout=widgets.Layout(width='240px', height='32px', margin='0 0 0 8px')
    )

    def _on_dl(_):
        from google.colab import files
        if Path(file_path).is_file():
            files.download(str(file_path))

    dl_btn.on_click(_on_dl)
    return widgets.HBox([copy_html_widget, dl_btn], layout=widgets.Layout(align_items='center', margin='0 0 8px 0'))


def quick_ingest_text_in_step4(raw_text, filename_or_title):
    '''在步驟 4 分頁 ③ 直接上傳或貼上 SRT / VTT / TXT 文稿並立即建立工作佇列'''
    global job_dirs
    p_obj = Path(filename_or_title)
    ext = p_obj.suffix.lower() if p_obj.suffix.lower() in {'.srt', '.vtt', '.txt', '.md'} else '.txt'
    stem = (p_obj.stem if p_obj.suffix.lower() in {'.srt', '.vtt', '.txt', '.md'} else filename_or_title).strip() or '現成文稿創作'
    safe_stem = re.sub(r'[\/:*?"<>|]+', '_', stem)

    cues_tuples, converted_segs, rule_hits, has_real_ts, duration_sec = parse_existing_text_or_srt(
        raw_text, ext=ext, replace_rules=replace_rules
    )
    txt_lines = [t for _, _, t in cues_tuples]
    outputs = build_outputs_from_cues(cues_tuples, converted_segs, intro_music_seconds=0, txt_lines=txt_lines)

    job_id = 'job_001'
    j_dir = work / 'jobs' / job_id
    j_dir.mkdir(parents=True, exist_ok=True)
    for out_name, out_content in outputs.items():
        (j_dir / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (j_dir / out_name).write_text(out_content, encoding='utf-8')
        (work / ('raw_' + out_name)).write_text(out_content, encoding='utf-8')
        (work / out_name).write_text(out_content, encoding='utf-8')

    raw_payload = {
        'job_id': job_id,
        'filename': f'{safe_stem}{ext}',
        'stem': safe_stem,
        'language': 'zh',
        'duration': duration_sec,
        'source_mode': 'direct_text',
        'has_real_timestamps': bool(has_real_ts),
        'youtube_id': extract_yt_id(blog_youtube_url),
        'youtube_url': blog_youtube_url.strip(),
        'drive_save_dir': '',
        'deterministic_rule_hits': rule_hits,
        'subtitle_settings': {
            'auto_split': False,
            'auto_punctuation': False,
            'max_subtitle_chars': 22,
            'max_subtitle_seconds': 6.0,
            'intro_music_seconds': 0.0,
            'timing_methods': ['imported_timestamps' if has_real_ts else 'estimated_reading_speed']
        },
        'cues': [{'start_ms': s, 'end_ms': e, 'text': t} for s, e, t in cues_tuples],
        'segments': converted_segs
    }
    ep_drive_dir = ensure_episode_drive_dir(raw_payload)
    raw_payload['drive_save_dir'] = ep_drive_dir

    raw_json_str = json.dumps(raw_payload, ensure_ascii=False, indent=2)
    (j_dir / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (j_dir / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'raw_segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'segments.json').write_text(raw_json_str, encoding='utf-8')
    (work / 'output_stem.txt').write_text(safe_stem, encoding='utf-8')

    job_dirs = [j_dir]
    return raw_payload


any_ai_enabled = any([
    enable_ai_proofread,
    generate_titles_and_chapters,
    generate_blog_post,
    generate_social_posts,
    generate_visual_prompts
])


def detect_initial_api_key():
    '''從 Colab Secrets 或當前執行階段記憶體讀取已快取的 API Key'''
    cached = (os.environ.get('ALTA_CACHED_GEMINI_KEY') or os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY') or '').strip()
    if cached:
        return cached, '執行階段記憶體快取'
    try:
        from google.colab import userdata
        for key_name in ('GEMINI_API_KEY', 'GOOGLE_API_KEY'):
            try:
                val = userdata.get(key_name)
                if val and val.strip():
                    os.environ['ALTA_CACHED_GEMINI_KEY'] = val.strip()
                    return val.strip(), f'Colab Secrets ({key_name})'
            except Exception:
                pass
    except Exception:
        pass
    return '', ''


def extract_9_title_choices(raw_titles_text, fallback_stem=''):
    '''從模組 B1 產出的 9 組爆款標題文字中解析出供單選按鈕使用的標題清單（顯示保留設計概念供參考，實際定案值自動剝離設計概念）'''
    choices = []
    seen = set()
    cur_scheme = ''
    for ln in (raw_titles_text or '').splitlines():
        s = strip_md_to_plain(ln).strip()
        if not s:
            continue
        if '方案 A' in s or '方案A' in s:
            cur_scheme = '方案A(認知衝擊)'
        elif '方案 B' in s or '方案B' in s:
            cur_scheme = '方案B(實用需求)'
        elif '方案 C' in s or '方案C' in s:
            cur_scheme = '方案C(情感共鳴)'
        if any(k in s for k in ('推薦首選標題', '正選', '備選一', '備選二', '備選1', '備選2')) and ('：' in s or ':' in s):
            parts = re.split(r'[：:]', s, maxsplit=1)
            if len(parts) == 2:
                lbl_part = parts[0].strip()
                raw_val = parts[1].strip()
                clean_val = re.sub(r'[（(]\s*設計概念[：:].*$', '', raw_val).strip()
                if len(clean_val) >= 4 and clean_val not in seen:
                    seen.add(clean_val)
                    prefix = f'[{cur_scheme}] ' if (cur_scheme and '推薦首選' not in lbl_part) else ''
                    choices.append((f'{prefix}{lbl_part}：{raw_val}', clean_val))
    if not choices and fallback_stem:
        choices.append((f'預設標題：{fallback_stem}', fallback_stem))
    return choices[:10]


def run_all_in_one_pipeline(active_api_key, run_output_area, two_stage_pause=False):
    '''在指定的輸出區塊內執行全套雙引擎 AI 創作流水線'''
    with run_output_area:
        clear_output(wait=True)
        safe_interval = max(4, int(api_safe_interval_seconds))
        _last_call_ts = [0.0]
        _resolved_models_cache = {}

        def resolve_model_candidates(requested_model):
            '''
            完整的自動降階梯隊（Auto-Downgrade Ladder）：
            若使用者的 API Key 無法使用 Gemini 4 Argon 或 Pro 等高階模型（例如免費版 Key 或無權限），
            將自動依序無縫降階至 3.5-flash-lite -> 3.1-flash-lite -> 2.5-flash -> 2.5-flash-lite -> 2.0-flash-lite！
            '''
            if 'flash-lite' in requested_model:
                ladder = [
                    requested_model,
                    'gemini-3.5-flash-lite',
                    'gemini-3.1-flash-lite',
                    'gemini-2.5-flash-lite',
                    'gemini-2.0-flash-lite',
                    'gemini-flash-lite-latest',
                    'gemini-2.5-flash',
                    'gemini-2.0-flash'
                ]
            else:
                ladder = [
                    requested_model,
                    'gemini-4-argon',
                    'gemini-2.5-pro',
                    'gemini-3.5-flash-lite',
                    'gemini-3.1-flash-lite',
                    'gemini-2.5-flash',
                    'gemini-2.5-flash-lite',
                    'gemini-2.0-flash-lite',
                    'gemini-flash-lite-latest',
                    'gemini-2.0-flash',
                    'gemini-1.5-flash'
                ]
            seen = []
            for m in ladder:
                if m and m not in seen:
                    seen.append(m)
            return seen

        def call_gemini_safe(prompt, system_instruction, requested_model, max_tokens=16384):
            '''
            具備嚴格頻率護欄、自動降階與熔斷保護的 Gemini API 呼叫器：
            - 若高階模型回傳 404 (無此模型)、403 (無權限)、400 (不支援) 或 429 (該高階模型配額為 0 或已滿)，
              自動無縫降階至下一個可用的輕量模型並快取，絕不中斷！
            '''
            candidates = (
                [_resolved_models_cache[requested_model]] + [m for m in resolve_model_candidates(requested_model) if m != _resolved_models_cache[requested_model]]
                if requested_model in _resolved_models_cache
                else resolve_model_candidates(requested_model)
            )

            for m_idx, model_name in enumerate(candidates):
                curr_max_tokens = max_tokens
                payload = {
                    'systemInstruction': {'parts': [{'text': system_instruction}]},
                    'contents': [{'parts': [{'text': prompt}]}],
                    'generationConfig': {
                        'temperature': 0.25,
                        'maxOutputTokens': curr_max_tokens
                    }
                }
                body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
                url = f'https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={active_api_key}'

                for attempt in range(3):
                    elapsed = time.time() - _last_call_ts[0]
                    if _last_call_ts[0] > 0 and elapsed < safe_interval:
                        time.sleep(safe_interval - elapsed)

                    _last_call_ts[0] = time.time()
                    req = urllib.request.Request(url, data=body, headers={'Content-Type': 'application/json'}, method='POST')
                    try:
                        with urllib.request.urlopen(req, timeout=150) as resp:
                            resp_data = json.loads(resp.read().decode('utf-8'))
                            cands = resp_data.get('candidates') or []
                            if not cands:
                                raise RuntimeError('Gemini API 未回傳候選內容。')
                            parts = (cands[0].get('content') or {}).get('parts') or []
                            text_out = ''.join(p.get('text', '') for p in parts).strip()
                            if _resolved_models_cache.get(requested_model) != model_name:
                                if model_name != requested_model:
                                    print(f'🔄 已自動降階並綁定可用模型：[{requested_model}] ➔ [{model_name}]', flush=True)
                                _resolved_models_cache[requested_model] = model_name
                            return text_out, model_name
                    except urllib.error.HTTPError as err:
                        err_body = ''
                        try:
                            err_body = err.read().decode('utf-8', errors='replace')
                        except Exception:
                            pass

                        # 若為 API Key 本身字串無效，立即提示使用者
                        if 'API_KEY_INVALID' in err_body or 'API key not valid' in err_body:
                            raise RuntimeError('❌ 您輸入的 Gemini API Key 無效，請點擊上方「👁️ 顯示金鑰」檢查是否有拼寫錯誤或多餘空白！') from err

                        # 若為 maxOutputTokens 超過舊模型上限，自動調降為 8192 重試
                        if err.code == 400 and 'maxOutputTokens' in err_body and curr_max_tokens > 8192:
                            curr_max_tokens = 8192
                            payload['generationConfig']['maxOutputTokens'] = 8192
                            body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
                            continue

                        # 若該 API Key 無此高階模型權限 (403/404/400) 或免費層對該模型配額為 0 (limit: 0)
                        has_next_model = (m_idx < len(candidates) - 1)
                        is_zero_quota = (err.code == 429 and ('limit: 0' in err_body or 'Quota exceeded' in err_body))
                        if err.code in (400, 403, 404) or (is_zero_quota and has_next_model):
                            if has_next_model:
                                next_m = candidates[m_idx + 1]
                                print(f'🔄 偵測到目前 API Key 無法使用模型 [{model_name}]（HTTP {err.code}），自動降階至 [{next_m}] 繼續執行...', flush=True)
                                break
                            else:
                                raise RuntimeError(f'Gemini API 請求失敗（[{model_name}] HTTP {err.code}）：{err_body[:250]}') from err

                        elif err.code in (429, 500, 502, 503, 504):
                            # 若還有下一個更輕量的備援模型且當前為高階模型，直接降階嘗試
                            if has_next_model and attempt >= 1:
                                next_m = candidates[m_idx + 1]
                                print(f'🔄 模型 [{model_name}] 忙碌或達上限（HTTP {err.code}），自動降階至 [{next_m}]...', flush=True)
                                break
                            if attempt < 2:
                                cooldown = 25 * (attempt + 1)
                                print(f'🛡️ 觸發 API 安全護欄（[{model_name}] HTTP {err.code}）：暫停冷卻 {cooldown} 秒後進行第 {attempt + 1}/2 次溫和重試...', flush=True)
                                time.sleep(cooldown)
                                continue
                            else:
                                raise RuntimeError(
                                    f'🛡️ 已啟動 API 熔斷保護（HTTP {err.code}）：目前伺服器繁忙或達配額上限，已停止發送請求以保護您的帳號。'
                                ) from err
                        else:
                            raise RuntimeError(f'Gemini API 請求失敗（[{model_name}] HTTP {err.code}）：{err_body[:250]}') from err
                    except Exception as err:
                        if attempt < 2:
                            time.sleep(12)
                            continue
                        raise RuntimeError(f'網路或 API 連線異常：{err}') from err

            raise RuntimeError(f'找不到可用的模型（已嘗試降階清單：{candidates}）。')

        def format_mmss(ms):
            total_sec = max(0, int(ms // 1000))
            hh, rem = divmod(total_sec, 3600)
            mm, ss = divmod(rem, 60)
            if hh > 0:
                return f'{hh:02d}:{mm:02d}:{ss:02d}'
            return f'{mm:02d}:{ss:02d}'

        pos_str = '、'.join(positive_terms) if positive_terms else 'ㄚ亮笑長'
        neg_str = ('；請避免出現以下常見誤聽詞並予以修正：' + '、'.join(negative_terms)) if negative_terms else ''
        rep_str = '、'.join(f"{r['original']}->{r['replacement']}" for r in replace_rules) if replace_rules else ''

        print(f'🛡️ 已啟動雙引擎 AI 工作台（校對首選：{proofread_model} ｜ 創作首選：{creative_model}［支援無權限自動降階］｜ 安全間隔鎖：{safe_interval} 秒）', flush=True)

        total_jobs = len(job_dirs)
        for j_idx, j_dir in enumerate(job_dirs, 1):
            raw_data = json.loads((j_dir / 'raw_segments.json').read_text(encoding='utf-8'))
            stem = raw_data.get('stem') or (work / 'output_stem.txt').read_text(encoding='utf-8').strip() or '轉錄結果'
            raw_cues = raw_data.get('cues') or []
            if not raw_cues:
                continue

            job_yt_id = extract_yt_id(blog_youtube_url) or raw_data.get('youtube_id') or ''
            total_cues = len(raw_cues)
            batch_size = 400
            num_batches = ((total_cues + batch_size - 1) // batch_size) if enable_ai_proofread else 0
            manuscript_chunk_size = 180
            num_manuscript_chunks = ((total_cues + manuscript_chunk_size - 1) // manuscript_chunk_size) if enable_ai_proofread else 0
            total_steps = (
                num_batches +
                num_manuscript_chunks +
                (2 if generate_titles_and_chapters else 0) +
                (1 if generate_blog_post else 0) +
                (1 if generate_social_posts else 0) +
                (1 if generate_visual_prompts else 0)
            )

            print(chr(10) + '========================================', flush=True)
            print(f'▶ [{j_idx}/{total_jobs}] 開始執行全媒體創作流水線：{stem}（預計僅需 {total_steps} 次 API 請求）', flush=True)
            has_real_ts = raw_data.get('has_real_timestamps', True)
            lights_state = {
                'A': ('wait', f'待校對 {total_cues} 句') if enable_ai_proofread else ('skip', '未啟用（保留原稿）'),
                'B': ('wait', '等待執行') if generate_titles_and_chapters else ('skip', '未啟用'),
                'C': ('wait', '等待執行') if generate_blog_post else ('skip', '未啟用'),
                'D': ('wait', '等待執行') if generate_social_posts else ('skip', '未啟用'),
                'E': ('wait', '等待執行') if generate_visual_prompts else ('skip', '未啟用'),
            }
            five_lights_box = widgets.HTML(value=render_five_lights_html(lights_state, stem))
            ai_progress = widgets.IntProgress(value=0, min=0, max=max(1, total_steps), description=f'總體進度 ({j_idx}/{total_jobs})', bar_style='info')
            ai_label = widgets.HTML(value='0%（準備中...）')
            display(five_lights_box, widgets.HBox([ai_progress, ai_label]))

            # 模組 A：AI 字幕精準校對
            proofread_texts = {}
            used_proof_model = proofread_model

            if enable_ai_proofread:
                filler_rule = (
                    '請自然過濾無意義的口語贅字、發語詞或口吃重複（如「呃、嗯、啊、那個那個、就是說」），保持語氣自然且不改變原意；若該句只有語助詞，請保留簡短字詞以免出現空行。'
                    if remove_filler_words else
                    '請保留講者原本的口語語助詞，僅修正錯別字與同音字。'
                )
                punct_rule = (
                    '請在句子內部適當補上自然的繁體中文標點符號（如逗號、頓號、問號），單行短字幕結尾可不加句號。'
                    if smart_punctuation else
                    '維持原有的標點與斷句格式。'
                )
                sys_prompt_cues = (
                    '你是一位頂尖的臺灣繁體中文影視字幕校對師。使用者會提供格式為「編號|字幕內容」的逐行字幕。' + chr(10) +
                    '【嚴格遵守規則】' + chr(10) +
                    '1. 完美保留時間碼對齊：請同樣以「編號|修正後字幕內容」逐行回傳，每行的「編號」必須與輸入一模一樣！絕對不可合併或拆分不同編號的句子，也不可將下一句的字詞搬移到上一句。不要輸出任何額外說明。' + chr(10) +
                    '2. 修正同音異字：根據上下文語境，修正語音辨識常見的同音錯字與臺灣繁體慣用語（如：影片、軟體、透過、品質、資訊）。' + chr(10) +
                    f'3. 語助詞處理：{filler_rule}' + chr(10) +
                    f'4. 智慧標點：{punct_rule}' + chr(10) +
                    f'5. 專有名詞詞庫：遇到發音相近詞請優先修正為【{pos_str}】{neg_str}。' +
                    (f'強制替換規則：【{rep_str}】。' if rep_str else '')
                )

                lights_state['A'] = ('run', f'校對中 (0/{num_batches} 批)...')
                five_lights_box.value = render_five_lights_html(lights_state, stem)
                for b_idx in range(num_batches):
                    start_i = b_idx * batch_size
                    end_i = min(total_cues, start_i + batch_size)
                    batch_lines = [f"{i}|{raw_cues[i]['text']}" for i in range(start_i, end_i)]
                    prompt_block = chr(10).join(batch_lines)

                    resp_text, used_proof_model = call_gemini_safe(
                        prompt=f'請校對以下第 {start_i} 至 {end_i - 1} 號字幕，嚴格以「編號|修正後文字」格式逐行回傳：' + chr(10) + prompt_block,
                        system_instruction=sys_prompt_cues,
                        requested_model=proofread_model
                    )

                    for raw_line in resp_text.splitlines():
                        line = raw_line.strip()
                        if not line or '|' not in line:
                            continue
                        line = line.lstrip('-*` ')
                        id_part, text_part = line.split('|', 1)
                        id_part = id_part.strip()
                        if id_part.isdigit():
                            idx = int(id_part)
                            cleaned, _ = clean_text_deterministic(text_part.strip(), replace_rules=replace_rules)
                            if cleaned and start_i <= idx < end_i:
                                proofread_texts[idx] = cleaned

                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（已用 {used_proof_model} 校對 {end_i}/{total_cues} 句字幕）'
                    lights_state['A'] = ('run', f'已校對 {end_i}/{total_cues} 句 ({used_proof_model})')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

            final_cues = []
            changed_examples = []
            for i, c in enumerate(raw_cues):
                orig_t = c['text']
                new_t = proofread_texts.get(i, orig_t)
                final_cues.append((c['start_ms'], c['end_ms'], new_t))
                if new_t != orig_t:
                    changed_examples.append((i + 1, orig_t, new_t))

            intro_seconds = float((raw_data.get('subtitle_settings') or {}).get('intro_music_seconds', 0))
            converted_segments = raw_data.get('segments') or []
            proofread_txt_lines = [t for _, _, t in final_cues]

            new_outputs = build_outputs_from_cues(
                final_cues,
                converted_segments,
                intro_music_seconds=intro_seconds,
                txt_lines=proofread_txt_lines
            )

            for name, content in new_outputs.items():
                (j_dir / name).write_text(content, encoding='utf-8')
                (work / name).write_text(content, encoding='utf-8')

            ep_drive_dir = ensure_episode_drive_dir(raw_data)
            updated_payload = {
                **raw_data,
                'drive_save_dir': ep_drive_dir,
                'youtube_id': job_yt_id,
                'ai_proofread': bool(enable_ai_proofread),
                'ai_model': used_proof_model,
                'ai_changed_count': len(changed_examples),
                'cues': [
                    {'start_ms': s, 'end_ms': e, 'text': t, 'original_text': raw_cues[i]['text']}
                    for i, (s, e, t) in enumerate(final_cues)
                ]
            }
            seg_json_str = json.dumps(updated_payload, ensure_ascii=False, indent=2)
            (j_dir / 'segments.json').write_text(seg_json_str, encoding='utf-8')
            (work / 'segments.json').write_text(seg_json_str, encoding='utf-8')
            (work / 'output_stem.txt').write_text(stem, encoding='utf-8')

            readable_full_txt = chr(10).join(proofread_txt_lines)
            if enable_ai_proofread:
                print('📄 [模組 A+] 正在將逐句字幕整理為「高可讀性全本順稿文稿 (.txt)」（100% 保留原敘述順序與細節、不省略內容）...', flush=True)
                lights_state['A'] = ('run', f'正在整理高可讀性全本順稿 ({num_manuscript_chunks} 段)...')
                five_lights_box.value = render_five_lights_html(lights_state, stem)
                sys_prompt_manuscript = (
                    '你是一位專業的臺灣繁體中文文字編輯。請將以下語音轉錄的短句逐字稿，改寫整理為一份「極具可讀性、標點完整、自然分段的完整順稿文稿（.txt）」。' + chr(10) +
                    '【嚴格遵守規則】' + chr(10) +
                    '1. **100% 保留原本音檔的敘述順序與實質細節**：嚴禁省略講者的案例、故事、步驟或對話脈絡！不可寫成簡短摘要，必須是一份「從頭到尾完整保留原意與細節的順暢全本讀本」。' + chr(10) +
                    '2. **消除碎句換行與口語贅字**：將原本被切成一行一行的零碎短句，串聯成完整流暢的自然段落（每段約 150~280 字，段落之間空一行），並補上精準的繁體中文標點符號（逗號、句號、問號、引號）。' + chr(10) +
                    '3. **加上清晰的段落主題小標**：當話題轉換時，請以純文字方括號加上段落小標（例如：`【一、開場與核心問題】`、`【二、教學現場的真實案例】`），嚴禁使用 `**` 或 `###` 等 Markdown 符號，確保 `.txt` 純文字檔打開時清爽好讀！' + chr(10) +
                    f'4. 專有名詞請統一使用：【{pos_str}】。'
                )
                manuscript_parts = []
                for m_idx in range(num_manuscript_chunks):
                    m_start = m_idx * manuscript_chunk_size
                    m_end = min(len(final_cues), m_start + manuscript_chunk_size)
                    chunk_text = chr(10).join(
                        f'[{format_mmss(final_cues[k][0])}] {final_cues[k][2]}' if (k == m_start or (k - m_start) % 25 == 0) else final_cues[k][2]
                        for k in range(m_start, m_end)
                    )
                    part_label = f'（第 {m_idx + 1}/{num_manuscript_chunks} 部分，請直接輸出順稿正文，不要加開場白）'
                    m_resp, used_proof_model = call_gemini_safe(
                        prompt=f'請將以下逐字稿段落整理為高可讀性完整順稿文稿{part_label}：' + chr(10) + chunk_text,
                        system_instruction=sys_prompt_manuscript,
                        requested_model=proofread_model
                    )
                    manuscript_parts.append(strip_md_to_plain(m_resp))
                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（已整理高可讀性全本順稿 {m_idx + 1}/{num_manuscript_chunks} 段）'

                joined_manuscript = (chr(10) + chr(10)).join(p for p in manuscript_parts if p).strip()
                if joined_manuscript:
                    header_txt = f'《{stem}》完整順稿文稿' + chr(10) + ('=' * 40) + chr(10) + chr(10)
                    readable_full_txt = header_txt + joined_manuscript + chr(10)
                    (j_dir / 'transcript.txt').write_text(readable_full_txt, encoding='utf-8')
                    (work / 'transcript.txt').write_text(readable_full_txt, encoding='utf-8')

                lights_state['A'] = ('done', f'完成！SRT修正 {len(changed_examples)} 句＋全本順稿 ({used_proof_model})')
                five_lights_box.value = render_five_lights_html(lights_state, stem)
                print(f'✅ [模組 A] 字幕校對與高可讀性全本順稿 (.txt) 完成！共檢查 {total_cues} 句，修正了 {len(changed_examples)} 句。', flush=True)
                if changed_examples:
                    for line_no, old_str, new_str in changed_examples[:5]:
                        print(f'   [第 {line_no:03d} 句] {old_str}  ➔  {new_str}', flush=True)

            timed_transcript_lines = []
            last_marked_sec = -20
            for s_ms, e_ms, t_str in final_cues:
                curr_sec = s_ms // 1000
                if curr_sec - last_marked_sec >= 15 or last_marked_sec < 0:
                    timed_transcript_lines.append(f'[{format_mmss(s_ms)}] {t_str}')
                    last_marked_sec = curr_sec
                else:
                    timed_transcript_lines.append(t_str)
            timed_transcript_text = chr(10).join(timed_transcript_lines)
            clean_full_text = chr(10).join(proofread_txt_lines)

            # 模組 B：兩段式核心（先產出 9 組爆款標題 ➔ 讓使用者挑選/微調定案標題 ➔ 再產出 YT純文字簡介與時間軸 + Blog + 社群 + 視覺提示詞！）
            titles_and_chapters_md = ''
            best_title = stem
            chapters_quick_list = []
            yt_desc_plain = ''
            yt_chapters_plain = ''
            titles_only_raw = ''

            def generate_9_titles_only():
                sys_prompt_titles = (
                    '你是一位頂尖的 YouTube / Podcast 內容策略總監。請根據提供的節目逐字稿，發想 **3 大方案 × 各 3 組（共 9 組）** 爆款主副標題建議（使用道地臺灣繁體中文，**嚴禁使用 `**`、`###` 等 Markdown 符號**）：' + chr(10) +
                    '請嚴格依照以下格式輸出：' + chr(10) +
                    '🏆 推薦首選標題：主標題！：副標題' + chr(10) + chr(10) +
                    '【方案 A：認知衝擊】' + chr(10) +
                    '正選：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選一：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選二：主標題：副標題（設計概念：...）' + chr(10) + chr(10) +
                    '【方案 B：實用需求】' + chr(10) +
                    '正選：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選一：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選二：主標題：副標題（設計概念：...）' + chr(10) + chr(10) +
                    '【方案 C：情感共鳴】' + chr(10) +
                    '正選：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選一：主標題：副標題（設計概念：...）' + chr(10) +
                    '備選二：主標題：副標題（設計概念：...）' + chr(10) +
                    f'專有名詞請統一使用：【{pos_str}】。'
                )
                t_raw, u_model = call_gemini_safe(
                    prompt='請根據以下內容發想 3 大方案共 9 組爆款主副標題：' + chr(10) + clean_full_text[:35000],
                    system_instruction=sys_prompt_titles,
                    requested_model=creative_model
                )
                return strip_md_to_plain(t_raw), u_model

            def run_stage2_with_anchored_title(anchored_title, pre_titles_raw):
                nonlocal best_title, titles_and_chapters_md, chapters_quick_list, yt_desc_plain, yt_chapters_plain, used_creative_model
                best_title = (anchored_title or stem).strip()
                chapters_quick_list = []

                if generate_titles_and_chapters:
                    lights_state['B'] = ('run', f'正以「{best_title[:16]}...」撰寫 YT 簡介與時間軸...')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)
                    print(f'🎬 [模組 B2] 鎖定定案標題「{best_title}」，正在產出 YouTube 簡介欄純文字與 00:00 時間軸章節...', flush=True)
                    sys_prompt_yt_desc = (
                        '你是一位頂尖的 YouTube / Podcast 內容策略總監。請根據提供帶有 `[MM:SS]` 時間標記的節目逐字稿，以及使用者**已經拍板定案的核心標題**，產出**可直接一鍵貼入 YouTube 影片資訊欄／Podcast 簡介欄的純文字完整文案**（使用道地臺灣繁體中文，**全程嚴禁出現 `**`、`###`、`*` 等任何 Markdown 語法符號**）：' + chr(10) + chr(10) +
                        f'【本集定案核心標題】：{best_title}' + chr(10) +
                        '請嚴格依照以下 4 個純文字區塊輸出：' + chr(10) +
                        '【📌 本集精華簡介】' + chr(10) +
                        '（150~250 字引人入勝的節目導讀，開頭第一句要緊扣定案核心標題的切角，自然分段）' + chr(10) + chr(10) +
                        '【✨ 本集三大核心重點】' + chr(10) +
                        '▸ 重點一：...' + chr(10) +
                        '▸ 重點二：...' + chr(10) +
                        '▸ 重點三：...' + chr(10) + chr(10) +
                        '【⏱️ 時間軸章節快速跳轉】' + chr(10) +
                        '00:00 開場與本集主題介紹' + chr(10) +
                        'MM:SS 章節標題（依據真實 `[MM:SS]` 時間點由小到大排列，共 5~10 個章節）' + chr(10) + chr(10) +
                        '【🗂️ 各章節重點導覽】' + chr(10) +
                        '▸ 00:00 開場與本集主題介紹：1~2 句重點說明' + chr(10) +
                        f'專有名詞請統一使用：【{pos_str}】。'
                    )
                    yt_raw_resp, used_creative_model = call_gemini_safe(
                        prompt=f'請圍繞定案標題「{best_title}」與以下逐字稿產出 YouTube 資訊欄純文字簡介與時間軸：' + chr(10) + timed_transcript_text[:45000],
                        system_instruction=sys_prompt_yt_desc,
                        requested_model=creative_model
                    )
                    yt_desc_plain = strip_md_to_plain(yt_raw_resp)
                    for line in yt_desc_plain.splitlines():
                        line_s = line.strip()
                        if re.match(r'^\d{1,2}:\d{2}(?::\d{2})?\s+[^▸]+$', line_s):
                            chapters_quick_list.append(line_s)
                    yt_chapters_plain = chr(10).join(chapters_quick_list)
                    titles_and_chapters_md = f'[TITLES_START]{chr(10)}{pre_titles_raw}{chr(10)}[TITLES_END]{chr(10)}{chr(10)}[YT_DESC_START]{chr(10)}{yt_desc_plain}{chr(10)}[YT_DESC_END]'
                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（[模組 B2] 已鎖定標題完成 YT 純文字簡介與時間軸！）'
                    lights_state['B'] = ('done', f'完成！已鎖定標題+YT簡介 ({used_creative_model})')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

                # 模組 C：升級版 Blog 專欄文章生成 ＋ 一鍵推送 Google Blogger
                blog_html_final = ''
                blog_md_final = ''

                if generate_blog_post:
                    lights_state['C'] = ('run', '正在撰寫升級版 Blog 專欄...')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)
                    print('✍️ [模組 C] 正在撰寫「升級版 Blog 專欄文章（含時間軸導讀卡、金句框、微清單與系列專屬 CTA）」...', flush=True)
                    cta_html, base_tags, series_prefix = get_series_cta_and_tags(blog_series_template, stem)

                    pov_rule = (
                        '請化身為本集主講者／來賓的「第一人稱（我）」視角娓娓道來（例如：「大家好，我是...」），保留講者的溫暖口吻、開場幽默自嘲與教學／生命現場的真實故事感。'
                        if '第一人稱' in blog_pov else
                        '請以專業專欄主編的「第三人稱」視角撰寫，生動側寫講者的核心觀點與實戰分享。'
                    )
                    if '疑問句' in blog_h2_style:
                        h2_rule = '每個 `<h2>` 段落標題請採用「引發讀者好奇與反思的疑問句」形式（例如：`<h2>為什麼發再多考卷，孩子還是學不會？</h2>`）。'
                    elif 'Emoji' in blog_h2_style:
                        h2_rule = '每個 `<h2>` 段落標題開頭請加上一個貼切的 Emoji，語氣活潑有感染力。'
                    else:
                        h2_rule = '每個 `<h2>` 段落標題請採用「主題背景：核心亮點」的雙段式簡潔有力標題。'

                    table_rule = (
                        '在文章最後一個段落結束後，請生成一個排版清晰的 HTML 重點回顧表格（`<table style="width:100%; border-collapse:collapse; margin:20px 0; font-size:15px;">`，表頭使用 `<th style="background:#00897b; color:#ffffff; padding:10px; border:1px solid #cfd8dc; text-align:left;">`，儲存格使用 `<td style="padding:10px; border:1px solid #cfd8dc; color:#263238;">`），彙整本集 3~5 個關鍵概念或實戰步驟。'
                        if blog_include_summary_table else
                        '不需生成 HTML 表格。'
                    )

                    sys_prompt_blog = (
                        '你是一位頂尖的臺灣繁體中文部落格專欄主編（專為「丫亮笑長練功坊 aliang6724.blogspot.com」撰稿）。' + chr(10) +
                        '請將提供的節目內容轉化為一篇兼具「人味溫度」與「極致視覺易讀性」的升級版 Blog 專欄文章。' + chr(10) + chr(10) +
                        '【嚴格輸出結構要求】' + chr(10) +
                        '請依序輸出兩個區塊，使用 `[BLOG_HTML_START]` 與 `[BLOG_HTML_END]` 包裝 HTML 正文，並使用 `[SEO_JSON_START]` 與 `[SEO_JSON_END]` 包裝 SEO JSON 資料：' + chr(10) + chr(10) +
                        '1. 在 `[BLOG_HTML_START] ... [BLOG_HTML_END]` 中（純 HTML，不要放入 ```html 標籤，不要包含 `<h1>` 或 YouTube `<iframe>` 或文末追蹤 CTA，這些會由系統自動組裝）：' + chr(10) +
                        f'   - **敘事視角與字數**：{pov_rule} 全文長度目標為【{blog_length}】，必須完整保留講者的實質案例、步驟與深度觀點。' + chr(10) +
                        '   - **開頭導讀卡片**：文章最開頭請先放一張「📌 本集精華導讀」HTML 卡片：' + chr(10) +
                        '     `<div style="background:#f4fbfa; border:1px solid #b2dfdb; border-left:5px solid #00897b; border-radius:8px; padding:16px 20px; margin:16px 0 24px 0; color:#263238; line-height:1.75;">`' + chr(10) +
                        '     內含 `<b>📌 本集 30 秒精華導讀</b>`、3 點核心收穫條列（`<ul><li>...</li></ul>`）' +
                        ('，以及 `<b>⏱️ 影片時間軸快速跳轉</b>`（放入真實 `00:00` 時間軸章節清單）' if (chapters_quick_list and has_real_ts) else '') + '，最後關閉 `</div>`。' + chr(10) +
                        f'   - **段落標題 (`<h2>`) 與分隔線 (`<hr>`)**：全篇規劃 4~6 個 `<h2>` 主題段落，{h2_rule} 每個 `<h2>` 區塊之間請加上 `<hr>`。' + chr(10) +
                        '   - **高易讀短段落 (`<p>`)**：為解決 Blogger 預設淺灰字視覺疲勞，每個 `<p style="color:#263238; font-size:16px; line-height:1.85; margin:14px 0;">` 嚴格控制在 **120~160 字以內**（長段落請自然拆分為 2~3 個 `<p>`），並將核心關鍵字以 `<strong>` 加粗。' + chr(10) +
                        '   - **💬 核心金句引言框 (`<blockquote>`)**：請在文中挑選 2 處最打動人心的金句，使用以下樣式呈現：' + chr(10) +
                        '     `<blockquote style="background:#fff8e1; border-left:5px solid #ffb300; padding:14px 18px; margin:18px 0; border-radius:6px; color:#4e342e; font-size:16px; font-weight:500;">💬 「金句內容...」</blockquote>`' + chr(10) +
                        '   - **實戰步驟微清單 (`<ul><li>`)**：當講到具體階段、步驟或多項技巧時，請在故事敘事後搭配清爽的 `<ul style="color:#263238; font-size:16px; line-height:1.8;"><li><strong>階段名稱</strong>：說明...</li></ul>`。' + chr(10) +
                        f'   - **文末重點回顧表格**：{table_rule}' + chr(10) + chr(10) +
                        '2. 在 `[SEO_JSON_START] ... [SEO_JSON_END]` 中，輸出合法的 JSON 物件：' + chr(10) +
                        '   `{"post_title": "吸睛部落格文章標題", "seo_title": "SEO搜尋標題(<=60字)", "meta_description": "搜尋描述(120~150字)", "slug": "english-url-slug", "labels": ["標籤1", "標籤2", "標籤3", "標籤4", "標籤5", "標籤6"], "core_keywords": ["核心詞1", "核心詞2", "核心詞3", "核心詞4", "核心詞5"], "long_tail_keywords": ["長尾詞1", "長尾詞2", "長尾詞3", "長尾詞4", "長尾詞5"]}`' + chr(10) +
                        f'專有名詞請統一使用：【{pos_str}】。'
                    )

                    chapters_ref = (chr(10) + '【本集真實時間軸章節參考】' + chr(10) + chr(10).join(chapters_quick_list[:12]) + chr(10)) if (chapters_quick_list and has_real_ts) else ''
                    blog_raw_resp, used_creative_model = call_gemini_safe(
                        prompt=f'請根據以下節目內容撰寫升級版 Blog 專欄文章（【務必以這組定案標題為核心命題與文章標題】：{best_title}，系列前綴參考：{series_prefix}）：' + chapters_ref + chr(10) + '【節目完整文稿】' + chr(10) + clean_full_text[:45000],
                        system_instruction=sys_prompt_blog,
                        requested_model=creative_model
                    )

                    m_html = re.search(r'\[BLOG_HTML_START\](.*?)\[BLOG_HTML_END\]', blog_raw_resp, re.S)
                    m_seo = re.search(r'\[SEO_JSON_START\](.*?)\[SEO_JSON_END\]', blog_raw_resp, re.S)

                    body_html = m_html.group(1).strip() if m_html else blog_raw_resp.split('[SEO_JSON_START]')[0].replace('[BLOG_HTML_START]', '').strip()
                    body_html = re.sub(r'^```html\s*|\s*```$', '', body_html, flags=re.I).strip()

                    seo_data = {
                        'post_title': f'{series_prefix}{best_title}' if series_prefix not in best_title else best_title,
                        'seo_title': best_title,
                        'meta_description': '',
                        'slug': 'aliang-edu-post',
                        'labels': list(base_tags),
                        'core_keywords': [],
                        'long_tail_keywords': []
                    }
                    if m_seo:
                        try:
                                seo_raw = re.sub(r'^```json\s*|\s*```$', '', m_seo.group(1).strip(), flags=re.I).strip()
                                parsed_seo = json.loads(seo_raw)
                                if isinstance(parsed_seo, dict):
                                    seo_data.update(parsed_seo)
                        except Exception:
                                pass

                    merged_labels = list(dict.fromkeys(base_tags + (seo_data.get('labels') or [])))
                    seo_data['labels'] = merged_labels
                    final_post_title = best_title if best_title and best_title != stem else (seo_data.get('post_title') or best_title)

                    yt_iframe_html = (
                        f'<iframe src="https://www.youtube.com/embed/{job_yt_id}" style="width: 100%; aspect-ratio: 16/9;" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen="allowfullscreen"></iframe>' + chr(10) + chr(10)
                    ) if job_yt_id else ''

                    blog_html_final = (
                        yt_iframe_html +
                        body_html + chr(10) +
                        '<hr>' + chr(10) +
                        cta_html + chr(10)
                    )

                    blog_md_final = (
                        f'# {final_post_title}' + chr(10) + chr(10) +
                        '## 📊 SEO 與關鍵字策略報告' + chr(10) +
                        f'- **建議文章標題**：{final_post_title}' + chr(10) +
                        f'- **SEO 搜尋標題**：{seo_data.get("seo_title", "")}' + chr(10) +
                        f'- **Meta 搜尋描述（<=150字）**：{seo_data.get("meta_description", "")}' + chr(10) +
                        f'- **建議英文固定網址（Slug）**：`{seo_data.get("slug", "")}`' + chr(10) +
                        f'- **Blogger 分類標籤（Labels）**：`{", ".join(merged_labels)}`' + chr(10) +
                        f'- **5 大核心關鍵字**：`{", ".join(seo_data.get("core_keywords") or [])}`' + chr(10) +
                        f'- **5 大長尾關鍵字**：`{", ".join(seo_data.get("long_tail_keywords") or [])}`' + chr(10) + chr(10) +
                        '---' + chr(10) + chr(10) +
                        '## 📝 部落格專欄文章預覽（HTML 已同步存為 `_部落格專欄文章.html`）' + chr(10) + chr(10) +
                        blog_html_final
                    )

                    (j_dir / 'blog_post.html').write_text(blog_html_final, encoding='utf-8')
                    (j_dir / 'blog_post.md').write_text(blog_md_final, encoding='utf-8')
                    (j_dir / 'blog_seo_meta.json').write_text(json.dumps({
                        'post_title': final_post_title,
                        'labels': merged_labels,
                        'seo_data': seo_data
                    }, ensure_ascii=False, indent=2), encoding='utf-8')
                    (work / 'blog_post.html').write_text(blog_html_final, encoding='utf-8')
                    (work / 'blog_post.md').write_text(blog_md_final, encoding='utf-8')
                    (work / 'blog_seo_meta.json').write_text(json.dumps({
                        'post_title': final_post_title,
                        'labels': merged_labels,
                        'seo_data': seo_data
                    }, ensure_ascii=False, indent=2), encoding='utf-8')
                    (j_dir / 'article_and_summary.md').write_text(blog_md_final, encoding='utf-8')
                    (j_dir / 'article_and_summary.txt').write_text(blog_md_final, encoding='utf-8')
                    (work / 'article_and_summary.md').write_text(blog_md_final, encoding='utf-8')
                    (work / 'article_and_summary.txt').write_text(blog_md_final, encoding='utf-8')

                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（[模組 C] 已用 {used_creative_model} 完成升級版 Blog 專欄文章！）'
                    lights_state['C'] = ('done', f'完成！已產出 HTML/MD ({used_creative_model})')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

                    if not blogger_publish_mode.startswith('不自動推送'):
                        is_draft_mode = '草稿' in blogger_publish_mode
                        push_or_update_post_to_blogger(
                                j_dir=j_dir,
                                post_title=final_post_title,
                                post_html=blog_html_final,
                                post_labels=merged_labels,
                                is_draft=is_draft_mode
                        )

                downstream_source_text = blog_html_final if blog_html_final else clean_full_text[:35000]

                # 模組 D：FB / IG / LINE 多平台社群宣傳貼文（無 Markdown 純文字直貼格式）
                social_md = ''
                if generate_social_posts:
                    lights_state['D'] = ('run', '正在撰寫 FB/IG/LINE 純文字貼文...')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)
                    print('📱 [模組 D] 正在產出「Facebook ＋ Instagram ＋ LINE」三平台專屬社群貼文（純文字無 Markdown 符號）...', flush=True)
                    sys_prompt_social = (
                        '你是一位精通社群行銷的自媒體總監。請根據提供的專欄文章與節目內容，一次產出三種平台的宣傳貼文（使用道地臺灣繁體中文，**嚴禁使用 `**` 或 `###` 等任何 Markdown 符號**，改用 Emoji 與空行排版，讓使用者複製後可直接貼上社群平台）：' + chr(10) +
                        f'【開頭鉤子（Hook）策略】：{social_hook_style}' + chr(10) + chr(10) +
                        '1. 在 `[FB_POST_START] ... [FB_POST_END]` 內輸出 Facebook 貼文：' + chr(10) +
                        '   - 吸睛開頭鉤子 ＋ 溫暖故事引入 ＋ 3~5 點以 `▸` 開頭的核心重點 ＋ 文末引發留言的互動問句 ＋ 收聽/閱讀行動呼籲（CTA）。' + chr(10) +
                        '2. 在 `[IG_POST_START] ... [IG_POST_END]` 內輸出 Instagram 貼文：' + chr(10) +
                        '   - 強化視覺感與 Emoji 排版、適合手機閱讀的短段落 ＋ 文末自動附上 8~10 個高觸及繁體中文 Hashtags（`#...`）。' + chr(10) +
                        '3. 在 `[LINE_POST_START] ... [LINE_POST_END]` 內輸出 LINE 社群／教師群組推播文案：' + chr(10) +
                        '   - 極致口語化、親切短小（約 100~180 字），並在關鍵處加上 `（建議搭配貼圖：...）` 的貼圖文字建議。' + chr(10) +
                        f'專有名詞請統一使用：【{pos_str}】。'
                    )
                    social_md, used_creative_model = call_gemini_safe(
                        prompt=f'請根據以下內容（推薦標題：{best_title}）撰寫 FB / IG / LINE 三平台純文字社群貼文：' + chr(10) + downstream_source_text[:35000],
                        system_instruction=sys_prompt_social,
                        requested_model=creative_model
                    )
                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（[模組 D] 已用 {used_creative_model} 完成 FB/IG/LINE 三平台社群貼文！）'
                    lights_state['D'] = ('done', f'完成！三平台貼文 ({used_creative_model})')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

                # 模組 E：四大視覺生圖提示詞大禮包（每張圖獨立標籤，方便單張一鍵複製）
                visual_md = ''
                if generate_visual_prompts:
                    lights_state['E'] = ('run', '正在設計四大視覺生圖提示詞...')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)
                    print('🎨 [模組 E] 正在產出「輪播圖 1:1 ＋ 資訊圖表 ＋ YT 封面 16:9 ＋ Reels 分鏡 9:16」四大視覺提示詞大禮包...', flush=True)
                    logo_rule = '並在畫面適當角落包含品牌 Logo 圖示（image2）。' if visual_include_logo else '不包含 Logo 圖示。'
                    CAROUSEL_HEADER_TEMPLATE = (
                        '# 多張圖片獨立生成任務' + chr(10) + chr(10) +
                        '本次任務共需生成 4 張獨立圖片。' + chr(10) + chr(10) +
                        '請使用 AI 原生繪圖工具，依照第 1 張至第 4 張的順序完成全部圖片。' + chr(10) + chr(10) +
                        '**執行規則：**' + chr(10) + chr(10) +
                        '1. 將每一張圖片視為獨立的繪圖任務。' + chr(10) +
                        '2. 每次呼叫繪圖工具時，只能使用目前編號對應的提示詞，且只生成 1 張圖片。' + chr(10) +
                        '3. 每次生成前，必須重新讀取目前編號的完整規格，包括人物設定、動作、場景、風格、標題、副標題與重點文字。' + chr(10) +
                        '4. 禁止直接重複使用前一張的完整提示詞。' + chr(10) +
                        '5. 各張圖片必須保留指定的共同視覺風格及人物身分特徵，但場景、動作、文字與畫面內容應依個別規格變化。' + chr(10) +
                        '6. 每張都是獨立完整的圖片，禁止將多張圖片合併成拼貼、九宮格、分割畫面、contact sheet、grid、panel 或 comic strip。' + chr(10) +
                        '7. 每完成一張，核對圖片編號與對應規格，再進行下一張。' + chr(10) +
                        '8. 如果圖片內容與目前編號不符，或明顯重複上一張，應判定為生成失敗，不可計入完成數量。' + chr(10) +
                        '9. 不得以重複圖片充數，也不得擅自改用程式合成或簡報排版冒充 AI 原生繪圖。' + chr(10) +
                        '10. 若無法完成指定張數，必須如實回報已完成與失敗的張數，不得宣稱全部完成。' + chr(10) + chr(10) +
                        '完成第 i 張並核對後，接續生成第 i+1 張，直到第 4 張完成；不要把完成單張視為整體任務結束，也不要等待使用者再次下達下一張指令。' + chr(10) + chr(10) +
                        '以下為各張圖片的獨立生成規格。'
                    )
                    REELS_HEADER_TEMPLATE = (
                        '# 多張圖片獨立生成任務' + chr(10) + chr(10) +
                        '本次任務共需生成 5 張獨立圖片。' + chr(10) + chr(10) +
                        '請使用 AI 原生繪圖工具，依照第 1 張至第 5 張的順序完成全部圖片。' + chr(10) + chr(10) +
                        '**執行規則：**' + chr(10) + chr(10) +
                        '1. 將每一張圖片視為獨立的繪圖任務。' + chr(10) +
                        '2. 每次呼叫繪圖工具時，只能使用目前編號對應的提示詞，且只生成 1 張圖片。' + chr(10) +
                        '3. 每次生成前，必須重新讀取目前編號的完整規格，包括人物設定、動作、場景、風格、標題、副標題與重點文字。' + chr(10) +
                        '4. 禁止直接重複使用前一張的完整提示詞。' + chr(10) +
                        '5. 各張圖片必須保留指定的共同視覺風格及人物身分特徵，但場景、動作、文字與畫面內容應依個別規格變化。' + chr(10) +
                        '6. 每張都是獨立完整的圖片，禁止將多張圖片合併成拼貼、九宮格、分割畫面、contact sheet、grid、panel 或 comic strip。' + chr(10) +
                        '7. 每完成一張，核對圖片編號與對應規格，再進行下一張。' + chr(10) +
                        '8. 如果圖片內容與目前編號不符，或明顯重複上一張，應判定為生成失敗，不可計入完成數量。' + chr(10) +
                        '9. 不得以重複圖片充數，也不得擅自改用程式合成或簡報排版冒充 AI 原生繪圖。' + chr(10) +
                        '10. 若無法完成指定張數，必須如實回報已完成與失敗的張數，不得宣稱全部完成。' + chr(10) + chr(10) +
                        '以下為各張圖片的獨立生成規格。'
                    )
                    SINGLE_IMAGE_RESTRICTION_TEMPLATE = (
                        '**單張完整圖片與獨立任務限制：**' + chr(10) + chr(10) +
                        '本段提示詞僅適用於第 N 張圖片。' + chr(10) + chr(10) +
                        '生成本張圖片時，必須使用本段指定的人物動作、場景、文字與構圖，不得混入其他編號的畫面內容或標題。' + chr(10) + chr(10) +
                        '這是一張單一且完整的圖片，禁止使用分割畫面、拼貼、多格漫畫、contact sheet、grid、panel、comic strip 或多圖合併。'
                    )
                    sys_prompt_visual = (
                        '你是一位頂尖的 AI 視覺藝術指導與提示詞工程師。請根據提供的內容，嚴格按照以下標籤分段輸出四大視覺生圖提示詞（使用臺灣繁體中文）：' + chr(10) +
                        f'【主角設定（image1）】：{visual_character_desc}，{logo_rule}' + chr(10) +
                        f'【視覺藝術風格】：{visual_style_preset}' + chr(10) + chr(10) +
                        '================================================' + chr(10) +
                        '1. 在 `[CAROUSEL_ALL_START] ... [CAROUSEL_ALL_END]` 標籤內，輸出一次可複製全部 4 張的「1:1 正方形社群輪播圖」完整生圖指令：' + chr(10) +
                        '   - 開頭必須一字不差放上以下多圖獨立任務標頭與 10 條執行規則：' + chr(10) +
                        CAROUSEL_HEADER_TEMPLATE + chr(10) +
                        '   - 接著依序輸出 `[第 1 張]`、`[第 2 張]`、`[第 3 張]`、`[第 4 張]`，每張規格如下：' + chr(10) +
                        '     * 開頭先寫：`請生成 1:1 正方形社群輪播圖。採用...風格...。畫面主體使用 image1 (講師：...)，...（具體描述人物動作、表情與場景佈景）。`' + chr(10) +
                        '     * 接著寫：`圖上排版與文字標示：`' + chr(10) +
                        '       - 第 1 張（封面）：`1. 圖片上方顯眼處直接呈現主標題與副標題： 主標題：「...」 副標題：「...」`、`2. 圖片底部直接以社群標籤 (Hashtag) 樣式呈現以下關鍵詞： #... #... #...`、`3. 圖片右下角設計行動呼籲 (CTA) 按鈕，文字為：「左滑解鎖完整祕笈 ➔」`' + chr(10) +
                        '       - 第 2 張（痛點/背景）：`1. 圖片上方顯眼處直接呈現主標題與副標題...`、`2. 圖片右側設計精美卡片區塊，直接以圓點條列排版以下內容： ● ... ● ... ● ...`、`3. 圖片底部直接以社群標籤 (Hashtag) 樣式呈現以下關鍵詞...`、`4. 圖片右下角設計行動呼籲 (CTA) 按鈕，文字為：「看下一頁解法 ➔」`' + chr(10) +
                        '       - 第 3 張（方法/步驟）：`1. 圖片上方顯眼處直接呈現主標題與副標題...`、`2. 圖片左側設計簡潔的步驟流程排版，直接以數字條列呈現以下內容： 1. ... 2. ... 3. ...`、`3. 圖片底部直接以社群標籤 (Hashtag) 樣式呈現以下關鍵詞...`、`4. 圖片右下角設計行動呼籲 (CTA) 按鈕，文字為：「收藏這篇實用貼文 ➔」`' + chr(10) +
                        '       - 第 4 張（結語/呼籲）：`1. 圖片上方顯眼處直接呈現主標題與副標題...`、`2. 圖片底部直接以社群標籤 (Hashtag) 樣式呈現以下關鍵詞...`、`3. 圖片右下角設計高亮且醒目的 CTA 按鈕，文字為：「立即點擊連結回看直播 ➔」`' + chr(10) +
                        '     * 每張圖規格後加上：`不可出現任何版權角色。整體要適合社群快速滑讀。`' + chr(10) +
                        '     * 每張圖最後必須一字不差加上以下三段式限制（將 N 替換為 1~4）：' + chr(10) +
                        SINGLE_IMAGE_RESTRICTION_TEMPLATE + chr(10) + chr(10) +
                        '================================================' + chr(10) +
                        '2. 在 `[REELS_ALL_START] ... [REELS_ALL_END]` 標籤內，輸出一次可複製全部 5 張的「Reels 9:16 直式分鏡（共 5 張）」完整生圖指令：' + chr(10) +
                        '   - 開頭必須一字不差放上以下 5 張多圖獨立任務標頭與 10 條執行規則：' + chr(10) +
                        REELS_HEADER_TEMPLATE + chr(10) +
                        '   - 接著依序輸出 `[第 1 張]`（封面／Hook）、`[第 2 張]`（問題背景）、`[第 3 張]`（方法一／重點一）、`[第 4 張]`（方法二／重點二）、`[第 5 張]`（結論／CTA），每張圖皆必須嚴格按以下欄位格式逐行填寫：' + chr(10) +
                        '     功能定位：第 N 張...（說明定位）' + chr(10) +
                        '     9:16 比例：9:16 垂直構圖，專為 Facebook Reels 設計。' + chr(10) +
                        '     整體風格：...' + chr(10) +
                        '     人物設定與對應 image：image1（講師：...），...' + chr(10) +
                        '     畫面主體與動作：image1（講師：...）...' + chr(10) +
                        '     場景背景：...' + chr(10) +
                        '     主標與副標：主標：...' + chr(10) +
                        '     副標：...' + chr(10) +
                        '     重點最多 3 點：' + chr(10) +
                        '     1. ...' + chr(10) +
                        '     2. ...' + chr(10) +
                        '     3. ...（第 5 張可為 2~3 點）' + chr(10) +
                        '     CTA：（第 1~4 張填「無」，第 5 張填「收藏起來，分享給需要的老師！關注更多教學應用」）' + chr(10) +
                        '     Logo 處理：' + ('右下角或適當角落呈現 image2 品牌 Logo' if visual_include_logo else '無') + chr(10) +
                        '     安全區與留白限制：上方留白 15%，下方預留 Reels 互動區不放重要文字與人臉，核心視覺集中於畫面中央黃金比例區。單張完整圖片與獨立任務限制：請獨立生成此張 9:16 直式圖片。' + chr(10) +
                        '     （接著空一行，附上與上方相同的 `**單張完整圖片與獨立任務限制：**` 三段式宣告，將 N 替換為 1~5！）' + chr(10) + chr(10) +
                        '================================================' + chr(10) +
                        '3. 在 `[REELS_CAPTION_START] ... [REELS_CAPTION_END]` 標籤內，輸出【Reels 短影音上架文案】（90~110 字吸睛短文案 ＋ Hashtags）。' + chr(10) +
                        '4. 在 `[INFOGRAPHIC_ZH_START] ... [INFOGRAPHIC_ZH_END]` 標籤內，輸出最適資訊圖表型態推薦說明 ＋ 完整繁體中文生圖提示詞。' + chr(10) +
                        '5. 在 `[INFOGRAPHIC_EN_START] ... [INFOGRAPHIC_EN_END]` 標籤內，輸出資訊圖表完整英文生圖提示詞 (English Prompt，指定圖內文字標籤為 Traditional Chinese)。' + chr(10) +
                        f'6. 在 `[YT_THUMB_START] ... [YT_THUMB_END]` 標籤內，輸出 YouTube 封面 `16:9` 六段式生圖提示詞（搭配標題：{best_title}，依序包含 `[人物設定]`、`[主體與動作]`、`[地點/背景]`、`[構圖/鏡頭]`、`[文字]`、`[藝術風格：畫面長寬比為 16:9。調整人物的光線與陰影以完全符合環境氛圍。]`）。'
                    )
                    visual_md, used_creative_model = call_gemini_safe(
                        prompt='請根據以下內容產出標籤化的【四大視覺生圖提示詞大禮包】：' + chr(10) + downstream_source_text[:35000],
                        system_instruction=sys_prompt_visual,
                        requested_model=creative_model
                    )
                    ai_progress.value += 1
                    ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（[模組 E] 已用 {used_creative_model} 完成四大視覺生圖提示詞大禮包！）'
                    lights_state['E'] = ('done', f'完成！四大視覺提示詞 ({used_creative_model})')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

                # 產出一頁式「🌟 全媒體創作戰情室.html」並儲存黃金 3 檔
                srt_final_text = (j_dir / 'transcript.srt').read_text(encoding='utf-8') if (j_dir / 'transcript.srt').is_file() else ''
                curr_seo_meta = {}
                if (j_dir / 'blog_seo_meta.json').is_file():
                    try:
                        curr_seo_meta = json.loads((j_dir / 'blog_seo_meta.json').read_text(encoding='utf-8')).get('seo_data') or {}
                    except Exception:
                        pass

                standalone_warroom_html, inline_warroom_html = build_all_in_one_warroom_html(
                    stem=stem,
                    readable_txt=readable_full_txt,
                    srt_txt=srt_final_text,
                    mod_b_raw=titles_and_chapters_md,
                    yt_desc_plain=yt_desc_plain,
                    yt_chapters_plain=yt_chapters_plain,
                    blog_title=(curr_seo_meta.get('post_title') or best_title),
                    blog_html=blog_html_final,
                    seo_data=curr_seo_meta,
                    social_raw=social_md,
                    visual_raw=visual_md
                )
                (j_dir / 'all_in_one_hub.html').write_text(standalone_warroom_html, encoding='utf-8')
                (work / 'all_in_one_hub.html').write_text(standalone_warroom_html, encoding='utf-8')

                ai_progress.value = total_steps
                ai_label.value = f'100%（[{stem}] 全媒體創作流水線全部完成！）'

                sync_job_to_drive(j_dir, updated_payload)

                # 在 Colab 畫面上直接展示這份一頁式「🌟 全媒體創作戰情室」＋ 頂部 3 大黃金檔快捷下載列！
                dl_hub_btn = widgets.Button(description=f'⬇️ 下載 {stem}_全媒體創作戰情室.html', button_style='success', layout=widgets.Layout(width='310px'))
                dl_srt_btn = widgets.Button(description=f'⬇️ 下載修正後 {stem}.srt', button_style='info', layout=widgets.Layout(width='230px'))
                dl_txt_btn = widgets.Button(description=f'⬇️ 下載全本順稿 {stem}.txt', button_style='info', layout=widgets.Layout(width='235px'))

                def _make_dl(p_file):
                    def _handler(_):
                        from google.colab import files
                        if Path(p_file).is_file():
                                files.download(str(p_file))
                    return _handler

                dl_hub_btn.on_click(_make_dl(j_dir / 'all_in_one_hub.html'))
                dl_srt_btn.on_click(_make_dl(j_dir / 'transcript.srt'))
                dl_txt_btn.on_click(_make_dl(j_dir / 'transcript.txt'))

                done_banner_html = (
                    '<div style="margin:12px 0 8px 0; padding:14px 18px; border:2px solid #2e7d32; background:#e8f5e9; border-radius:10px; color:#1b5e20; font-size:14.5px; line-height:1.7;">'
                    f'<b>🎉 階段 4/6 全媒體創作工作台執行完畢！（作品：<code>{stem}</code>）</b><br>'
                    '💾 所有黃金 3 檔（<code>.srt</code> 修正字幕、<code>.txt</code> 高可讀性全本順稿、<code>_全媒體創作戰情室.html</code>）皆已自動存回您的 Google 雲端硬碟專屬同名子資料夾！您可直接點擊下方各區塊的「📋 一鍵複製」按鈕，或繼續執行「步驟 5：下載全媒體創作包」。'
                    '</div>'
                )
                display(
                    HTML(done_banner_html),
                    widgets.HBox([dl_hub_btn, dl_srt_btn, dl_txt_btn], layout=widgets.Layout(gap='8px', margin='10px 0')),
                    HTML(inline_warroom_html)
                )

                if total_jobs > 1 and j_idx < total_jobs:
                    print('⏳ 批次模式：等待 8 秒安全冷卻後開始處理下一集...', flush=True)
                    time.sleep(8)



            # 判斷是否啟用「兩段式定調」（單集且有開啟模組 B 與任一下游模組 C/D/E 時）
            has_downstream = any([generate_blog_post, generate_social_posts, generate_visual_prompts])
            if generate_titles_and_chapters:
                lights_state['B'] = ('run', '正在發想 9 組爆款標題供您挑選...')
                five_lights_box.value = render_five_lights_html(lights_state, stem)
                print('🔥 [模組 B1] 正在發想 3 大方案共 9 組爆款主副標題建議...', flush=True)
                titles_only_raw, used_creative_model = generate_9_titles_only()
                parsed_choices = extract_9_title_choices(titles_only_raw, fallback_stem=stem)
                best_title = parsed_choices[0][1] if parsed_choices else stem
                ai_progress.value += 1
                ai_label.value = f'{int(ai_progress.value / total_steps * 100)}%（[模組 B1] 已產出 {len(parsed_choices)} 組爆款標題！）'

                if two_stage_pause and total_jobs == 1 and has_downstream:
                    lights_state['B'] = ('run', '⏸️ 等待您挑選／微調定案標題...')
                    five_lights_box.value = render_five_lights_html(lights_state, stem)

                    title_radio = widgets.RadioButtons(
                        options=parsed_choices,
                        value=best_title,
                        layout=widgets.Layout(width='98%')
                    )
                    title_edit = widgets.Text(
                        value=best_title,
                        description='✏️ 定案標題：',
                        placeholder='可直接在這裡微調修正您選定的主副標題文字',
                        style={'description_width': '95px'},
                        layout=widgets.Layout(width='95%')
                    )
                    title_radio.observe(lambda ch: setattr(title_edit, 'value', ch['new']) if ch.get('new') else None, names='value')

                    regen_titles_btn = widgets.Button(description='🔄 重新發想 9 組標題', button_style='', layout=widgets.Layout(width='185px'))
                    proceed_stage2_btn = widgets.Button(description='✨ 第二階段：鎖定此標題，立即產出 YT簡介・Blog・社群・視覺提示詞', button_style='success', layout=widgets.Layout(width='460px'))
                    stage2_out = widgets.Output()
                    _s2_running = [False]

                    def on_regen_titles(_):
                        regen_titles_btn.disabled = True
                        regen_titles_btn.description = '⏳ 重新發想中...'
                        try:
                            new_raw, _ = generate_9_titles_only()
                            new_choices = extract_9_title_choices(new_raw, fallback_stem=stem)
                            titles_only_raw_box[0] = new_raw
                            title_radio.options = new_choices
                            if new_choices:
                                title_radio.value = new_choices[0][1]
                                title_edit.value = new_choices[0][1]
                        finally:
                            regen_titles_btn.disabled = False
                            regen_titles_btn.description = '🔄 重新發想 9 組標題'

                    titles_only_raw_box = [titles_only_raw]

                    def on_proceed_stage2(_):
                        if _s2_running[0]:
                            return
                        chosen = title_edit.value.strip() or title_radio.value or stem
                        _s2_running[0] = True
                        proceed_stage2_btn.disabled = True
                        proceed_stage2_btn.description = '⏳ 正在鎖定標題全速產出中...'
                        stage2_out.clear_output(wait=True)
                        try:
                            with stage2_out:
                                run_stage2_with_anchored_title(chosen, titles_only_raw_box[0])
                        finally:
                            _s2_running[0] = False
                            proceed_stage2_btn.disabled = False
                            proceed_stage2_btn.description = '✨ 第二階段：鎖定此標題，重新產出全媒體作品'

                    regen_titles_btn.on_click(on_regen_titles)
                    proceed_stage2_btn.on_click(on_proceed_stage2)

                    picker_card = widgets.VBox([
                        widgets.HTML(
                            '<div style="background:#fff8e1; border-left:5px solid #ffb300; padding:10px 14px; border-radius:8px; color:#4e342e; margin-bottom:8px;">'
                            '<b>🏆 第一階段完成！請從下方 9 組建議中「點選最喜歡的標題」（亦可手動微調字句），再按綠色按鈕聯動產出後續所有作品：</b>'
                            '</div>'
                        ),
                        title_radio,
                        title_edit,
                        widgets.HBox([proceed_stage2_btn, regen_titles_btn], layout=widgets.Layout(gap='10px', margin='8px 0 0 0'))
                    ], layout=widgets.Layout(padding='14px', border='2px solid #ffb300', border_radius='10px', margin='10px 0'))
                    display(picker_card, stage2_out)
                    return

            # 若為全自動模式、批次模式或未勾選模組 B，直接接續執行第二階段
            run_stage2_with_anchored_title(best_title, titles_only_raw)
        print(chr(10) + '🎉 階段 4/6 全媒體創作工作台執行完畢！所有檔案皆已自動存回雲端硬碟專屬同名子資料夾（亦可繼續執行「步驟 5：下載 ZIP 結果」）。', flush=True)


# ==============================================================================
# 🎛️ Web App 級分頁式互動控制中心（含情境快捷鍵、Blogger 選單、臨時匯入 SRT/TXT 文稿）
# ==============================================================================
banner_step4_widget = widgets.HTML(value=render_step4_banner(job_dirs))
initial_key, key_source_label = detect_initial_api_key()

pw_input = widgets.Password(
    value=initial_key,
    placeholder='請貼上您的 Gemini API Key（預設隱藏 ••••••••）',
    description='🔑 API 金鑰：',
    style={'description_width': '95px'},
    layout=widgets.Layout(width='400px', display='flex')
)
txt_input = widgets.Text(
    value=initial_key,
    placeholder='請貼上您的 Gemini API Key',
    description='🔑 API 金鑰：',
    style={'description_width': '95px'},
    layout=widgets.Layout(width='400px', display='none')
)
widgets.jslink((pw_input, 'value'), (txt_input, 'value'))

toggle_eye_btn = widgets.Button(
    description='👁️ 顯示金鑰',
    tooltip='切換顯示或隱藏 API 金鑰明文',
    layout=widgets.Layout(width='115px')
)
run_ai_btn = widgets.Button(
    description='🚀 開始執行 AI 全媒體創作',
    button_style='success',
    layout=widgets.Layout(width='230px')
)

# 互動式模組勾選開關（預設帶入左側表單設定，並可透過情境快捷鍵一鍵切換！）
cb_mod_a = widgets.Checkbox(value=bool(enable_ai_proofread), description='🔤 模組 A：AI 字幕同音字校對', indent=False, layout=widgets.Layout(width='230px'))
cb_mod_b = widgets.Checkbox(value=bool(generate_titles_and_chapters), description='🔥 模組 B：9 組爆款標題 & 章節', indent=False, layout=widgets.Layout(width='235px'))
cb_mod_c = widgets.Checkbox(value=bool(generate_blog_post), description='✍️ 模組 C：升級版 Blog 專欄', indent=False, layout=widgets.Layout(width='215px'))
cb_mod_d = widgets.Checkbox(value=bool(generate_social_posts), description='📱 模組 D：FB/IG/LINE 貼文', indent=False, layout=widgets.Layout(width='215px'))
cb_mod_e = widgets.Checkbox(value=bool(generate_visual_prompts), description='🎨 模組 E：四大視覺生圖提示詞', indent=False, layout=widgets.Layout(width='235px'))

preset_all_btn = widgets.Button(description='⚡ 情境 1：全套影音創作者大禮包（全選）', button_style='info', layout=widgets.Layout(width='275px'))
preset_blog_btn = widgets.Button(description='✍️ 情境 2：只要寫 Blog 專欄＋推 Blogger', button_style='', layout=widgets.Layout(width='270px'))
preset_srt_btn = widgets.Button(description='🔤 情境 3：只要 AI 校對 SRT 字幕', button_style='', layout=widgets.Layout(width='230px'))
cb_two_stage_mode = widgets.Checkbox(value=True, description='🎯 兩段式定調：先產出 9 組標題讓我挑選微調，再聯動產出 YT簡介・Blog・社群・視覺提示詞（單集推薦！）', indent=False, layout=widgets.Layout(width='98%'))

def apply_preset(a, b, c, d, e):
    cb_mod_a.value, cb_mod_b.value, cb_mod_c.value, cb_mod_d.value, cb_mod_e.value = a, b, c, d, e

preset_all_btn.on_click(lambda _: apply_preset(True, True, True, True, True))
preset_blog_btn.on_click(lambda _: apply_preset(False, True, True, False, False))
preset_srt_btn.on_click(lambda _: apply_preset(True, False, False, False, False))

key_status_html = widgets.HTML(
    value=(
        f'<span style="color:#2e7d32; font-size:13px;">✅ 金鑰已就緒（來源：{key_source_label}，預設遮罩保護中）｜ 雙引擎：校對 <code>{proofread_model}</code> ＋ 創作 <code>{creative_model}</code>（內建無權限自動降階）</span>'
        if initial_key else
        '<span style="color:#e65100; font-size:13px;">👈 請先貼上 Gemini API Key，確認勾選模組後點擊「🚀 開始執行 AI 全媒體創作」</span>'
    )
)
pipeline_out = widgets.Output()

_is_visible = [False]

def on_toggle_eye(_):
    _is_visible[0] = not _is_visible[0]
    if _is_visible[0]:
        pw_input.layout.display = 'none'
        txt_input.layout.display = 'flex'
        toggle_eye_btn.description = '🙈 隱藏金鑰'
        toggle_eye_btn.button_style = 'warning'
    else:
        txt_input.layout.display = 'none'
        pw_input.layout.display = 'flex'
        toggle_eye_btn.description = '👁️ 顯示金鑰'
        toggle_eye_btn.button_style = ''

_is_pipeline_running = [False]

def on_click_run_ai(_):
    global enable_ai_proofread, generate_titles_and_chapters, generate_blog_post, generate_social_posts, generate_visual_prompts, job_dirs
    if _is_pipeline_running[0]:
        key_status_html.value = '<span style="color:#e65100; font-weight:bold;">⏳ AI 全媒體創作流水線正在執行中，請稍候（請勿重複點擊）...</span>'
        return
    enable_ai_proofread = bool(cb_mod_a.value)
    generate_titles_and_chapters = bool(cb_mod_b.value)
    generate_blog_post = bool(cb_mod_c.value)
    generate_social_posts = bool(cb_mod_d.value)
    generate_visual_prompts = bool(cb_mod_e.value)

    job_dirs = scan_ready_job_dirs()
    if not job_dirs:
        key_status_html.value = '<span style="color:#d32f2f; font-weight:bold;">⚠️ 目前尚未載入任何音檔或文稿！請切換至右方「📝 分頁 ③：直接匯入現成 SRT/TXT 文稿」上傳或貼上文字後再按執行！</span>'
        ctrl_tabs.selected_index = 2
        return

    k = pw_input.value.strip()
    if not k:
        key_status_html.value = '<span style="color:#d32f2f; font-weight:bold;">❌ 金鑰不可為空白！請貼上您的 Gemini API Key。</span>'
        return
    os.environ['ALTA_CACHED_GEMINI_KEY'] = k
    if _is_visible[0]:
        on_toggle_eye(None)
    _is_pipeline_running[0] = True
    run_ai_btn.disabled = True
    run_ai_btn.description = '⏳ AI 創作執行中...'
    key_status_html.value = '<span style="color:#1565c0; font-weight:bold; font-size:13px;">🔒 金鑰已安全鎖定於記憶體，正在執行全媒體創作流水線...</span>'
    try:
        run_all_in_one_pipeline(k, pipeline_out, two_stage_pause=bool(cb_two_stage_mode.value))
        key_status_html.value = '<span style="color:#2e7d32; font-weight:bold; font-size:13.5px;">🎉 全媒體創作流水線執行完畢！成果已顯示於下方並同步存回雲端硬碟！</span>'
    finally:
        _is_pipeline_running[0] = False
        run_ai_btn.disabled = False
        run_ai_btn.description = '🚀 開始執行 AI 全媒體創作'

toggle_eye_btn.on_click(on_toggle_eye)
run_ai_btn.on_click(on_click_run_ai)

tab1_ai_box = widgets.VBox([
    widgets.HTML('<div style="font-size:13px; color:#004d40; margin-bottom:4px;"><b>⚡ 快速情境組合（點一下自動勾選對應模組）：</b></div>'),
    widgets.HBox([preset_all_btn, preset_blog_btn, preset_srt_btn], layout=widgets.Layout(flex_flow='row wrap', gap='6px', margin='0 0 8px 0')),
    widgets.HBox([cb_mod_a, cb_mod_b, cb_mod_c, cb_mod_d, cb_mod_e], layout=widgets.Layout(flex_flow='row wrap', margin='0 0 6px 0')),
    cb_two_stage_mode,
    widgets.HBox([pw_input, txt_input, toggle_eye_btn, run_ai_btn], layout=widgets.Layout(align_items='center')),
    key_status_html
], layout=widgets.Layout(padding='8px'))

# ==========================================================================
# 分頁 ②：Google Blogger 一鍵連結 ＋ 自動抓取個人網誌下拉選單 ＋ 雲端永久記憶
# ==========================================================================
initial_b_auth = load_blogger_auth()
initial_b_token = (initial_b_auth.get('access_token') or '').strip()

b_pw_input = widgets.Password(
    value=initial_b_token,
    placeholder='貼上 Blogger Access Token 或 OAuth JSON（預設隱藏 ••••••••）',
    description='🔑 Blogger 授權：',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='390px', display='flex')
)
b_txt_input = widgets.Text(
    value=initial_b_token,
    placeholder='貼上 Blogger Access Token 或 OAuth JSON',
    description='🔑 Blogger 授權：',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='390px', display='none')
)
widgets.jslink((b_pw_input, 'value'), (b_txt_input, 'value'))

b_eye_btn = widgets.Button(description='👁️ 顯示', layout=widgets.Layout(width='90px'))
b_connect_btn = widgets.Button(description='🔗 連結並抓網誌清單', button_style='info', layout=widgets.Layout(width='175px'))
b_disconnect_btn = widgets.Button(description='🔌 清除記憶', layout=widgets.Layout(width='105px'))

b_blog_dropdown = widgets.Dropdown(
    options=[('（尚未連結 — 請先完成上方 Blogger 授權連結）', '')],
    value='',
    description='📝 選擇網誌：',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='450px')
)
b_push_draft_btn = widgets.Button(description='📤 一鍵推送至【草稿夾】', button_style='success', layout=widgets.Layout(width='195px'))
b_push_pub_btn = widgets.Button(description='🌐 直接公開發布', button_style='warning', layout=widgets.Layout(width='145px'))

b_guide_html = widgets.HTML(
    value=(
        '<div style="font-size:13px; color:#37474f; line-height:1.75; margin-bottom:8px;">'
        '<b>☁️ 個人雲端永久記憶（存於個人雲端硬碟，跨帳號適用，分享 Colab 絕不外洩帳號）</b><br>'
        '💡 <b>快速連結 3 步驟</b>：'
        '① 先點此按鈕 👉 '
        '<button onclick="navigator.clipboard.writeText(\'https://www.googleapis.com/auth/blogger\').then(()=>{this.innerHTML=\'✅ 已複製權限網址！\';this.style.background=\'#2e7d32\';})" '
        'style="background:#00796b; color:#fff; border:none; border-radius:5px; padding:2px 10px; font-size:12.5px; cursor:pointer; font-weight:bold;">📋 點此一鍵複製 Blogger 權限網址</button> '
        f'➔ ② 開啟乾淨版 👉 <a href="{OAUTH_PLAYGROUND_BLOGGER_URL}" target="_blank" style="color:#00695c; font-weight:bold; text-decoration:underline;">Google 官方 OAuth Playground 授權頁</a>，'
        '貼到左下角 <code>Input your own scopes</code> 框框後按藍色 <b>Authorize APIs</b>（請選擇<b>擁有 Blogger 網誌的那個 Google 帳號</b>登入！）<br>'
        '➔ ③ 在 Step 2 點擊 <b>Exchange authorization code for tokens</b>，複製右侧回應的 <b>整段 JSON（或 <code>Refresh token</code> / <code>Access token</code>）</b> 貼入下方並按「<b>🔗 連結並抓網誌清單</b>」！（貼入含 <code>Refresh token</code> 或 JSON 可自動續約！）'
        '</div>'
    )
)
b_status_html = widgets.HTML(value='')
b_action_out = widgets.Output()

_b_visible = [False]

def on_b_toggle_eye(_):
    _b_visible[0] = not _b_visible[0]
    if _b_visible[0]:
        b_pw_input.layout.display = 'none'
        b_txt_input.layout.display = 'flex'
        b_eye_btn.description = '🙈 隱藏'
        b_eye_btn.button_style = 'warning'
    else:
        b_txt_input.layout.display = 'none'
        b_pw_input.layout.display = 'flex'
        b_eye_btn.description = '👁️ 顯示'
        b_eye_btn.button_style = ''

def populate_blog_dropdown(blogs, selected_id=''):
    if not blogs:
        b_blog_dropdown.options = [('（此帳號名下查無 Blogger 網誌）', '')]
        b_blog_dropdown.value = ''
        return
    opts = []
    valid_ids = []
    for b in blogs:
        b_id = str(b['id'])
        b_name = b.get('name') or '未命名網誌'
        b_url = b.get('url') or ''
        opts.append((f'📗 {b_name} ({b_url})', b_id))
        valid_ids.append(b_id)
    b_blog_dropdown.options = opts
    b_blog_dropdown.value = selected_id if selected_id in valid_ids else valid_ids[0]

def on_blog_dropdown_change(change):
    new_id = change.get('new')
    if not new_id:
        return
    cur_auth = load_blogger_auth()
    blogs = cur_auth.get('blogs') or []
    found = next((b for b in blogs if str(b.get('id')) == str(new_id)), None)
    if found:
        cur_auth['selected_blog_id'] = str(found['id'])
        cur_auth['selected_blog_name'] = found.get('name') or ''
        cur_auth['selected_blog_url'] = found.get('url') or ''
        save_blogger_auth(cur_auth)
        b_status_html.value = f'<span style="color:#00695c; font-size:13px;">✅ 已切換預設發布網誌為：<b>{found.get("name")}</b> ({found.get("url")})［已自動寫入雲端永久記憶］</span>'

b_blog_dropdown.observe(on_blog_dropdown_change, names='value')

def on_b_connect(_):
    raw_val = b_pw_input.value.strip()
    cur_auth = load_blogger_auth()
    if raw_val:
        cur_auth = parse_user_blogger_token_input(raw_val, cur_auth)
    if _b_visible[0]:
        on_b_toggle_eye(None)
    b_status_html.value = '<span style="color:#1565c0; font-size:13px;">⏳ 正在連線 Google Blogger API 並自動抓取您名下的網誌清單...</span>'
    try:
        blogs, valid_tok = fetch_user_blogs_with_auth(cur_auth)
        b_pw_input.value = valid_tok
        populate_blog_dropdown(blogs, str(cur_auth.get('selected_blog_id') or ''))
        if blogs:
            sel_id = b_blog_dropdown.value
            sel_b = next((b for b in blogs if str(b['id']) == str(sel_id)), blogs[0])
            b_status_html.value = f'<span style="color:#2e7d32; font-size:13px;">✅ <b>已成功連結 Google Blogger！</b>共抓到 <b>{len(blogs)}</b> 個個人網誌，目前選中：<b>{sel_b["name"]}</b></span>'
        else:
            b_status_html.value = '<span style="color:#e65100; font-size:13px;">⚠️ 授權成功，但此帳號名下尚未建立任何 Blogger 網誌。</span>'
    except Exception as err:
        b_status_html.value = f'<span style="color:#d32f2f; font-size:13px;">❌ 連結失敗：{err}（請點擊上方「Google 官方 Blogger 一鍵快速授權頁」取得最新 Access Token 貼上）</span>'

def on_b_disconnect(_):
    clear_blogger_auth()
    b_pw_input.value = ''
    b_blog_dropdown.options = [('（已清除雲端授權記憶 — 請重新連結）', '')]
    b_blog_dropdown.value = ''
    b_status_html.value = '<span style="color:#78909c; font-size:13px;">🔌 已從雲端硬碟與記憶體中清除您的 Blogger 授權紀錄。</span>'

def trigger_manual_blogger_push(is_draft_mode):
    b_action_out.clear_output(wait=True)
    cur_auth = load_blogger_auth()
    if b_pw_input.value.strip() and (b_pw_input.value.strip() != cur_auth.get('access_token') or not cur_auth.get('blogs')):
        on_b_connect(None)
    pushed_any = False
    for j_dir in scan_ready_job_dirs():
        html_f = j_dir / 'blog_post.html'
        if not html_f.is_file():
            continue
        pushed_any = True
        html_content = html_f.read_text(encoding='utf-8')
        meta_f = j_dir / 'blog_seo_meta.json'
        post_title, post_labels = '', []
        if meta_f.is_file():
            try:
                m = json.loads(meta_f.read_text(encoding='utf-8'))
                post_title = m.get('post_title') or ''
                post_labels = m.get('labels') or []
            except Exception:
                pass
        if not post_title:
            raw_info = json.loads((j_dir / 'raw_segments.json').read_text(encoding='utf-8')) if (j_dir / 'raw_segments.json').is_file() else {}
            post_title = raw_info.get('stem') or '升級版部落格專欄文章'
        push_or_update_post_to_blogger(j_dir=j_dir, post_title=post_title, post_html=html_content, post_labels=post_labels, is_draft=is_draft_mode, out_widget=b_action_out)
    if not pushed_any:
        with b_action_out:
            print('⚠️ 尚未找到已生成的 Blog 專欄文章！請先在【分頁 ①】勾選「模組 C」並按「🚀 開始執行 AI 全媒體創作」。', flush=True)

b_eye_btn.on_click(on_b_toggle_eye)
b_connect_btn.on_click(on_b_connect)
b_disconnect_btn.on_click(on_b_disconnect)
b_push_draft_btn.on_click(lambda _: trigger_manual_blogger_push(True))
b_push_pub_btn.on_click(lambda _: trigger_manual_blogger_push(False))

if initial_b_auth.get('access_token') or initial_b_auth.get('refresh_token'):
    try:
        blogs_live, valid_tok = fetch_user_blogs_with_auth(initial_b_auth)
        b_pw_input.value = valid_tok
        populate_blog_dropdown(blogs_live, str(initial_b_auth.get('selected_blog_id') or ''))
        if blogs_live:
            sel_id = b_blog_dropdown.value
            sel_b = next((b for b in blogs_live if str(b['id']) == str(sel_id)), blogs_live[0])
            b_status_html.value = f'<span style="color:#2e7d32; font-size:13px;">✅ 已從您的雲端永久記憶自動載入 Blogger 連結！目前選定網誌：<b>{sel_b["name"]}</b> ({sel_b["url"]})</span>'
    except Exception:
        cached_blogs = initial_b_auth.get('blogs') or []
        if cached_blogs:
            populate_blog_dropdown(cached_blogs, str(initial_b_auth.get('selected_blog_id') or ''))
        b_status_html.value = '<span style="color:#e65100; font-size:13px;">ℹ️ 雲端永久記憶中的短期 Access Token 已逾時，請點上方「Google 官方 Blogger 一鍵快速授權頁」複製最新 Token 貼上即可續用！</span>'
else:
    b_status_html.value = '<span style="color:#546e7a; font-size:13px;">ℹ️ 尚未連結個人 Blogger（若需一鍵推送至 Blogger 草稿夾，請依上方 30 秒步驟貼入授權後按「🔗 連結並抓網誌清單」）。</span>'

tab2_blogger_box = widgets.VBox([
    b_guide_html,
    widgets.HBox([b_pw_input, b_txt_input, b_eye_btn, b_connect_btn, b_disconnect_btn]),
    widgets.HBox([b_blog_dropdown, b_push_draft_btn, b_push_pub_btn], layout=widgets.Layout(margin='6px 0 0 0')),
    b_status_html,
    b_action_out
], layout=widgets.Layout(padding='8px'))

# ==========================================================================
# 分頁 ③：📝 免轉錄！直接匯入現成 .srt / .vtt / .txt 文稿或貼上文字（免跑步驟 1~3）
# ==========================================================================
q_upload_widget = widgets.FileUpload(
    accept='.srt,.vtt,.txt,.md',
    multiple=False,
    description='📂 選取 SRT/TXT 檔',
    button_style='info',
    layout=widgets.Layout(width='210px')
)
q_title_input = widgets.Text(
    value='',
    placeholder='選填：自訂作品標題（留空則自動使用上傳檔名）',
    description='📌 作品名稱：',
    style={'description_width': '95px'},
    layout=widgets.Layout(width='420px')
)
q_paste_area = widgets.Textarea(
    value='',
    placeholder='或者直接在此貼上您的 .srt 字幕全文、會議紀錄、演講逐字稿或文章筆記...',
    description='✍️ 直接貼文稿：',
    style={'description_width': '95px'},
    layout=widgets.Layout(width='95%', height='135px')
)
q_apply_btn = widgets.Button(
    description='⚡ 載入此文稿並立刻切換至【分頁 ①】準備 AI 創作',
    button_style='success',
    layout=widgets.Layout(width='360px', margin='6px 0 0 100px')
)
q_status_html = widgets.HTML(
    value='<span style="color:#00695c; font-size:12.5px;">💡 若您手邊已經有 <code>.srt</code> 字幕或 <code>.txt</code> 文字檔，連步驟 1~3 都不用跑，直接在此選取檔案或貼上文字即可！</span>'
)

def _extract_uploaded_file(up_val):
    '''相容 ipywidgets 7 (dict) 與 ipywidgets 8 (tuple) 的 FileUpload 值解析'''
    if not up_val:
        return '', b''
    if isinstance(up_val, dict):
        fname = next(iter(up_val.keys()))
        item = up_val[fname]
        content = item.get('content', b'') if isinstance(item, dict) else b''
        return fname, bytes(content)
    if isinstance(up_val, (list, tuple)) and len(up_val) > 0:
        item = up_val[0]
        fname = item.get('name', 'uploaded.srt')
        content = item.get('content', b'')
        return fname, bytes(content)
    return '', b''

def on_q_upload_change(change):
    fname, raw_b = _extract_uploaded_file(q_upload_widget.value)
    if not raw_b:
        return
    raw_str = ''
    for enc in ('utf-8-sig', 'utf-8', 'cp950', 'big5', 'utf-16'):
        try:
            raw_str = raw_b.decode(enc)
            break
        except Exception:
            continue
    if not raw_str:
        raw_str = raw_b.decode('utf-8', errors='replace')
    info = quick_ingest_text_in_step4(raw_str, q_title_input.value.strip() or fname)
    banner_step4_widget.value = render_step4_banner(job_dirs)
    ts_msg = '含時間碼字幕（100% 保留原始時間軸）' if info['has_real_timestamps'] else '純文字文稿（已智慧切分句子）'
    q_status_html.value = f'<span style="color:#2e7d32; font-weight:bold; font-size:13px;">✅ 已成功載入檔案 <code>{fname}</code>（{ts_msg}，共 {len(info["cues"])} 句，已套用錯字替換 {info["deterministic_rule_hits"]} 處）！請切回【分頁 ①】點擊「🚀 開始執行 AI 全媒體創作」！</span>'
    key_status_html.value = f'<span style="color:#2e7d32; font-weight:bold; font-size:13px;">⚡ 已載入現成文稿 <code>{info["stem"]}</code>（共 {len(info["cues"])} 句）！請點擊「🚀 開始執行 AI 全媒體創作」立即產出！</span>'

q_upload_widget.observe(on_q_upload_change, names='value')

def on_q_apply_paste(_):
    body = q_paste_area.value.strip()
    if not body:
        q_status_html.value = '<span style="color:#d32f2f; font-weight:bold;">❌ 請先選取檔案或在上方文字框貼上文稿內容！</span>'
        return
    t_name = q_title_input.value.strip() or '現成文稿直通創作'
    info = quick_ingest_text_in_step4(body, t_name)
    banner_step4_widget.value = render_step4_banner(job_dirs)
    ts_msg = '含時間碼 SRT/VTT 字幕' if info['has_real_timestamps'] else '純文字文稿（已智慧分句）'
    q_status_html.value = f'<span style="color:#2e7d32; font-weight:bold; font-size:13px;">✅ 已成功載入文稿 <code>{info["stem"]}</code>（{ts_msg}，共 {len(info["cues"])} 句）！已為您切換至【分頁 ①】，請按「🚀 開始執行 AI 全媒體創作」！</span>'
    key_status_html.value = f'<span style="color:#2e7d32; font-weight:bold; font-size:13px;">⚡ 已載入現成文稿 <code>{info["stem"]}</code>（共 {len(info["cues"])} 句）！請點擊「🚀 開始執行 AI 全媒體創作」立即產出！</span>'
    ctrl_tabs.selected_index = 0

q_apply_btn.on_click(on_q_apply_paste)

tab3_quick_ingest_box = widgets.VBox([
    q_status_html,
    widgets.HBox([q_upload_widget, q_title_input], layout=widgets.Layout(align_items='center', margin='6px 0')),
    q_paste_area,
    q_apply_btn
], layout=widgets.Layout(padding='8px'))

ctrl_tabs = widgets.Tab(children=[tab1_ai_box, tab2_blogger_box, tab3_quick_ingest_box])
ctrl_tabs.set_title(0, '🔐 ① 金鑰、情境快捷鍵 & 啟動 AI')
ctrl_tabs.set_title(1, '🔗 ② Blogger 一鍵連結 & 網誌選單')
ctrl_tabs.set_title(2, '📝 ③ 免轉錄！直接匯入現成 SRT / TXT')

if not job_dirs:
    ctrl_tabs.selected_index = 2

display(banner_step4_widget, ctrl_tabs, pipeline_out)

# 面板顯示完畢後，等待使用者確認勾選模組並點擊「🚀 開始執行 AI 全媒體創作」按鈕（避免按 ▶ 執行儲存格與點按鈕重複觸發兩次！）
if job_dirs and initial_key and any_ai_enabled:
    key_status_html.value = (
        f'<span style="color:#2e7d32; font-weight:bold; font-size:13.5px;">✅ 金鑰與文稿皆已就緒（來源：{key_source_label}）！請確認上方勾選的模組後，點擊綠色「🚀 開始執行 AI 全媒體創作」按鈕啟動！</span>'
    )
elif job_dirs and not any_ai_enabled:
    for j_dir in job_dirs:
        for name in ('transcript.txt', 'transcript.srt', 'transcript.vtt', 'segments.json'):
            raw_file = j_dir / ('raw_' + name)
            if raw_file.is_file():
                shutil.copy2(raw_file, j_dir / name)
                if j_dir != work:
                    shutil.copy2(raw_file, work / name)
        seg_info = json.loads((j_dir / 'segments.json').read_text(encoding='utf-8'))
        sync_job_to_drive(j_dir, seg_info)
    with pipeline_out:
        print('ℹ️ 所有 AI 模組開關皆為「關閉」，已保留初步字幕並同步至雲端。您也可在上方【分頁 ①】勾選模組後點擊「🚀 開始執行 AI 全媒體創作」！', flush=True)

## <font color="#6a1b9a"><b>5. 下載全媒體創作包</b></font>

In [ ]:
# @title 🟪 **5. 下載全媒體創作包**
import json
import zipfile
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import files

clear_output(wait=True)
print('階段 5/6：正在打包精簡版全媒體創作包（僅包含最終修正版 .srt、高可讀性順稿 .txt 與一頁式全媒體創作戰情室 .html）...', flush=True)

work = Path('/content/alta_transcribe_work')
jobs_root = work / 'jobs'
completed_jobs = []
if jobs_root.is_dir():
    for d in sorted(jobs_root.iterdir()):
        if d.is_dir() and (d / 'transcript.srt').is_file():
            completed_jobs.append(d)

if len(completed_jobs) > 1:
    archive = work / '整批全媒體創作包_ㄚ亮笑長助手.zip'
    pack_progress = widgets.IntProgress(value=0, min=0, max=len(completed_jobs), description='批次打包進度', bar_style='success')
    pack_label = widgets.HTML(value='0%')
    display(widgets.HBox([pack_progress, pack_label]))

    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
        for idx, j_dir in enumerate(completed_jobs, 1):
            seg_info = json.loads((j_dir / 'segments.json').read_text(encoding='utf-8')) if (j_dir / 'segments.json').is_file() else {}
            stem = seg_info.get('stem') or j_dir.name
            candidates = [
                ('transcript.srt', f'{stem}/{stem}.srt'),
                ('transcript.txt', f'{stem}/{stem}.txt'),
                ('all_in_one_hub.html', f'{stem}/{stem}_全媒體創作戰情室.html'),
            ]
            for src_name, arc_name in candidates:
                if (j_dir / src_name).is_file():
                    output.write(j_dir / src_name, arcname=arc_name)
            pack_progress.value = idx
            pack_label.value = f'{idx * 100 // len(completed_jobs)}%'
            print(f'階段 5/6：已打包第 {idx}/{len(completed_jobs)} 個作品（{stem}：.srt + .txt + _全媒體創作戰情室.html）。', flush=True)

    files.download(str(archive))
    print(f'階段 5/6 完成：已提出整批 ZIP 下載要求（{archive.name}）！', flush=True)
else:
    target_dir = completed_jobs[0] if completed_jobs else work
    if not (target_dir / 'transcript.srt').is_file():
        raise RuntimeError('找不到字幕與創作檔案，請先執行步驟 3 或步驟 4！')

    output_stem_path = work / 'output_stem.txt'
    output_stem = output_stem_path.read_text(encoding='utf-8').strip() if output_stem_path.is_file() else '轉錄結果'

    core_candidates = [
        ('transcript.srt', f'{output_stem}.srt'),
        ('transcript.txt', f'{output_stem}.txt'),
        ('all_in_one_hub.html', f'{output_stem}_全媒體創作戰情室.html'),
    ]
    files_to_pack = [(s, a) for s, a in core_candidates if (target_dir / s).is_file()]

    pack_progress = widgets.IntProgress(value=0, min=0, max=len(files_to_pack), description='打包進度', bar_style='success')
    pack_label = widgets.HTML(value='0%')
    display(widgets.HBox([pack_progress, pack_label]))

    archive = work / f'{output_stem}_全媒體創作包.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
        for index, (src_name, arc_name) in enumerate(files_to_pack, 1):
            output.write(target_dir / src_name, arcname=arc_name)
            pack_progress.value = index
            pack_label.value = f'{index * 100 // len(files_to_pack)}%'
            print(f'階段 5/6：已打包 {index}/{len(files_to_pack)} 個核心檔案（{arc_name}）。', flush=True)

    files.download(str(archive))
    print(f'階段 5/6 完成：已提出下載要求（{archive.name}）！', flush=True)

## <font color="#455a64"><b>6. 清除本次暫存資料</b></font>

In [ ]:
# @title ⬜ **6. 清除本次暫存資料並將畫面還原至初始狀態**
import shutil
from pathlib import Path
from IPython.display import clear_output, display, HTML, Javascript
import ipywidgets as widgets

# 1. 一鍵關閉並移除步驟 1~5 所有畫面上展開的互動面板、分頁預覽與進度條！
try:
    widgets.Widget.close_all()
except Exception:
    pass

clear_output(wait=True)

# 2. 清除後台工作目錄暫存檔（不影響已存回 Google 雲端硬碟的成果與永久記憶詞庫）
work = Path('/content/alta_transcribe_work')
if work.exists():
    for item in work.glob('*'):
        try:
            if item.is_file() or item.is_symlink():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
        except Exception:
            pass

# 3. 嘗試透過 Colab 前端指令一併清除其他儲存格殘留的純文字輸出並平滑捲回頁面頂部
display(Javascript('''
try {
  if (window.google && google.colab && google.colab.kernel) {
    // 將頁面平滑捲動回上方步驟區，方便直接開始下一輪任務
    window.scrollTo({top: 0, behavior: "smooth"});
  }
} catch (e) {}
'''))

display(HTML(
    '<div style="padding:12px 16px; border:2px solid #00897b; background:#e0f2f1; border-radius:10px; color:#004d40; line-height:1.75;">'
    '<b>✨ 已清除本次暫存資料，並自動關閉收合前方的所有互動面板與預覽畫面！</b><br>'
    '💡 <b>小技巧：</b>若要連同各步驟殘留的純文字紀錄也一併清空，可點選 Colab 上方選單的 <b>「編輯 (Edit) ➔ 清除所有輸出內容 (Clear all outputs)」</b>。<br>'
    '👉 若要處理下一部音檔或文稿，請直接執行 <b>「步驟 2A」</b> 或 <b>「步驟 2B」</b>（無須重跑步驟 1）！'
    '</div>'
))